# Cognitive–Cultural NLP Pipeline
## Lexicon-Based Cultural Signal Extraction (Research-First)

**Notebook:** 06_NLP_lexicon_scoring.ipynb

**Project:** Cognitive–Cultural Fit Model

**Author:** David Suarez

**Language:** Python

**Purpose:** Exploratory, theory-driven extraction of cultural signals from text

## Context and Scope

This notebook implements the NLP layer of the Cognitive–Cultural Fit project.

The goal is to extract interpretable cultural signals from text (e.g., organizational
mission statements, values, interview responses) using a **theory-driven, lexicon-based
approach** aligned with the psychometric model previously developed.

This NLP layer is:
- Complementary to the psychometric instrument
- Exploratory and descriptive (not predictive)
- Designed for interpretability and academic defensibility

Outputs from this notebook are intended to:
- Generate approximate cultural θ-scores from text
- Enable text-based organizational profiling
- Support future research extensions (embeddings, supervised models)


In [336]:
import re
import numpy as np
import pandas as pd
from collections import Counter


In [337]:
def preprocess_text(text: str) -> list:
    """
    Basic preprocessing:
    - lowercase
    - remove punctuation
    - tokenize by whitespace
    """
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    tokens = text.split()
    return tokens


## Dimension 1 — Goal-Oriented vs. Means-Oriented

This dimension captures whether attention in discourse is oriented toward
final outcomes and impact (Goal-Oriented) or toward processes, procedures,
and methods (Means-Oriented).

The lexicon below is directly derived from the conceptual and psychometric
definitions of Dimension 1.


In [338]:
LEXICON_D1 = {
    "goal": {
        "keywords": [
            "goal", "outcome", "result", "impact", "value", "performance",
            "success", "delivery", "achievement", "metrics"
        ],
        "bigrams": [
            "business impact", "measurable results", "key outcomes",
            "value creation", "performance metrics"
        ],
        "verbs": [
            "achieve", "deliver", "optimize", "accelerate", "drive",
            "improve", "maximize"
        ]
    },
    "means": {
        "keywords": [
            "process", "procedure", "methodology", "framework", "protocol",
            "standard", "guideline", "workflow", "compliance", "documentation"
        ],
        "bigrams": [
            "best practices", "standard operating procedures",
            "defined process", "quality assurance"
        ],
        "verbs": [
            "follow", "document", "ensure", "comply",
            "standardize", "control", "formalize"
        ]
    }
}


In [339]:
def score_dimension_1(text: str, lexicon=LEXICON_D1):
    tokens = preprocess_text(text)
    token_counts = Counter(tokens)
    
    score_goal = 0
    score_means = 0
    
    # Keywords
    for w in lexicon["goal"]["keywords"]:
        score_goal += token_counts.get(w, 0)
    for w in lexicon["means"]["keywords"]:
        score_means += token_counts.get(w, 0)
    
    # Verbs
    for v in lexicon["goal"]["verbs"]:
        score_goal += token_counts.get(v, 0)
    for v in lexicon["means"]["verbs"]:
        score_means += token_counts.get(v, 0)
    
    # Bigrams
    text_lower = text.lower()
    for bg in lexicon["goal"]["bigrams"]:
        score_goal += text_lower.count(bg)
    for bg in lexicon["means"]["bigrams"]:
        score_means += text_lower.count(bg)
    
    theta = (score_goal - score_means) / (score_goal + score_means + 1e-6)
    
    return {
        "score_goal": score_goal,
        "score_means": score_means,
        "theta_D1_text": round(theta, 3)
    }


## Demo — Organizational Text Example

The following example simulates an organizational mission statement and
demonstrates how Dimension 1 is inferred from text.


In [340]:
demo_text = """
We focus on delivering measurable results and maximizing business impact.
Our teams adapt processes pragmatically to achieve strategic goals and
create value for our clients.
"""

score_dimension_1(demo_text)


{'score_goal': 5, 'score_means': 0, 'theta_D1_text': 1.0}

### Interpretation

- Positive θ_D1_text → Goal-Oriented discourse
- Negative θ_D1_text → Means-Oriented discourse
- Magnitude reflects clarity of emphasis

This output is designed to be comparable (conceptually, not metrically) with psychometric θ estimates.


## Methodological Note

This lexicon-based approach does not aim to infer latent traits at the
individual level. Instead, it extracts **cultural signals** from text
based on theoretically grounded linguistic markers.

The approach prioritizes:
- Interpretability
- Transparency
- Alignment with the psychometric model


### Next Steps
- Extend lexicon to Dimensions 2–6
- Aggregate scores to build an Organizational Cultural Profile
- Compare text-derived profiles with psychometric profiles
- Explore hybrid approaches (lexicon + embeddings)


## Dimension 2 — Externally Driven vs. Internally Driven

This dimension captures the predominant source of motivation:

**Externally Driven** → behavior guided by incentives, recognition, pressure, imposed KPIs, and supervision.

**Internally Driven** → behavior guided by autonomy, purpose, values, interest, learning, and mastery.

The lexicon below is directly derived from the conceptual and psychometric
definitions of Dimension 2.

In [341]:
LEXICON_D2 = {
    "internal": {
        "keywords": [
            "autonomy", "purpose", "meaning", "mastery", "growth", "learning",
            "curiosity", "passion", "ownership", "intrinsic", "selfdriven",
            "selfdirected", "values", "missiondriven"
        ],
        "bigrams": [
            "continuous learning", "personal growth", "sense of purpose",
            "intrinsic motivation", "autonomous teams", "ownership mindset",
            "mastery culture", "self directed"
        ],
        "verbs": [
            "learn", "grow", "explore", "improve", "master", "innovate",
            "own", "develop"
        ]
    },
    "external": {
        "keywords": [
            "reward", "bonus", "incentive", "recognition", "approval",
            "compensation", "promotion", "evaluation", "rating", "ranking",
            "targets", "quota", "kpi", "compliance", "supervision", "monitoring",
            "pressure", "accountability"
        ],
        "bigrams": [
            "performance bonus", "performance review", "meet targets",
            "hit quota", "external validation", "manager approval",
            "reward system", "strict kpis", "meet expectations"
        ],
        "verbs": [
            "reward", "incentivize", "evaluate", "monitor", "enforce",
            "supervise", "pressure", "comply"
        ]
    }
}


In [342]:
def score_dimension_2(text: str, lexicon=LEXICON_D2):
    tokens = preprocess_text(text)
    token_counts = Counter(tokens)

    score_internal = 0
    score_external = 0

    # keywords
    for w in lexicon["internal"]["keywords"]:
        score_internal += token_counts.get(w, 0)
    for w in lexicon["external"]["keywords"]:
        score_external += token_counts.get(w, 0)

    # verbs
    for v in lexicon["internal"]["verbs"]:
        score_internal += token_counts.get(v, 0)
    for v in lexicon["external"]["verbs"]:
        score_external += token_counts.get(v, 0)

    # bigrams
    text_lower = text.lower()
    for bg in lexicon["internal"]["bigrams"]:
        score_internal += text_lower.count(bg)
    for bg in lexicon["external"]["bigrams"]:
        score_external += text_lower.count(bg)

    theta = (score_internal - score_external) / (score_internal + score_external + 1e-6)

    return {
        "score_internal": score_internal,
        "score_external": score_external,
        "theta_D2_text": round(theta, 3)
    }


In [343]:
demo_text_d2 = """
We empower autonomous teams and encourage an ownership mindset.
People are motivated by purpose, learning, and continuous growth.
"""

score_dimension_2(demo_text_d2)


{'score_internal': 6, 'score_external': 0, 'theta_D2_text': 1.0}

### Interpretation

θ_D2_text positive → Internally Driven

θ_D2_text negative → Externally Driven

Magnitude reflects clarity of emphasis.

This output is designed to be comparable (conceptually, not metrically) with psychometric θ estimates.

## DIMENSIÓN 3 — Flexibility vs. Structure
Easygoing vs. Strict (D3_ES)

**Conceptual Definition**

This dimension captures the degree to which individuals or organizations prefer:

**Flexible / Easygoing environments**, where rules are treated as contextual guidelines and adaptation is encouraged,
versus

**Structured / Strict environments**, where rules function as mandatory standards and deviations require formal justification.

From a cognitive perspective, this dimension reflects how norms are represented:
as context-sensitive heuristics or as stable, enforceable constraints.

In [344]:
LEXICON_D3 = {
    "easygoing": {
        "keywords": [
            "flexible", "flexibility", "adaptability", "adapt", "agile", "autonomy",
            "pragmatic", "experiment", "iterate", "ambiguity", "dynamic",
            "improvise", "lightweight", "trust"
        ],
        "bigrams": [
            "move fast", "rapid iteration", "flexible approach", "adapt quickly",
            "bias for action", "test and learn", "continuous iteration"
        ],
        "verbs": [
            "adapt", "iterate", "experiment", "adjust", "pivot", "explore", "improvise"
        ]
    },
    "strict": {
        "keywords": [
            "policy", "procedure", "compliance", "governance", "standards", "audit",
            "control", "approval", "documentation", "regulation", "mandatory",
            "rules", "protocol"
        ],
        "bigrams": [
            "strict compliance", "formal approval", "documented process",
            "policy adherence", "standard operating procedures", "audit trail"
        ],
        "verbs": [
            "comply", "enforce", "regulate", "approve", "document", "control", "standardize"
        ]
    }
}


In [345]:
def score_dimension_3(text: str, lexicon=LEXICON_D3):
    tokens = preprocess_text(text)
    token_counts = Counter(tokens)

    score_easy = 0
    score_strict = 0

    # keywords
    for w in lexicon["easygoing"]["keywords"]:
        score_easy += token_counts.get(w, 0)
    for w in lexicon["strict"]["keywords"]:
        score_strict += token_counts.get(w, 0)

    # verbs
    for v in lexicon["easygoing"]["verbs"]:
        score_easy += token_counts.get(v, 0)
    for v in lexicon["strict"]["verbs"]:
        score_strict += token_counts.get(v, 0)

    # bigrams
    text_lower = text.lower()
    for bg in lexicon["easygoing"]["bigrams"]:
        score_easy += text_lower.count(bg)
    for bg in lexicon["strict"]["bigrams"]:
        score_strict += text_lower.count(bg)

    theta = (score_strict - score_easy) / (score_strict + score_easy + 1e-6)

    return {
        "score_easygoing": score_easy,
        "score_strict": score_strict,
        "theta_D3_text": round(theta, 3)
    }


In [346]:
demo_text_d3 = """
We value flexibility and rapid iteration. Teams are encouraged to experiment,
adapt quickly, and adjust their approach as needed.
"""

score_dimension_3(demo_text_d3)


{'score_easygoing': 8, 'score_strict': 0, 'theta_D3_text': -1.0}

In [347]:
demo_text_d3_strict = """
We enforce strict compliance with policies and documented procedures.
Changes require formal approval and audit trails.
"""

score_dimension_3(demo_text_d3_strict)


{'score_easygoing': 0, 'score_strict': 7, 'theta_D3_text': 1.0}

### Interpretation of Textual Scores

The NLP scoring produces a continuous value θ_D3_text:

Positive values (θ > 0) → Strict / Structured orientation

Negative values (θ < 0) → Easygoing / Flexible orientation

Magnitude reflects the clarity and strength of the emphasis

Interpretive bands (suggested):

|θ| < 0.15 → Balanced / ambiguous

0.15 ≤ |θ| < 0.40 → Moderate tendency

|θ| ≥ 0.40 → Clear orientation

These scores are conceptually comparable (but not metrically equivalent) to psychometric θ estimates.

**Methodological Note**

This lexicon-based approach is theory-driven and exploratory.

It does not attempt to infer latent traits at the individual level.
Instead, it identifies cultural signals embedded in discourse, prioritizing:

- Interpretability

- Transparency

- Alignment with the psychometric framework

This makes the approach suitable for academic reporting and mixed-method research designs.

## Dimension 4 — Open System vs. Closed System (D4_OS)
**Objective**

This dimension aims to identify whether a text emphasizes openness to external signals, learning from the environment, and adaptation to stakeholders (Open System) or internal focus, boundary control, and stability of internal processes (Closed System).

The objective is to extract cultural signals indicating how organizations cognitively allocate attention between internal coherence and external responsiveness.

**Conceptual Definition**

This dimension captures the extent to which individuals or organizations orient their attention and decision-making toward:

Open systems, characterized by permeability to external information (customers, market, partners, feedback),
versus

Closed systems, characterized by a primary focus on internal processes, norms, and institutional stability.

From a cognitive standpoint, this dimension reflects what is considered relevant information and where learning is expected to come from.

Psychometric–NLP Bridge

In the psychometric model, this dimension measures the degree of openness versus closure in organizational systems.

In text, this orientation is observable through:

References to customers, markets, and external stakeholders

Framing of feedback and learning sources

Language indicating adaptation versus protection of internal coherence

Thus, linguistic cues provide measurable proxies for systemic orientation.

In [348]:
LEXICON_D4 = {
    "open": {
        "keywords": [
            "customer", "clients", "users", "market", "stakeholders", "partners",
            "feedback", "external", "benchmarking", "competition", "trends",
            "ecosystem", "community", "engagement"
        ],
        "bigrams": [
            "customer feedback", "market trends", "external stakeholders",
            "user research", "voice of the customer", "competitive analysis",
            "industry benchmarks"
        ],
        "verbs": [
            "listen", "adapt", "learn", "respond", "engage", "benchmark",
            "collaborate", "iterate"
        ]
    },
    "closed": {
        "keywords": [
            "internal", "stability", "consistency", "boundaries", "control",
            "policy", "procedure", "governance", "standards", "compliance",
            "confidential", "restricted", "approval"
        ],
        "bigrams": [
            "internal processes", "strict internal", "controlled access",
            "internal alignment", "policy compliance", "restricted information"
        ],
        "verbs": [
            "protect", "control", "restrict", "standardize", "enforce",
            "approve", "document", "filter"
        ]
    }
}

def score_dimension_4(text: str, lexicon=LEXICON_D4):
    tokens = preprocess_text(text)
    token_counts = Counter(tokens)

    score_open = 0
    score_closed = 0

    # keywords
    for w in lexicon["open"]["keywords"]:
        score_open += token_counts.get(w, 0)
    for w in lexicon["closed"]["keywords"]:
        score_closed += token_counts.get(w, 0)

    # verbs
    for v in lexicon["open"]["verbs"]:
        score_open += token_counts.get(v, 0)
    for v in lexicon["closed"]["verbs"]:
        score_closed += token_counts.get(v, 0)

    # bigrams
    text_lower = text.lower()
    for bg in lexicon["open"]["bigrams"]:
        score_open += text_lower.count(bg)
    for bg in lexicon["closed"]["bigrams"]:
        score_closed += text_lower.count(bg)

    # Positive => Open System
    theta = (score_open - score_closed) / (score_open + score_closed + 1e-6)

    return {
        "score_open": score_open,
        "score_closed": score_closed,
        "theta_D4_text": round(theta, 3)
    }


**Interpretation of Textual Scores**

The NLP scoring produces a continuous value θ_D4_text:

Positive values (θ > 0) → Open System orientation

Negative values (θ < 0) → Closed System orientation

Magnitude reflects the clarity and strength of the emphasis

Interpretive bands (suggested):

|θ| < 0.15 → Balanced or mixed orientation

0.15 ≤ |θ| < 0.40 → Moderate tendency

|θ| ≥ 0.40 → Clear orientation

These values are intended to be conceptually aligned with psychometric θ estimates.

Typical Linguistic Patterns

Open System discourse typically:

References customers, users, markets, or external partners

Emphasizes feedback, learning, and continuous adaptation

Frames change as a response to external signals

Closed System discourse typically:

Emphasizes internal processes, standards, and stability

Frames external input as filtered or constrained

Prioritizes internal coherence over rapid adaptation

**Methodological Note**

This lexicon-based NLP approach is theory-driven and interpretive.

It does not seek to infer individual dispositions but rather to extract system-level cultural signals embedded in organizational discourse.

The emphasis on transparency and theoretical grounding makes this approach suitable for academic research and mixed-method applications.

## Dimension 5 — People-Oriented vs. Task-Oriented (D5_EJ)
**Objective**
This dimension aims to identify whether a text emphasizes people, relationships, well-being, and socioemotional climate (People-Oriented) or tasks, performance, efficiency, and execution (Task-Oriented).
The goal is to extract cultural signals indicating how success is cognitively defined: through human impact and relational quality versus output, productivity, and task completion.

**Conceptual Definition**
This dimension captures the degree to which individuals or organizations prioritize:


**People-Oriented cultures**, where attention is directed toward interpersonal relationships, psychological safety, and employee well-being,
versus


**Task-Oriented cultures**, where attention is directed toward execution, efficiency, deliverables, and measurable performance.


From a cognitive perspective, this dimension reflects what is considered central to effectiveness and how coordination is justified.

**Psychometric–NLP Bridge**
In the psychometric instrument, this dimension measures preference for people-focused versus task-focused organizational orientations.
In text, this preference becomes observable through:


- How success and effectiveness are described
- Whether language prioritizes human experience or operational output
- The framing of feedback, coordination, and decision-making criteria
- Thus, linguistic emphasis serves as a proxy for underlying cultural priorities.

**Interpretation of Textual Scores**
The NLP scoring produces a continuous value θ_D5_text:

Positive values (θ > 0) → Task-Oriented orientation
Negative values (θ < 0) → People-Oriented orientation
Magnitude reflects the clarity and strength of the emphasis


**Interpretive bands (suggested):**


|θ| < 0.15 → Balanced or mixed orientation

0.15 ≤ |θ| < 0.40 → Moderate tendency

|θ| ≥ 0.40 → Clear orientation


These scores are conceptually comparable (but not metrically equivalent) to psychometric θ estimates.

**Typical Linguistic Patterns**
People-Oriented discourse typically:


- Emphasizes well-being, support, collaboration, and care


- Frames effectiveness in terms of team health and relationships


- Prioritizes communication quality and psychological safety


**Task-Oriented discourse typically:**


- Emphasizes efficiency, execution, and performance outcomes


- Frames effectiveness in terms of output and delivery


- Prioritizes accountability, throughput, and results



**Methodological Note**
This lexicon-based approach is theory-driven and exploratory.
It is designed to extract cultural emphases from discourse, not to diagnose individual traits.
The focus on interpretability ensures alignment with the psychometric framework and suitability for academic reporting.

In [349]:
LEXICON_D5 = {
    "people": {
        "keywords": [
            "wellbeing", "well-being", "support", "care", "empathy",
            "collaboration", "teamwork", "psychological", "safety",
            "inclusion", "belonging", "respect", "trust", "relationships",
            "growth", "development", "coaching"
        ],
        "bigrams": [
            "psychological safety", "employee wellbeing", "team support",
            "people first", "strong relationships", "inclusive culture",
            "human centered"
        ],
        "verbs": [
            "support", "care", "mentor", "coach", "collaborate",
            "listen", "help", "encourage"
        ]
    },
    "task": {
        "keywords": [
            "performance", "efficiency", "execution", "deliverables", "output",
            "productivity", "throughput", "deadlines", "goals", "targets",
            "metrics", "results", "accountability", "kpi", "quality"
        ],
        "bigrams": [
            "high performance", "meet deadlines", "deliver results",
            "performance metrics", "execution excellence", "operational efficiency"
        ],
        "verbs": [
            "deliver", "execute", "optimize", "prioritize", "measure",
            "improve", "achieve", "track"
        ]
    }
}

def score_dimension_5(text: str, lexicon=LEXICON_D5):
    tokens = preprocess_text(text)
    token_counts = Counter(tokens)

    score_people = 0
    score_task = 0

    # keywords
    for w in lexicon["people"]["keywords"]:
        score_people += token_counts.get(w, 0)
    for w in lexicon["task"]["keywords"]:
        score_task += token_counts.get(w, 0)

    # verbs
    for v in lexicon["people"]["verbs"]:
        score_people += token_counts.get(v, 0)
    for v in lexicon["task"]["verbs"]:
        score_task += token_counts.get(v, 0)

    # bigrams
    text_lower = text.lower()
    for bg in lexicon["people"]["bigrams"]:
        score_people += text_lower.count(bg)
    for bg in lexicon["task"]["bigrams"]:
        score_task += text_lower.count(bg)

    # Positive => Task-oriented
    theta = (score_task - score_people) / (score_task + score_people + 1e-6)

    return {
        "score_people": score_people,
        "score_task": score_task,
        "theta_D5_text": round(theta, 3)
    }


## Dimension 6 — Parochial vs. Professional Identity (D6_PP)
**Objective**

This dimension aims to identify whether a text emphasizes organizational belonging, internal loyalty, and institution-specific identity (Parochial) or professional standards, expertise, and external disciplinary identity (Professional).

The objective is to extract cultural signals indicating where the primary locus of work identity is anchored: within the organization itself or within a broader professional community.

**Conceptual Definition**

This dimension captures the degree to which individuals or organizations orient identity and legitimacy toward:

Parochial identity, where the professional self is primarily defined through membership in a specific organization and adherence to its internal norms,
versus

Professional identity, where the self is primarily defined through expertise, external standards, and professional communities beyond the organization.

From a cognitive perspective, this dimension reflects how individuals evaluate quality, success, and ethical alignment in their work.

**Psychometric–NLP Bridge**

In the psychometric instrument, this dimension measures whether identity is anchored internally (organization-centered) or externally (profession-centered).

In text, this orientation is observable through:

- References to organizational loyalty versus professional standards

- Emphasis on internal recognition versus external credibility

- Language framing expertise as context-specific versus transferable

Thus, linguistic patterns serve as proxies for identity orientation.

**Interpretation of Textual Scores**

The NLP scoring produces a continuous value θ_D6_text:

Positive values (θ > 0) → Professional identity orientation

Negative values (θ < 0) → Parochial identity orientation

Magnitude reflects the clarity and strength of the emphasis

**Interpretive bands (suggested):**

|θ| < 0.15 → Balanced or mixed identity

0.15 ≤ |θ| < 0.40 → Moderate tendency

|θ| ≥ 0.40 → Clear orientation

These values are conceptually aligned with psychometric θ estimates.

**Typical Linguistic Patterns**

Parochial identity discourse typically:

- Emphasizes loyalty to the organization and internal culture

- Frames success in terms of internal recognition and alignment

- References organization-specific norms, values, and ways of working

- Professional identity discourse typically:

- Emphasizes expertise, craft, and professional standards

- Frames success in terms of external credibility and best practices

- References industry benchmarks, certifications, or professional communities

**Methodological Note**

This lexicon-based approach is theory-driven and interpretive.

It focuses on extracting identity-related cultural signals from discourse, rather than diagnosing individual professional traits.
The emphasis on transparency and theoretical grounding supports academic rigor and reproducibility.

In [350]:
LEXICON_D6 = {
    "parochial": {
        "keywords": [
            "loyalty", "belonging", "family", "our way", "culture",
            "internal", "insider", "tenure", "tradition", "legacy",
            "commitment", "alignment", "values", "identity"
        ],
        "bigrams": [
            "our culture", "our values", "the way we", "internal recognition",
            "organizational loyalty", "strong belonging"
        ],
        "verbs": [
            "align", "belong", "commit", "support", "represent", "embrace"
        ]
    },
    "professional": {
        "keywords": [
            "expertise", "standards", "best practices", "benchmark",
            "industry", "certification", "professional", "craft",
            "community", "discipline", "ethics", "specialization"
        ],
        "bigrams": [
            "industry standards", "best practices", "professional community",
            "external benchmarks", "technical excellence", "professional ethics"
        ],
        "verbs": [
            "benchmark", "specialize", "certify", "master", "improve", "upskill"
        ]
    }
}

def score_dimension_6(text: str, lexicon=LEXICON_D6):
    tokens = preprocess_text(text)
    token_counts = Counter(tokens)

    score_parochial = 0
    score_professional = 0

    # keywords
    for w in lexicon["parochial"]["keywords"]:
        score_parochial += token_counts.get(w, 0)
    for w in lexicon["professional"]["keywords"]:
        score_professional += token_counts.get(w, 0)

    # verbs
    for v in lexicon["parochial"]["verbs"]:
        score_parochial += token_counts.get(v, 0)
    for v in lexicon["professional"]["verbs"]:
        score_professional += token_counts.get(v, 0)

    # bigrams
    text_lower = text.lower()
    for bg in lexicon["parochial"]["bigrams"]:
        score_parochial += text_lower.count(bg)
    for bg in lexicon["professional"]["bigrams"]:
        score_professional += text_lower.count(bg)

    # Positive => Professional identity
    theta = (score_professional - score_parochial) / (score_professional + score_parochial + 1e-6)

    return {
        "score_parochial": score_parochial,
        "score_professional": score_professional,
        "theta_D6_text": round(theta, 3)
    }


## Score Text - All Dimensions

In [351]:
def score_text_all_dimensions(text: str):
    results = {}
    raw = {}

    # D1
    d1 = score_dimension_1(text)
    results["theta_D1_GO_text"] = d1["theta_D1_text"]
    raw["D1_GO"] = {k: v for k, v in d1.items() if k != "theta_D1_text"}

    # D2
    d2 = score_dimension_2(text)
    results["theta_D2_IDED_text"] = d2["theta_D2_text"]
    raw["D2_IDED"] = {k: v for k, v in d2.items() if k != "theta_D2_text"}

    # D3
    d3 = score_dimension_3(text)
    results["theta_D3_ES_text"] = d3["theta_D3_text"]
    raw["D3_ES"] = {k: v for k, v in d3.items() if k != "theta_D3_text"}

    # D4
    d4 = score_dimension_4(text)
    results["theta_D4_OS_text"] = d4["theta_D4_text"]
    raw["D4_OS"] = {k: v for k, v in d4.items() if k != "theta_D4_text"}

    # D5
    d5 = score_dimension_5(text)
    results["theta_D5_EJ_text"] = d5["theta_D5_text"]
    raw["D5_EJ"] = {k: v for k, v in d5.items() if k != "theta_D5_text"}

    # D6
    d6 = score_dimension_6(text)
    results["theta_D6_PP_text"] = d6["theta_D6_text"]
    raw["D6_PP"] = {k: v for k, v in d6.items() if k != "theta_D6_text"}

    return results, raw



In [352]:
demo_text = """
We focus on delivering measurable results and maximizing business impact.
At the same time, we empower autonomous teams and foster continuous learning,
ownership, and growth.
"""

thetas_text, raw_scores = score_text_all_dimensions(demo_text)

thetas_text, raw_scores


({'theta_D1_GO_text': 1.0,
  'theta_D2_IDED_text': 1.0,
  'theta_D3_ES_text': 0.0,
  'theta_D4_OS_text': 0.0,
  'theta_D5_EJ_text': 0.0,
  'theta_D6_PP_text': 0.0},
 {'D1_GO': {'score_goal': 3, 'score_means': 0},
  'D2_IDED': {'score_internal': 5, 'score_external': 0},
  'D3_ES': {'score_easygoing': 0, 'score_strict': 0},
  'D4_OS': {'score_open': 0, 'score_closed': 0},
  'D5_EJ': {'score_people': 1, 'score_task': 1},
  'D6_PP': {'score_parochial': 0, 'score_professional': 0}})

In [353]:
def score_corpus_texts(texts: list, ids: list = None):
    """
    Scores multiple texts and returns a DataFrame with theta per dimension.
    """
    if ids is None:
        ids = list(range(1, len(texts)+1))

    rows = []
    for _id, txt in zip(ids, texts):
        thetas, raw = score_text_all_dimensions(txt)
        row = {"id": _id, "text": txt}
        row.update(thetas)
        rows.append(row)

    return pd.DataFrame(rows)


In [354]:
texts = [
    "We ensure compliance with standard operating procedures and rigorous documentation.",
    "We empower autonomy, purpose, and learning to achieve meaningful impact.",
    "We deliver measurable results and optimize performance through structured workflows."
]

df_scores = score_corpus_texts(texts, ids=["Org_A", "Org_B", "Org_C"])
df_scores


,id,text,theta_D1_GO_text,theta_D2_IDED_text,theta_D3_ES_text,theta_D4_OS_text,theta_D5_EJ_text,theta_D6_PP_text
0,Org_A,We ensure compliance with standard operating p...,-1.0,-1.0,1.0,-1.0,0.0,0.0
1,Org_B,"We empower autonomy, purpose, and learning to ...",1.0,1.0,-1.0,0.0,1.0,0.0
2,Org_C,We deliver measurable results and optimize per...,1.0,0.0,0.0,0.0,1.0,0.0


In [355]:
demo_org_text = """
We deliver measurable impact for our clients and optimize performance through clear goals.
Teams are empowered with autonomy and a strong sense of purpose, learning, and growth.
We adapt quickly to market feedback and continuously benchmark against industry standards.
We value psychological safety and collaboration, while maintaining high accountability and execution.
We uphold professional ethics, technical excellence, and best practices across our discipline.
"""

thetas, raw = score_text_all_dimensions(demo_org_text)
thetas, raw


({'theta_D1_GO_text': 0.667,
  'theta_D2_IDED_text': 0.667,
  'theta_D3_ES_text': -0.6,
  'theta_D4_OS_text': 0.667,
  'theta_D5_EJ_text': 0.091,
  'theta_D6_PP_text': 1.0},
 {'D1_GO': {'score_goal': 5, 'score_means': 1},
  'D2_IDED': {'score_internal': 5, 'score_external': 1},
  'D3_ES': {'score_easygoing': 4, 'score_strict': 1},
  'D4_OS': {'score_open': 5, 'score_closed': 1},
  'D5_EJ': {'score_people': 5, 'score_task': 6},
  'D6_PP': {'score_parochial': 0, 'score_professional': 11}})

## Organizational Cultural Profile (Text-derived)

In [356]:
DIM_ORDER = ["D1_GO", "D2_IDED", "D3_ES", "D4_OS", "D5_EJ", "D6_PP"]

THETA_KEY_MAP = {
    "D1_GO": "theta_D1_GO_text",
    "D2_IDED": "theta_D2_IDED_text",
    "D3_ES": "theta_D3_ES_text",
    "D4_OS": "theta_D4_OS_text",
    "D5_EJ": "theta_D5_EJ_text",
    "D6_PP": "theta_D6_PP_text",
}

def score_org_documents(docs: list, doc_labels: list = None):
    """
    docs: list of text documents (mission, values, interview, etc.)
    Returns:
      df_doc_scores: document-level theta scores
    """
    if doc_labels is None:
        doc_labels = [f"doc_{i+1}" for i in range(len(docs))]

    rows = []
    for label, text in zip(doc_labels, docs):
        thetas, raw = score_text_all_dimensions(text)
        row = {"doc": label, "text": text}
        row.update(thetas)
        rows.append(row)

    return pd.DataFrame(rows)

def aggregate_org_profile(df_doc_scores: pd.DataFrame):
    """
    Aggregates document-level scores into an organization profile.
    Outputs:
      profile_mean: dict with mean theta per dimension
      profile_sd: dict with sd theta per dimension
    """
    profile_mean = {}
    profile_sd = {}

    for d in DIM_ORDER:
        col = THETA_KEY_MAP[d]
        profile_mean[d] = float(df_doc_scores[col].mean())
        profile_sd[d] = float(df_doc_scores[col].std(ddof=1)) if len(df_doc_scores) > 1 else 0.0

    return profile_mean, profile_sd


In [357]:
def interpret_theta(theta: float, pos_pole_label: str, neg_pole_label: str):
    """
    Returns (dominant_pole, strength_label).
    """
    abs_t = abs(theta)
    if abs_t < 0.15:
        strength = "Balanced / unclear"
    elif abs_t < 0.40:
        strength = "Moderate"
    else:
        strength = "Clear"

    if theta > 0:
        pole = pos_pole_label
    elif theta < 0:
        pole = neg_pole_label
    else:
        pole = "Balanced"

    return pole, strength


In [358]:
DIM_POLES_EN = {
    "D1_GO": {"pos": "Goal-Oriented", "neg": "Means-Oriented"},
    "D2_IDED": {"pos": "Internally Driven", "neg": "Externally Driven"},
    "D3_ES": {"pos": "Strict / Structured", "neg": "Easygoing / Flexible"},
    "D4_OS": {"pos": "Open System", "neg": "Closed System"},
    "D5_EJ": {"pos": "Task-Oriented", "neg": "People-Oriented"},
    "D6_PP": {"pos": "Professional Identity", "neg": "Parochial Identity"},
}


In [359]:
def org_profile_summary_table(profile_mean: dict, profile_sd: dict):
    rows = []
    for d in DIM_ORDER:
        theta = profile_mean[d]
        poles = DIM_POLES_EN[d]
        pole, strength = interpret_theta(theta, poles["pos"], poles["neg"])
        rows.append({
            "Dimension": d,
            "Theta_mean": round(theta, 3),
            "Theta_sd": round(profile_sd[d], 3),
            "Dominant_pole": pole,
            "Strength": strength
        })
    return pd.DataFrame(rows)


In [360]:
org_docs = [
    # Mission
    "We deliver measurable impact for customers, adapting quickly to market feedback and evolving needs.",
    # Values
    "We empower autonomy and learning. Teams take ownership and continuously improve through experimentation.",
    # Policy / governance snippet
    "Key changes require formal approval and documented procedures to ensure compliance and auditability.",
    # Job description
    "We seek high accountability, strong execution, and measurable performance outcomes in fast-paced environments."
]

labels = ["mission", "values", "governance", "job_description"]

df_docs = score_org_documents(org_docs, labels)
profile_mean, profile_sd = aggregate_org_profile(df_docs)
df_profile = org_profile_summary_table(profile_mean, profile_sd)

df_docs, df_profile


(               doc                                               text  \
 0          mission  We deliver measurable impact for customers, ad...   
 1           values  We empower autonomy and learning. Teams take o...   
 2       governance  Key changes require formal approval and docume...   
 3  job_description  We seek high accountability, strong execution,...   
 
    theta_D1_GO_text  theta_D2_IDED_text  theta_D3_ES_text  theta_D4_OS_text  \
 0               1.0                 0.0               0.0               1.0   
 1               1.0                 1.0              -1.0               0.0   
 2              -1.0                -1.0               1.0              -1.0   
 3               1.0                -1.0               0.0               0.0   
 
    theta_D5_EJ_text  theta_D6_PP_text  
 0               1.0               0.0  
 1               1.0               1.0  
 2               0.0               0.0  
 3               1.0               0.0  ,
   Dimension  Theta_

In [361]:
org_thetas_text = {d: profile_mean[d] for d in DIM_ORDER}
org_thetas_text


{'D1_GO': 0.5,
 'D2_IDED': -0.25,
 'D3_ES': 0.0,
 'D4_OS': 0.0,
 'D5_EJ': 0.75,
 'D6_PP': 0.25}

In [362]:
ORG_TEXTS = {
    "Org_A_GovernedEnterprise": {
        "mission": "We ensure stability, compliance, and predictable delivery through standardized operating procedures.",
        "values": "We value discipline, documentation, auditability, and consistency in execution.",
        "governance": "All changes require formal approval, policy adherence, and documented justification for exceptions.",
        "job_description": "We seek reliable contributors who follow processes, maintain quality controls, and respect governance."
    },
    "Org_B_StartupImpact": {
        "mission": "We deliver measurable impact by iterating fast, adapting to customers, and optimizing outcomes.",
        "values": "We empower autonomy, experimentation, and ownership. We learn fast and improve continuously.",
        "governance": "Teams decide locally. Lightweight guidelines exist, but pragmatism and speed matter when learning.",
        "job_description": "We look for self-driven builders who thrive in ambiguity, move fast, and deliver outcomes."
    },
    "Org_C_PeopleCenteredCommunity": {
        "mission": "We build human-centered experiences by listening deeply to users and supporting our communities.",
        "values": "We prioritize psychological safety, inclusion, empathy, and strong relationships across teams.",
        "governance": "We make decisions collaboratively, ensuring well-being and sustainable teamwork.",
        "job_description": "We seek caring collaborators who coach others, communicate thoughtfully, and support team growth."
    }
}


In [363]:
def build_org_profiles(org_texts_dict):
    profiles = {}
    profiles_sd = {}
    doc_tables = {}

    for org_name, docs_map in org_texts_dict.items():
        labels = list(docs_map.keys())
        docs = list(docs_map.values())

        df_docs = score_org_documents(docs, labels)
        mean_prof, sd_prof = aggregate_org_profile(df_docs)

        doc_tables[org_name] = df_docs
        profiles[org_name] = mean_prof
        profiles_sd[org_name] = sd_prof

    return profiles, profiles_sd, doc_tables

profiles, profiles_sd, doc_tables = build_org_profiles(ORG_TEXTS)
profiles


{'Org_A_GovernedEnterprise': {'D1_GO': -0.33325,
  'D2_IDED': -0.5,
  'D3_ES': 1.0,
  'D4_OS': -1.0,
  'D5_EJ': 0.25,
  'D6_PP': 0.25},
 'Org_B_StartupImpact': {'D1_GO': 0.75,
  'D2_IDED': 0.75,
  'D3_ES': -0.75,
  'D4_OS': 0.25,
  'D5_EJ': 0.75,
  'D6_PP': 0.25},
 'Org_C_PeopleCenteredCommunity': {'D1_GO': 0.0,
  'D2_IDED': 0.25,
  'D3_ES': 0.0,
  'D4_OS': 0.25,
  'D5_EJ': -0.6875,
  'D6_PP': -0.25}}

In [364]:
def profiles_to_df(profiles_dict):
    rows = []
    for org, prof in profiles_dict.items():
        row = {"org": org}
        for d in DIM_ORDER:
            row[d] = prof[d]
        rows.append(row)
    return pd.DataFrame(rows)

df_org_profiles = profiles_to_df(profiles)
df_org_profiles


,org,D1_GO,D2_IDED,D3_ES,D4_OS,D5_EJ,D6_PP
0,Org_A_GovernedEnterprise,-0.33325,-0.50,1.00,-1.00,0.2500,0.25
1,Org_B_StartupImpact,0.75000,0.75,-0.75,0.25,0.7500,0.25
2,Org_C_PeopleCenteredCommunity,0.00000,0.25,0.00,0.25,-0.6875,-0.25


In [366]:
def interpret_org_profile(org_name, mean_prof, sd_prof):
    df = org_profile_summary_table(mean_prof, sd_prof)
    df.insert(0, "org", org_name)
    return df

df_interpret = pd.concat(
    [interpret_org_profile(org, profiles[org], profiles_sd[org]) for org in profiles.keys()],
    ignore_index=True
)
df_interpret


,org,Dimension,Theta_mean,Theta_sd,Dominant_pole,Strength
0,Org_A_GovernedEnterprise,D1_GO,-0.333,0.471,Means-Oriented,Moderate
1,Org_A_GovernedEnterprise,D2_IDED,-0.500,0.577,Externally Driven,Clear
2,Org_A_GovernedEnterprise,D3_ES,1.000,0.000,Strict / Structured,Clear
3,Org_A_GovernedEnterprise,D4_OS,-1.000,0.000,Closed System,Clear
4,Org_A_GovernedEnterprise,D5_EJ,0.250,0.500,Task-Oriented,Moderate
5,Org_A_GovernedEnterprise,D6_PP,0.250,0.500,Professional Identity,Moderate
6,Org_B_StartupImpact,D1_GO,0.750,0.500,Goal-Oriented,Clear
7,Org_B_StartupImpact,D2_IDED,0.750,0.500,Internally Driven,Clear
8,Org_B_StartupImpact,D3_ES,-0.750,0.500,Easygoing / Flexible,Clear
9,Org_B_StartupImpact,D4_OS,0.250,0.500,Open System,Moderate


In [367]:
df_org_profiles.set_index("org").T


org,Org_A_GovernedEnterprise,Org_B_StartupImpact,Org_C_PeopleCenteredCommunity
D1_GO,-0.33325,0.75,0.0000
D2_IDED,-0.50000,0.75,0.2500
D3_ES,1.00000,-0.75,0.0000
D4_OS,-1.00000,0.25,0.2500
D5_EJ,0.25000,0.75,-0.6875
D6_PP,0.25000,0.25,-0.2500


## Cosine Similarity

In [368]:
import numpy as np
import pandas as pd

def profile_to_vector(profile_dict):
    """Return vector in DIM_ORDER order."""
    return np.array([profile_dict[d] for d in DIM_ORDER], dtype=float)


In [369]:
def cosine_sim(a, b, eps=1e-9):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    return float(np.dot(a, b) / ((np.linalg.norm(a) * np.linalg.norm(b)) + eps))


In [370]:
def cosine_similarity_matrix(profiles_dict):
    orgs = list(profiles_dict.keys())
    vectors = {org: profile_to_vector(profiles_dict[org]) for org in orgs}

    mat = np.zeros((len(orgs), len(orgs)))
    for i, oi in enumerate(orgs):
        for j, oj in enumerate(orgs):
            mat[i, j] = cosine_sim(vectors[oi], vectors[oj])

    df_sim = pd.DataFrame(mat, index=orgs, columns=orgs)
    return df_sim

df_cosine = cosine_similarity_matrix(profiles)
df_cosine


,Org_A_GovernedEnterprise,Org_B_StartupImpact,Org_C_PeopleCenteredCommunity
Org_A_GovernedEnterprise,1.000000,-0.565843,-0.47567
Org_B_StartupImpact,-0.565843,1.000000,-0.26205
Org_C_PeopleCenteredCommunity,-0.475670,-0.262050,1.00000


In [371]:
def most_similar_orgs(df_sim):
    out = []
    for org in df_sim.index:
        sims = df_sim.loc[org].drop(org)  # drop self
        best_org = sims.idxmax()
        best_val = sims.max()
        out.append({"org": org, "most_similar_org": best_org, "cosine_similarity": round(best_val, 3)})
    return pd.DataFrame(out)

df_most_similar = most_similar_orgs(df_cosine)
df_most_similar


,org,most_similar_org,cosine_similarity
0,Org_A_GovernedEnterprise,Org_C_PeopleCenteredCommunity,-0.476
1,Org_B_StartupImpact,Org_C_PeopleCenteredCommunity,-0.262
2,Org_C_PeopleCenteredCommunity,Org_B_StartupImpact,-0.262


In [372]:
def similarity_ranking(df_sim, target_org):
    sims = df_sim.loc[target_org].drop(target_org).sort_values(ascending=False)
    df = sims.reset_index()
    df.columns = ["org", "cosine_similarity"]
    df.insert(0, "target_org", target_org)
    df["cosine_similarity"] = df["cosine_similarity"].round(3)
    return df

df_rank_vs_B = similarity_ranking(df_cosine, "Org_B_StartupImpact")
df_rank_vs_B


,target_org,org,cosine_similarity
0,Org_B_StartupImpact,Org_C_PeopleCenteredCommunity,-0.262
1,Org_B_StartupImpact,Org_A_GovernedEnterprise,-0.566


In [373]:
def profile_vector_table(profiles_dict):
    rows = []
    for org, prof in profiles_dict.items():
        row = {"org": org}
        for d in DIM_ORDER:
            row[d] = round(prof[d], 3)
        rows.append(row)
    return pd.DataFrame(rows)

df_vectors = profile_vector_table(profiles)
df_vectors


,org,D1_GO,D2_IDED,D3_ES,D4_OS,D5_EJ,D6_PP
0,Org_A_GovernedEnterprise,-0.333,-0.50,1.00,-1.00,0.250,0.25
1,Org_B_StartupImpact,0.750,0.75,-0.75,0.25,0.750,0.25
2,Org_C_PeopleCenteredCommunity,0.000,0.25,0.00,0.25,-0.688,-0.25


**Quick Interpretation (For Paper/Demo)**

Cosine similarity measures similarity based on the direction of the cultural vector (the general pattern), rather than magnitude.

Extremely useful for determining "how similar the overall cultural profile is."

If two organizations have a similar orientation across several dimensions (even with different intensities), the cosine similarity is usually high.

In [374]:
import numpy as np
import pandas as pd

def fit_metrics(person_vec, org_vec):
    """
    person_vec, org_vec: np arrays (DIM_ORDER)
    Returns dict: abs diffs + summary stats + fit%
    """
    person_vec = np.array(person_vec, dtype=float)
    org_vec = np.array(org_vec, dtype=float)

    diff = org_vec - person_vec
    abs_diff = np.abs(diff)

    mean_abs = float(abs_diff.mean())
    median_abs = float(np.median(abs_diff))

    # Fit% interpretativo (simple, demo-friendly):
    # 0 gap -> 100%, gap >= 1.5 -> 0%
    fit_pct = float(np.clip(1 - (mean_abs / 1.5), 0, 1) * 100)

    return {
        "diff": diff,
        "abs_diff": abs_diff,
        "mean_abs": mean_abs,
        "median_abs": median_abs,
        "fit_pct": fit_pct
    }

def global_fit_label(mean_abs):
    # Ajusta si ya tienes tu versión definitiva en inglés; esto es un ejemplo consistente.
    if mean_abs < 0.25:
        return "Very High overall fit"
    elif mean_abs < 0.40:
        return "High overall fit"
    elif mean_abs < 0.60:
        return "Moderate overall fit"
    elif mean_abs < 0.85:
        return "Moderate-to-low overall fit"
    else:
        return "Low overall fit"


In [375]:
# Example person (instrument-derived θ)
person_thetas = {
    "D1_GO":  0.60,  # Goal-oriented
    "D2_IDED": 0.80, # Internally driven
    "D3_ES": -0.20,  # Easygoing/flexible
    "D4_OS":  0.40,  # Open system
    "D5_EJ": -0.50,  # People-oriented
    "D6_PP":  0.70   # Professional identity
}

def dict_to_vec(d):
    return np.array([d[k] for k in DIM_ORDER], dtype=float)

person_vec = dict_to_vec(person_thetas)
person_vec


array([ 0.6,  0.8, -0.2,  0.4, -0.5,  0.7])

In [376]:
def rank_person_fit_against_orgs(person_thetas, org_profiles):
    person_vec = dict_to_vec(person_thetas)

    rows = []
    for org_name, org_prof in org_profiles.items():
        org_vec = dict_to_vec(org_prof)

        m = fit_metrics(person_vec, org_vec)

        rows.append({
            "org": org_name,
            "overall_fit_label": global_fit_label(m["mean_abs"]),
            "fit_pct": round(m["fit_pct"], 1),
            "mean_abs_diff": round(m["mean_abs"], 3),
            "median_abs_diff": round(m["median_abs"], 3),
        })

    df = pd.DataFrame(rows).sort_values(by=["mean_abs_diff"], ascending=True).reset_index(drop=True)
    return df

df_fit_rank = rank_person_fit_against_orgs(person_thetas, profiles)
df_fit_rank


,org,overall_fit_label,fit_pct,mean_abs_diff,median_abs_diff
0,Org_B_StartupImpact,Moderate overall fit,71.1,0.433,0.300
1,Org_C_PeopleCenteredCommunity,Moderate overall fit,70.7,0.440,0.375
2,Org_A_GovernedEnterprise,Low overall fit,33.0,1.006,1.067


In [377]:
def per_dimension_fit_table(person_thetas, org_thetas, org_name="Org"):
    person_vec = dict_to_vec(person_thetas)
    org_vec = dict_to_vec(org_thetas)

    diff = org_vec - person_vec
    abs_diff = np.abs(diff)

    rows = []
    for i, d in enumerate(DIM_ORDER):
        poles = DIM_POLES_EN[d]
        # Dominant poles:
        p_pole = poles["pos"] if person_vec[i] > 0 else poles["neg"]
        o_pole = poles["pos"] if org_vec[i] > 0 else poles["neg"]

        rows.append({
            "dimension": d,
            "person_theta": round(person_vec[i], 3),
            "org_theta": round(org_vec[i], 3),
            "delta_theta": round(diff[i], 3),
            "abs_delta": round(abs_diff[i], 3),
            "person_tendency": p_pole,
            "org_tendency": o_pole
        })

    df = pd.DataFrame(rows).sort_values("abs_delta", ascending=False).reset_index(drop=True)
    df.insert(0, "org", org_name)
    return df

best_org = df_fit_rank.iloc[0]["org"]
df_dim_detail = per_dimension_fit_table(person_thetas, profiles[best_org], org_name=best_org)
df_dim_detail


,org,dimension,person_theta,org_theta,delta_theta,abs_delta,person_tendency,org_tendency
0,Org_B_StartupImpact,D5_EJ,-0.5,0.75,1.25,1.25,People-Oriented,Task-Oriented
1,Org_B_StartupImpact,D3_ES,-0.2,-0.75,-0.55,0.55,Easygoing / Flexible,Easygoing / Flexible
2,Org_B_StartupImpact,D6_PP,0.7,0.25,-0.45,0.45,Professional Identity,Professional Identity
3,Org_B_StartupImpact,D1_GO,0.6,0.75,0.15,0.15,Goal-Oriented,Goal-Oriented
4,Org_B_StartupImpact,D4_OS,0.4,0.25,-0.15,0.15,Open System,Open System
5,Org_B_StartupImpact,D2_IDED,0.8,0.75,-0.05,0.05,Internally Driven,Internally Driven


## Fit Instrument

In [378]:
import re
import numpy as np
import pandas as pd

# Minimal lexicon v0.1 — interpretable and editable
LEXICON_MINI = {
    "D1_GO": {  # + => Goal, - => Means
        "pos": ["outcome", "impact", "result", "deliverable", "goal", "target", "metric", "KPI", "ship", "deadline", "value"],
        "neg": ["process", "procedure", "checklist", "protocol", "documentation", "traceability", "compliance", "steps", "standard", "policy"]
    },
    "D2_IDED": {  # + => Internal, - => External
        "pos": ["purpose", "meaning", "autonomy", "ownership", "learning", "curiosity", "mastery", "intrinsic", "growth", "self-driven"],
        "neg": ["reward", "bonus", "promotion", "recognition", "approval", "supervision", "pressure", "status", "rating", "evaluation"]
    },
    "D3_ES": {  # + => Strict, - => Easygoing
        "pos": ["rule", "policy", "governance", "approval", "audit", "control", "standard", "mandatory", "strict", "compliance"],
        "neg": ["flexible", "adapt", "pragmatic", "iterate", "improvise", "ambiguity", "adjust", "lightweight", "context", "exception"]
    },
    "D4_OS": {  # + => Open, - => Closed
        "pos": ["customer", "market", "feedback", "user", "stakeholder", "benchmark", "trend", "external", "listen", "insight"],
        "neg": ["internal", "inside", "stability", "protect", "confidential", "boundary", "filter", "alignment", "control", "consistency"]
    },
    "D5_EJ": {  # + => Task, - => People
        "pos": ["performance", "execution", "efficiency", "productivity", "deliver", "accountability", "throughput", "output", "prioritize", "results"],
        "neg": ["well-being", "empathy", "support", "psychological safety", "care", "relationships", "collaboration", "coaching", "trust", "inclusion"]
    },
    "D6_PP": {  # + => Professional, - => Parochial
        "pos": ["industry", "best practice", "standards", "profession", "craft", "expertise", "certification", "community", "external standards", "ethics"],
        "neg": ["our way", "company norms", "internal culture", "loyalty", "belonging", "internal standards", "institutional", "org identity", "here", "inside"]
    }
}

def tokenize(text):
    # simple tokenization; keeps phrases through substring match in hit function
    t = text.lower()
    t = re.sub(r"\s+", " ", t)
    return t

def count_hits(text, terms):
    """
    counts substring matches (good for phrases like 'psychological safety')
    """
    t = tokenize(text)
    hits = 0
    found = []
    for term in terms:
        term_l = term.lower()
        # count occurrences
        n = t.count(term_l)
        if n > 0:
            hits += n
            found.append((term, n))
    return hits, found

def mini_score_dimension(text, dim_key, norm=6):
    lex = LEXICON_MINI[dim_key]
    pos_hits, pos_found = count_hits(text, lex["pos"])
    neg_hits, neg_found = count_hits(text, lex["neg"])
    raw = pos_hits - neg_hits
    theta = float(np.clip(raw / norm, -2, 2))  # simple scaled score
    return {
        "theta": theta,
        "pos_hits": pos_hits,
        "neg_hits": neg_hits,
        "pos_found": pos_found,
        "neg_found": neg_found
    }

def mini_score_all_dimensions(text):
    out = {}
    for d in LEXICON_MINI.keys():
        out[d] = mini_score_dimension(text, d)
    return out


In [379]:
answers = {
    "Q1": "PASTE YOUR Q1 ANSWER HERE",
    "Q2": "PASTE YOUR Q2 ANSWER HERE",
    "Q3": "PASTE YOUR Q3 ANSWER HERE",
}


In [380]:
# Merge answers as one text block (you can also score per question)
full_text = "\n\n".join([f"{k}: {v}" for k, v in answers.items()])

mini = mini_score_all_dimensions(full_text)

rows = []
for d in DIM_ORDER:
    poles = DIM_POLES_EN[d]
    theta = mini[d]["theta"]
    dom = poles["pos"] if theta > 0 else poles["neg"] if theta < 0 else "Balanced"
    rows.append({
        "dimension": d,
        "theta_mini": round(theta, 3),
        "dominant_pole": dom,
        "pos_hits": mini[d]["pos_hits"],
        "neg_hits": mini[d]["neg_hits"],
        "pos_terms_found": mini[d]["pos_found"][:5],
        "neg_terms_found": mini[d]["neg_found"][:5],
    })

df_mini = pd.DataFrame(rows)
df_mini


,dimension,theta_mini,dominant_pole,pos_hits,neg_hits,pos_terms_found,neg_terms_found
0,D1_GO,0.0,Balanced,0,0,[],[]
1,D2_IDED,0.0,Balanced,0,0,[],[]
2,D3_ES,0.0,Balanced,0,0,[],[]
3,D4_OS,0.0,Balanced,0,0,[],[]
4,D5_EJ,0.0,Balanced,0,0,[],[]
5,D6_PP,-0.5,Parochial Identity,0,3,[],"[(here, 3)]"


## LLM Integration

In [381]:
# =========================================================
# MOCK LLM OUTPUT (JSON-style dict)
# =========================================================

llm_output_mock = {
    "D1_GO": {
        "pole": "Goal-Oriented",
        "confidence": 0.85,
        "evidence": [
            "La prioridad fue el resultado: lanzamos una versión funcional",
            "para no bloquear el despliegue"
        ],
        "tags": ["outcome_priority", "deadline_pressure", "process_tradeoff"]
    },
    "D2_IDED": {
        "pole": "Internally Driven",
        "confidence": 0.90,
        "evidence": [
            "me mueve la satisfacción de ver que lo que construyo funciona",
            "me gusta tener autonomía y que confíen en mi criterio profesional"
        ],
        "tags": ["autonomy", "intrinsic_motivation", "mastery"]
    },
    "D3_ES": {
        "pole": "Easygoing / Flexible",
        "confidence": 0.75,
        "evidence": [
            "fui flexible con nuestra estructura habitual",
            "documentar esos atajos"
        ],
        "tags": ["flexibility", "pragmatism", "guardrails"]
    },
    "D4_OS": {
        "pole": "Open System",
        "confidence": 0.88,
        "evidence": [
            "si los usuarios dicen que la interfaz es confusa, la cambio",
            "uso el feedback para pivotar rápido"
        ],
        "tags": ["user_feedback", "external_signals"]
    },
    "D5_EJ": {
        "pole": "People-Oriented",
        "confidence": 0.70,
        "evidence": [
            "un equipo quemado comete errores caros",
            "prefiero sentarme con el PM y ser honesto"
        ],
        "tags": ["wellbeing", "psychological_safety"]
    },
    "D6_PP": {
        "pole": "Professional Identity",
        "confidence": 0.92,
        "evidence": [
            "cumple con los estándares de la industria",
            "hacer las cosas bien por ética profesional"
        ],
        "tags": ["professional_standards", "ethics"]
    }
}


In [382]:
llm_output_mock.keys()


dict_keys(['D1_GO', 'D2_IDED', 'D3_ES', 'D4_OS', 'D5_EJ', 'D6_PP'])

In [383]:
rows = []
for dim, info in llm_output_mock.items():
    rows.append({
        "dimension": dim,
        "pole": info["pole"],
        "confidence": info["confidence"],
        "evidence_count": len(info["evidence"]),
        "tags": ", ".join(info["tags"])
    })

df_llm = pd.DataFrame(rows)
df_llm


,dimension,pole,confidence,evidence_count,tags
0,D1_GO,Goal-Oriented,0.85,2,"outcome_priority, deadline_pressure, process_t..."
1,D2_IDED,Internally Driven,0.90,2,"autonomy, intrinsic_motivation, mastery"
2,D3_ES,Easygoing / Flexible,0.75,2,"flexibility, pragmatism, guardrails"
3,D4_OS,Open System,0.88,2,"user_feedback, external_signals"
4,D5_EJ,People-Oriented,0.70,2,"wellbeing, psychological_safety"
5,D6_PP,Professional Identity,0.92,2,"professional_standards, ethics"


In [384]:
def fuse_llm_and_lexicon(lexicon_scores, llm_json):
    """
    Combina:
    - señales cuantitativas (lexicon)
    - señales semánticas (LLM)
    Devuelve θ final + nivel de confianza
    """


In [385]:
# =========================================================
# POLE → SIGN MAP (for hybrid theta)
# =========================================================

POLE_SIGN = {
    "D1_GO": {
        "Goal-Oriented": +1,
        "Means-Oriented": -1,
    },
    "D2_IDED": {
        "Internally Driven": +1,
        "Externally Driven": -1,
    },
    "D3_ES": {
        "Strict / Structured": +1,
        "Easygoing / Flexible": -1,
        "Strict": +1,
        "Easygoing": -1,
    },
    "D4_OS": {
        "Open System": +1,
        "Closed System": -1,
    },
    "D5_EJ": {
        "Task-Oriented": +1,
        "People-Oriented": -1,
    },
    "D6_PP": {
        "Professional Identity": +1,
        "Parochial Identity": -1,
        "Parochial": -1,
        "Professional": +1,
    }
}

def pole_to_sign(dim, pole_label):
    if dim not in POLE_SIGN:
        return 0
    return POLE_SIGN[dim].get(pole_label, 0)


In [421]:
# =========================================================
# HYBRID FUSION: Lexicon + LLM(JSON) → theta
# =========================================================

def fuse_one_dimension(dim, lex_info, llm_info=None, w_min=0.35, w_max=0.85):
    """
    dim: "D1_GO"... 
    lex_info: output from mini_score_dimension or mini_score_all_dimensions[dim]
    llm_info: llm_output_mock[dim] dict with pole/confidence/evidence/tags
    w_min/w_max: clamps how much LLM can dominate (keeps interpretability)
    """
    theta_lex = float(lex_info["theta"])
    pos_hits = int(lex_info["pos_hits"])
    neg_hits = int(lex_info["neg_hits"])

    # default (no llm)
    result = {
        "theta_lex": theta_lex,
        "theta_llm": np.nan,
        "theta_hybrid": theta_lex,
        "llm_conf": 0.0,
        "lex_hits_total": pos_hits + neg_hits,
        "llm_pole": None,
        "evidence": [],
        "tags": [],
        "notes": "Lexicon-only"
    }

    if llm_info is None:
        return result

    pole = llm_info.get("pole", None)
    conf = float(llm_info.get("confidence", 0.0))
    evidence = llm_info.get("evidence", []) or []
    tags = llm_info.get("tags", []) or []

    sign = pole_to_sign(dim, pole)
    if sign == 0:
        # unknown label: fallback to lexicon
        result.update({
            "llm_conf": conf,
            "llm_pole": pole,
            "evidence": evidence,
            "tags": tags,
            "notes": "LLM label not recognized → Lexicon-only"
        })
        return result

    # Map confidence [0..1] → LLM theta magnitude [0..2]
    theta_llm = float(np.clip(sign * (2.0 * conf), -2, 2))

    # Weight of LLM based on confidence (clamped)
    w = float(np.clip(conf, 0, 1))
    w = w_min + (w_max - w_min) * w

    theta_h = float(np.clip(w * theta_llm + (1 - w) * theta_lex, -2, 2))

    # Heuristic uncertainty flag: low evidence or low hits
    low_lex_signal = (pos_hits + neg_hits) < 2
    low_llm_conf = conf < 0.55
    if low_lex_signal and low_llm_conf:
        notes = "Low signal: consider follow-up question"
    elif low_llm_conf:
        notes = "LLM low confidence: lexicon weighted more"
    elif low_lex_signal:
        notes = "Lexicon low hits: LLM weighted more"
    else:
        notes = "Hybrid OK"

    result.update({
        "theta_llm": theta_llm,
        "theta_hybrid": theta_h,
        "llm_conf": conf,
        "llm_pole": pole,
        "evidence": evidence,
        "tags": tags,
        "notes": notes
    })
    return result

def fused_to_evidence_light(fused: dict) -> dict:
    """
    Adapter: fuse_one_dimension output -> render_evidence_light_md expected schema
    Expected by renderer:
      { pole, confidence, evidence, lex_hits_total, notes }
    """
    if fused is None:
        return {}

    return {
        "pole": fused.get("llm_pole", None),
        "confidence": float(fused.get("llm_conf", 0.0) or 0.0),
        "evidence": fused.get("evidence", []) or [],
        "lex_hits_total": int(fused.get("lex_hits_total", 0) or 0),
        "notes": fused.get("notes", "") or ""
    }

def fuse_all_dimensions(text, llm_json):
    """
    Returns:
      - df with per-dimension summary
      - person_thetas dict ready for report builder
      - evidence_by_dim dict ready for dimension_block_en(evidence_light=None...)
    """
    mini = mini_score_all_dimensions(text)

    rows = []
    person_thetas = {}
    evidence_by_dim = {}

    for dim in ["D1_GO","D2_IDED","D3_ES","D4_OS","D5_EJ","D6_PP"]:
        fused = fuse_one_dimension(dim, mini[dim], llm_json.get(dim))
        person_thetas[dim] = fused["theta_hybrid"]

        # ✅ Evidence pack (light) per dimension
        evidence_by_dim[dim] = fused_to_evidence_light(fused)

        rows.append({
            "dimension": dim,
            "theta_lex": round(fused["theta_lex"], 3),
            "theta_llm": None if pd.isna(fused["theta_llm"]) else round(fused["theta_llm"], 3),
            "theta_hybrid": round(fused["theta_hybrid"], 2),
            "llm_conf": round(fused["llm_conf"], 3),
            "lex_hits_total": fused["lex_hits_total"],
            "llm_pole": fused["llm_pole"],
            "evidence_n": len(fused["evidence"]),
            "notes": fused["notes"]
        })

    df = pd.DataFrame(rows)
    return df, person_thetas, evidence_by_dim



In [494]:
answers = {
    "Q1": """Hace poco tuvimos que sacar una actualización crítica de seguridad antes de un fin de semana largo...""",
    "Q2": """Lo que me mueve es la satisfacción de ver que lo que construyo funciona y ayuda a alguien...""",
    "Q3": """Si el equipo está al límite, prefiero sentarme con el Project Manager y ser honesto..."""
}

full_text = "\n\n".join([f"{k}: {v}" for k,v in answers.items()])

df_hybrid, person_thetas_hybrid, evidence_by_dim = fuse_all_dimensions(full_text, llm_output_mock)

df_hybrid


,dimension,theta_lex,theta_llm,theta_hybrid,llm_conf,lex_hits_total,llm_pole,evidence_n,notes
0,D1_GO,0.0,1.70,1.32,0.85,0,Goal-Oriented,2,Lexicon low hits: LLM weighted more
1,D2_IDED,0.0,1.80,1.44,0.90,0,Internally Driven,2,Lexicon low hits: LLM weighted more
2,D3_ES,0.0,-1.50,-1.09,0.75,0,Easygoing / Flexible,2,Lexicon low hits: LLM weighted more
3,D4_OS,0.0,1.76,1.39,0.88,0,Open System,2,Lexicon low hits: LLM weighted more
4,D5_EJ,0.0,-1.40,-0.98,0.70,0,People-Oriented,2,Lexicon low hits: LLM weighted more
5,D6_PP,0.0,1.84,1.49,0.92,0,Professional Identity,2,Lexicon low hits: LLM weighted more


In [389]:
person_thetas_hybrid


{'D1_GO': 1.3175,
 'D2_IDED': 1.4400000000000002,
 'D3_ES': -1.0875,
 'D4_OS': 1.3904,
 'D5_EJ': -0.9799999999999999,
 'D6_PP': 1.4904000000000002}

## Report Builder

In [390]:
# ==========================================
# RULE ENGINE v1.0 — IF/ELSE AUTOMATION
# Plantillas -> Reglas automáticas
# ==========================================

import numpy as np

# ---------
# Helpers
# ---------
def gap_percentage(theta_p, theta_o, max_range=2.0):
    return abs(theta_p - theta_o) / max_range * 100

def compatibility_level(abs_diff):
    # 5 niveles (granular)
    if abs_diff < 0.20:
        return "muy alta compatibilidad"
    elif abs_diff < 0.35:
        return "alta compatibilidad"
    elif abs_diff < 0.55:
        return "compatibilidad moderada"
    elif abs_diff < 0.75:
        return "compatibilidad baja-moderada"
    else:
        return "baja compatibilidad"

def frame_type(abs_diff):
    # Clasificación para frases de “marcos cognitivos”
    if abs_diff < 0.20:
        return "similares"
    elif abs_diff < 0.55:
        return "parcialmente_divergentes"
    else:
        return "opuestos"

def pole_from_theta(theta, dim_cfg):
    """
    Define hacia qué polo cae un θ.
    Si theta_positive_maps_to = 'B', entonces θ>=0 => Polo B.
    Si theta_positive_maps_to = 'A', entonces θ>=0 => Polo A.
    """
    pos_map = dim_cfg.get("theta_positive_maps_to", "B")
    if theta >= 0:
        return pos_map
    return "A" if pos_map == "B" else "B"

def pole_label(theta, dim_cfg):
    pk = pole_from_theta(theta, dim_cfg)
    return dim_cfg["poles"][pk]["label"]

def opposite_key(k):
    return "B" if k == "A" else "A"

# -----------------------------------
# RULES: Diagnóstico (Plantilla 1)
# -----------------------------------
def rule_diagnosis(dim_cfg, theta_p, theta_o):
    signed = float(theta_p - theta_o)
    absd = abs(signed)
    gap_pct = gap_percentage(theta_p, theta_o)
    level = compatibility_level(absd)
    frame = frame_type(absd)

    p_pole = pole_label(theta_p, dim_cfg)
    o_pole = pole_label(theta_o, dim_cfg)

    if frame == "similares":
        frame_text = "marcos cognitivos similares"
    elif frame == "parcialmente_divergentes":
        frame_text = "marcos cognitivos parcialmente divergentes"
    else:
        frame_text = "marcos cognitivos opuestos"

    # Dirección interpretativa (quién enfatiza más)
    if absd < 0.15:
        direction_text = "La alineación es cercana y no se esperan tensiones relevantes por esta dimensión."
    else:
        if signed > 0:
            direction_text = "El individuo enfatiza este aspecto en mayor medida que la organización."
        else:
            direction_text = "La organización enfatiza este aspecto en mayor medida que el individuo."

    return {
        "signed_diff": signed,
        "abs_diff": absd,
        "gap_pct": gap_pct,
        "compatibility_level": level,
        "frame_type": frame,
        "frame_text": frame_text,
        "person_pole": p_pole,
        "org_pole": o_pole,
        "direction_text": direction_text
    }

# -----------------------------------
# RULES: Implicaciones prácticas (Plantilla 5)
# -----------------------------------
def rule_implications(dim_cfg, diag):
    level = diag["compatibility_level"]
    frame = diag["frame_type"]
    dim_name = dim_cfg["name"]

    # 1) Desempeño
    if frame == "similares":
        perf = (f"En **{dim_name}**, la alta alineación favorece consistencia en la toma de decisiones y en la ejecución del rol.")
    elif frame == "parcialmente_divergentes":
        perf = (f"En **{dim_name}**, la diferencia puede influir en criterios de prioridad y evaluación del trabajo. "
                f"Puede requerirse clarificación temprana de expectativas para evitar interpretaciones distintas.")
    else:
        perf = (f"En **{dim_name}**, el desajuste podría afectar de forma más visible la coordinación y el desempeño, "
                f"especialmente en decisiones bajo presión o ambigüedad. Requiere gestión explícita.")

    # 2) Onboarding / integración cultural
    if level in ("muy alta compatibilidad", "alta compatibilidad"):
        onboard = (f"En onboarding, conviene **reforzar y mantener** expectativas ya alineadas en **{dim_name}**, "
                   f"con ejemplos concretos del rol.")
    elif level == "compatibilidad moderada":
        onboard = (f"En onboarding, se recomienda **explicitar** cómo se traduce **{dim_name}** en prácticas del equipo "
                   f"(criterios de éxito, ejemplos, límites, casos típicos).")
    else:
        onboard = (f"En onboarding, se recomienda un abordaje **estructurado y explícito** sobre **{dim_name}**, "
                   f"incluyendo acuerdos prácticos (qué se espera, qué se valora, qué se evita).")

    # 3) Comunicación / relaciones
    if frame == "similares":
        comm = ("La comunicación tiende a ser fluida porque ambas partes usan marcos interpretativos parecidos.")
    elif frame == "parcialmente_divergentes":
        comm = ("Puede haber fricciones leves por diferencias en criterios implícitos. "
                "Conviene hacer explícitos supuestos (qué significa ‘bien hecho’, ‘prioritario’, ‘aceptable’).")
    else:
        comm = ("Es más probable que surjan fricciones por interpretaciones opuestas. "
                "Se recomienda acordar lenguaje común, criterios y ejemplos conductuales observables.")

    # 4) Riesgo de fricción / rotación (warning, pero académico)
    if level in ("muy alta compatibilidad", "alta compatibilidad"):
        risk = ("Riesgo de fricción bajo por esta dimensión, salvo cambios contextuales o expectativas no comunicadas.")
    elif level == "compatibilidad moderada":
        risk = ("Riesgo de fricción moderado: puede aparecer en momentos de evaluación, presión o cambios. "
                "Mitigable con clarificación temprana.")
    else:
        risk = ("Riesgo de fricción más elevado si no se gestiona: podría traducirse en desgaste motivacional o conflicto de expectativas. "
                "Se recomienda monitoreo y realineación periódica.")

    return {
        "performance": perf,
        "onboarding": onboard,
        "communication": comm,
        "risk": risk
    }

# -----------------------------------
# RULES: Recomendaciones accionables (Plantilla 6)
# -----------------------------------
def rule_recommendations(dim_cfg, diag):
    level = diag["compatibility_level"]
    frame = diag["frame_type"]
    dim_name = dim_cfg["name"]

    # Onboarding actions
    if level in ("muy alta compatibilidad", "alta compatibilidad"):
        r_onboard = (f"Reforzar prácticas ya alineadas en **{dim_name}** y documentar ejemplos del “estándar esperado”.")
    elif level == "compatibilidad moderada":
        r_onboard = (f"Incluir una sección explícita en onboarding sobre **{dim_name}**: criterios, ejemplos, "
                     f"y señales de éxito en el rol.")
    else:
        r_onboard = (f"Diseñar onboarding específico para **{dim_name}** con acuerdos prácticos y checkpoints en semanas 2–4.")

    # Coaching actions
    if frame == "similares":
        r_coach = "Coaching ligero: feedback temprano para consolidar criterios compartidos."
    elif frame == "parcialmente_divergentes":
        r_coach = ("Mentoring o coaching focalizado: revisar decisiones reales y alinear el marco de referencia "
                   "con ejemplos concretos.")
    else:
        r_coach = ("Acompañamiento estructurado: sesiones recurrentes para calibrar expectativas y prevenir fricción, "
                   "especialmente al inicio del rol.")

    # Role/expectations
    if level in ("muy alta compatibilidad", "alta compatibilidad"):
        r_role = "Mantener el diseño del rol. Asegurar que prioridades y criterios se mantengan explícitos."
    elif level == "compatibilidad moderada":
        r_role = ("Clarificar límites de autonomía, prioridades y criterios de evaluación vinculados a esta dimensión.")
    else:
        r_role = ("Considerar ajustes de alcance, autonomía o forma de evaluación; definir criterios explícitos y medibles.")

    # Risks to monitor
    if level in ("muy alta compatibilidad", "alta compatibilidad"):
        r_risk = "Monitorear cambios contextuales que puedan alterar expectativas sin comunicación explícita."
    elif level == "compatibilidad moderada":
        r_risk = ("Monitorear señales de desalineación interpretativa (feedback repetido, frustración, divergencias de criterio).")
    else:
        r_risk = ("Monitorear fricción recurrente, señales de desgaste o conflicto de expectativas; intervenir temprano.")

    return {
        "onboarding": r_onboard,
        "coaching": r_coach,
        "role": r_role,
        "risk_monitoring": r_risk
    }

# -----------------------------------
# MASTER: Ejecutar reglas por dimensión
# -----------------------------------
def build_dimension_outputs(dim_key, theta_p, theta_o, DIMENSIONS):
    dim_cfg = DIMENSIONS[dim_key]
    diag = rule_diagnosis(dim_cfg, theta_p, theta_o)
    impl = rule_implications(dim_cfg, diag)
    recs = rule_recommendations(dim_cfg, diag)

    # Bloques conceptuales (no-if): vienen del diccionario
    # (sirven para completar el reporte con contenido estable)
    p_dom_key = pole_from_theta(theta_p, dim_cfg)  # polo dominante de la persona
    p_opp_key = opposite_key(p_dom_key)

    cognitive_block = {
        "dominant_label": dim_cfg["poles"][p_dom_key]["label"],
        "dominant_items": dim_cfg["cognitive"][p_dom_key],
        "opposite_label": dim_cfg["poles"][p_opp_key]["label"],
        "opposite_items": dim_cfg["cognitive"][p_opp_key]
    }

    behaviors_block = dim_cfg["behaviors_vs"]  # estable VS
    softskills_block = dim_cfg["softskills"]   # estable

    return {
        "dim_key": dim_key,
        "dim_name": dim_cfg["name"],
        "definition": dim_cfg["definition"],
        "diagnosis": diag,
        "cognitive": cognitive_block,
        "behaviors_vs": behaviors_block,
        "softskills": softskills_block,
        "implications": impl,
        "recommendations": recs
    }

def build_report_outputs(person_thetas, org_thetas, DIMENSIONS, order):
    outputs = []
    for dk in order:
        outputs.append(build_dimension_outputs(dk, float(person_thetas[dk]), float(org_thetas[dk]), DIMENSIONS))
    return outputs


In [ ]:
# =========================================================
# LANGUAGE PACK v1.0 — English templates for the Fit Report
# Uses: Fit + Compatibility (as requested)
# =========================================================

TEXT_TEMPLATES = {
    "EN": {
        "report_title": "Cognitive–Cultural Fit Assessment Report",

        "sections": {
            "identification": "Report Identification",
            "executive_summary": "Executive Summary",
            "overall_fit": "Overall Fit Result",
            "dimension_results": "Dimension-Level Results",
        },

        "executive_summary": (
            "This report presents an automated cognitive–cultural fit assessment between an individual and an organizational "
            "profile. The analysis relies on a theory-driven, multi-dimensional model grounded in Cognitive Science and "
            "Organizational Psychology. Results highlight areas of alignment, partial mismatch, and potential sources of "
            "cognitive or behavioral friction, offering actionable insights for onboarding and integration."
        ),

        "overall_fit_block": (
            "**Overall Cognitive–Cultural Fit:** {overall_fit_level}.\n\n"
            "This result reflects the aggregated degree of alignment across six cognitive–cultural dimensions, considering "
            "both magnitude and direction of differences in latent cultural orientations."
        ),

        # ---------- Dimension block (full) ----------
        "dimension_block": (
            "## {dim_name}\n\n"
            "### Compatibility Diagnosis\n"
            "Compatibility is **{compat_level}**, with an absolute difference of **{gap_abs:.2f}** (**{gap_pct:.0f}%**).\n"
            "The individual tends toward **{person_pole}** (θ = {theta_person:.2f}), while the organization tends toward "
            "**{org_pole}** (θ = {theta_org:.2f}).\n"
            "{compat_sentence}\n\n"
            "### Conceptual Definition\n"
            "{definition}\n\n"
            "### Cognitive Attributes\n"
            "{cognitive_block}\n\n"
            "### Observable Cultural Behaviors\n"
            "{behaviors_block}\n\n"
            "### Practical Implications\n"
            "{implications_block}\n\n"
            "### Actionable Recommendations\n"
            "{recommendations_block}\n"
        ),

        # ---------- Pole-aware helpers ----------
        "compat_sentence": {
            "aligned": "This pattern suggests broadly aligned cognitive–cultural frames in this dimension.",
            "partial": "This pattern suggests partially divergent cognitive–cultural frames that may require calibration.",
            "opposing": "This pattern suggests opposing cognitive–cultural frames and may increase friction if not managed."
        },

        "cognitive_header_A": "When **{poleA}** predominates, individuals tend to:",
        "cognitive_header_B": "When **{poleB}** predominates, individuals tend to:",

        "behaviors_header_A": "**{poleA}** contexts often display:",
        "behaviors_header_B": "**{poleB}** contexts often display:",

        "implications_default": (
            "This configuration may influence onboarding dynamics, daily coordination, and how feedback and expectations "
            "are interpreted. Without explicit alignment, differences may increase ambiguity or perceived misfit during "
            "early stages of integration."
        ),

        "recommendations_default": [
            "**Onboarding:** Make expectations explicit (standards, feedback norms, priorities).",
            "**Role design:** Align autonomy/structure and decision criteria with the dominant orientation.",
            "**Monitoring:** Track early signs of friction and recalibrate norms through coaching and explicit agreements."
        ],

        "closing_note": (
            "*This assessment is intended as a decision-support tool and should not be interpreted as a deterministic "
            "evaluation of individual performance or potential.*"
        )
    }
}

# =========================================================
# Helper functions to build EN text blocks from DIMENSIONS
# =========================================================

def _format_bullets(items):
    return "\n".join([f"- {x}" for x in items])

def _build_cognitive_block_en(dim_cfg):
    poleA = dim_cfg["poles"]["A"]["label"]
    poleB = dim_cfg["poles"]["B"]["label"]

    a_lines = [f"- {x['desc']}" if isinstance(x, dict) else f"- {x}" for x in dim_cfg["cognitive"]["A"]]
    b_lines = [f"- {x['desc']}" if isinstance(x, dict) else f"- {x}" for x in dim_cfg["cognitive"]["B"]]

    headerA = TEXT_TEMPLATES["EN"]["cognitive_header_A"].format(poleA=poleA, poleB=poleB)
    headerB = TEXT_TEMPLATES["EN"]["cognitive_header_B"].format(poleA=poleA, poleB=poleB)

    return (
        f"{headerA}\n" + "\n".join(a_lines) + "\n\n" +
        f"{headerB}\n" + "\n".join(b_lines)
    )

def _build_behaviors_block_en(dim_cfg):
    poleA = dim_cfg["poles"]["A"]["label"]
    poleB = dim_cfg["poles"]["B"]["label"]

    headerA = TEXT_TEMPLATES["EN"]["behaviors_header_A"].format(poleA=poleA, poleB=poleB)
    headerB = TEXT_TEMPLATES["EN"]["behaviors_header_B"].format(poleA=poleA, poleB=poleB)

    a_lines = _format_bullets(dim_cfg["behaviors_vs"]["A"])
    b_lines = _format_bullets(dim_cfg["behaviors_vs"]["B"])

    return f"{headerA}\n{a_lines}\n\n{headerB}\n{b_lines}"

def _infer_compat_bucket(gap_abs, thresholds=(0.25, 0.60)):
    # You can keep your existing compatibility bucketing if you already have it.
    # This is a sensible default.
    lo, hi = thresholds
    if gap_abs <= lo:
        return "aligned"
    if gap_abs <= hi:
        return "partial"
    return "opposing"

def generate_dimension_text_en(
    dim_key,
    person_theta,
    org_theta,
    dim_cfg,
    compat_level="Moderate",
    gap_abs=0.0,
    gap_pct=0.0,
    thresholds=(0.25, 0.60),
    implications_override=None,
    recommendations_override=None
):
    # Poles (direction)
    # (Assumes you already standardized theta via z-score BEFORE generating the report.)
    pos_map = dim_cfg.get("theta_positive_maps_to", "B")

    def pole_for(theta):
        if theta >= 0:
            return pos_map
        return "A" if pos_map == "B" else "B"

    kp = pole_for(person_theta)
    ko = pole_for(org_theta)

    person_pole = dim_cfg["poles"][kp]["short"]
    org_pole = dim_cfg["poles"][ko]["short"]

    # Sentence bucket
    bucket = _infer_compat_bucket(gap_abs, thresholds=thresholds)
    compat_sentence = TEXT_TEMPLATES["EN"]["compat_sentence"][bucket]

    cognitive_block = _build_cognitive_block_en(dim_cfg)
    behaviors_block = _build_behaviors_block_en(dim_cfg)

    implications_block = implications_override or TEXT_TEMPLATES["EN"]["implications_default"]
    recs = recommendations_override or TEXT_TEMPLATES["EN"]["recommendations_default"]
    recommendations_block = "\n".join([f"- {r}" for r in recs])

    return TEXT_TEMPLATES["EN"]["dimension_block"].format(
        dim_name=dim_cfg["name"],
        compat_level=compat_level,
        gap_abs=float(gap_abs),
        gap_pct=float(gap_pct),
        person_pole=person_pole,
        org_pole=org_pole,
        theta_person=float(person_theta),
        theta_org=float(org_theta),
        compat_sentence=compat_sentence,
        definition=dim_cfg["definition"],
        cognitive_block=cognitive_block,
        behaviors_block=behaviors_block,
        implications_block=implications_block,
        recommendations_block=recommendations_block
    )

# =========================================================
# Glue: Generate a full EN markdown report using your thetas
# (You can plug this into your existing export-to-docx pipeline)
# =========================================================

def generate_full_report_md_EN(
    person_thetas,
    org_thetas,
    DIMENSIONS,
    order,
    meta=None,
    overall_fit_level="High",
    thresholds=(0.25, 0.60)
):
    meta = meta or {}
    title = TEXT_TEMPLATES["EN"]["report_title"]

    person_id = meta.get("person_id", "P-UNKNOWN")
    org_id = meta.get("organization_id", "O-UNKNOWN")
    model_version = meta.get("model_version", "v1.2")
    instrument = meta.get("instrument_version", "P-v1.0")
    date = meta.get("date", "")

    md = []
    md.append(f"# {title}\n")

    md.append("## Report Identification\n")
    md.append(
        f"- **Person ID:** {person_id}\n"
        f"- **Organization ID:** {org_id}\n"
        f"- **Model Version:** {model_version}\n"
        f"- **Instrument:** {instrument}\n"
        + (f"- **Date:** {date}\n" if date else "")
        + "\n"
    )

    md.append("## Executive Summary\n")
    md.append(TEXT_TEMPLATES["EN"]["executive_summary"] + "\n\n")

    md.append("## Overall Fit Result\n")
    md.append(TEXT_TEMPLATES["EN"]["overall_fit_block"].format(overall_fit_level=overall_fit_level) + "\n\n")

    md.append("## Dimension-Level Results\n")

    for dk in order:
        dim_cfg = DIMENSIONS[dk]
        tp = float(person_thetas[dk])
        to = float(org_thetas[dk])
        gap_abs = abs(tp - to)
        # A simple interpretable % (you can replace with your own if you already compute it)
        gap_pct = min(100.0, (gap_abs / 2.0) * 100.0)

        # You likely already compute compat_level elsewhere; keeping a simple mapping:
        if gap_abs <= thresholds[0]:
            compat_level = "High"
        elif gap_abs <= thresholds[1]:
            compat_level = "Moderate"
        else:
            compat_level = "Low"

        md.append(
            generate_dimension_text_en(
                dim_key=dk,
                person_theta=tp,
                org_theta=to,
                dim_cfg=dim_cfg,
                compat_level=compat_level,
                gap_abs=gap_abs,
                gap_pct=gap_pct,
                thresholds=thresholds
            )
        )
        md.append("\n")

    md.append(TEXT_TEMPLATES["EN"]["closing_note"] + "\n")
    return "\n".join(md)

# =========================================================
# Example call (use your z-scored thetas)
# =========================================================
# order = ["D1_GO","D2_IDED","D3_ES","D4_OS","D5_EJ","D6_PP"]
# person_z = zscore_dict(person_thetas, order)
# org_z    = zscore_dict(org_thetas, order)
#
# meta = {
#   "person_id":"P-DEMO-01",
#   "organization_id":"O-DEMO-01",
#   "model_version":"v1.2",
#   "instrument_version":"P-v1.0",
#   "date":""
# }
#
# md_en = generate_full_report_md_EN(person_z, org_z, DIMENSIONS, order, meta=meta, overall_fit_level="High")
# print(md_en[:2000])


In [391]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
import subprocess
from datetime import datetime

# -----------------------------
# 1) Radar chart (smaller, readable)
# -----------------------------
def radar_chart(values, labels, out_png, title=None):
    """
    values: list/array length N
    labels: list length N
    out_png: output path .png
    """
    values = np.array(values, dtype=float)
    n = len(values)

    # close the polygon
    angles = np.linspace(0, 2*np.pi, n, endpoint=False).tolist()
    angles += angles[:1]
    vals = values.tolist()
    vals += vals[:1]

    # Smaller figure + better label spacing
    fig = plt.figure(figsize=(6.2, 6.2), dpi=160)
    ax = plt.subplot(111, polar=True)

    ax.plot(angles, vals, linewidth=2)
    ax.fill(angles, vals, alpha=0.15)

    # Labels
    ax.set_xticks(np.linspace(0, 2*np.pi, n, endpoint=False))
    ax.set_xticklabels(labels, fontsize=9)
    ax.tick_params(pad=10)

    # Optional: set a symmetric range for visual consistency
    # You can adjust if your z-scored thetas typically fall outside [-2, 2]
    ax.set_ylim(-2.5, 2.5)

    # Grid styling (keep minimal)
    ax.grid(True, alpha=0.3)

    if title:
        plt.title(title, y=1.08, fontsize=12)

    plt.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)

    return out_png


# -----------------------------
# 2) Default English report generator
# (uses the EN functions you already pasted)
# -----------------------------
def generate_report_md_default_EN(
    person_thetas,
    org_thetas,
    DIMENSIONS,
    order,
    out_dir="outputs",
    out_basename="fit_report_en",
    overall_fit_level="High",
    thresholds=(0.25, 0.60),
    meta=None,
    include_radar=True
):
    os.makedirs(out_dir, exist_ok=True)

    meta = meta or {}
    meta = {
        "person_id": meta.get("person_id", "P-DEMO-01"),
        "organization_id": meta.get("organization_id", "O-DEMO-01"),
        "model_version": meta.get("model_version", "v1.2"),
        "instrument_version": meta.get("instrument_version", "P-v1.0"),
        "date": meta.get("date", datetime.now().strftime("%Y-%m-%d")),
    }

    radar_png = None
    if include_radar:
        # Build labels from DIMENSIONS names
        labels = [DIMENSIONS[k]["name"] for k in order]
        vals = [float(person_thetas[k]) for k in order]  # person profile radar (you can add org too later)

        radar_png = os.path.join(out_dir, f"{out_basename}_radar.png")
        radar_chart(
            values=vals,
            labels=labels,
            out_png=radar_png,
            title="Cognitive–Cultural Profile (θ, z-scored)"
        )

    # Generate markdown (English default)
    md = generate_full_report_md_EN(
        person_thetas=person_thetas,
        org_thetas=org_thetas,
        DIMENSIONS=DIMENSIONS,
        order=order,
        meta=meta,
        overall_fit_level=overall_fit_level,
        thresholds=thresholds
    )

    # If radar exists, inject near the top (after identification)
    if radar_png:
        radar_rel = os.path.basename(radar_png)
        inject = f"\n\n## Radar Chart\n\n![]({radar_rel})\n\n"
        # Insert after "Report Identification" block by simple heuristic:
        marker = "## Executive Summary"
        if marker in md:
            md = md.replace(marker, inject + marker, 1)
        else:
            md = md + inject

    md_path = os.path.join(out_dir, f"{out_basename}.md")
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(md)

    return {"md": md_path, "radar_png": radar_png, "meta": meta}


# -----------------------------
# 3) Pandoc export: MD -> DOCX
# -----------------------------
def export_md_to_docx(md_path, out_docx_path):
    """
    Requires pandoc installed and available in PATH.
    """
    cmd = ["pandoc", md_path, "-o", out_docx_path]
    proc = subprocess.run(cmd, capture_output=True, text=True)

    if proc.returncode != 0:
        raise RuntimeError(
            "Pandoc export failed.\n"
            f"STDOUT:\n{proc.stdout}\n\nSTDERR:\n{proc.stderr}"
        )
    return out_docx_path


# -----------------------------
# 4) One-call pipeline (EN default)
# -----------------------------
def generate_and_export_fit_report_EN_default(
    person_thetas_raw,
    org_thetas_raw,
    DIMENSIONS,
    order=("D1_GO","D2_IDED","D3_ES","D4_OS","D5_EJ","D6_PP"),
    out_dir="outputs",
    out_basename="fit_report_en",
    overall_fit_level="High",
    thresholds=(0.25, 0.60),
    meta=None
):
    # z-score before interpreting poles (critical)
    person_z = zscore_dict(person_thetas_raw, list(order))
    org_z    = zscore_dict(org_thetas_raw, list(order))

    artifacts = generate_report_md_default_EN(
        person_thetas=person_z,
        org_thetas=org_z,
        DIMENSIONS=DIMENSIONS,
        order=list(order),
        out_dir=out_dir,
        out_basename=out_basename,
        overall_fit_level=overall_fit_level,
        thresholds=thresholds,
        meta=meta,
        include_radar=True
    )

    docx_path = os.path.join(out_dir, f"{out_basename}.docx")
    export_md_to_docx(artifacts["md"], docx_path)

    artifacts["docx"] = docx_path
    return artifacts


# -----------------------------
# 5) Example usage (plug your real thetas)
# -----------------------------
# artifacts = generate_and_export_fit_report_EN_default(
#     person_thetas_raw=person_thetas,
#     org_thetas_raw=org_thetas,
#     DIMENSIONS=DIMENSIONS,
#     order=("D1_GO","D2_IDED","D3_ES","D4_OS","D5_EJ","D6_PP"),
#     out_dir="outputs",
#     out_basename="fit_report_en_demo",
#     overall_fit_level="High",
#     thresholds=(0.25, 0.60),
#     meta={"person_id":"P-DEMO-01","organization_id":"O-DEMO-01","model_version":"v1.2"}
# )
# artifacts


In [392]:
import os
import numpy as np
import matplotlib.pyplot as plt

def radar_chart_dual(person_vals, org_vals, labels, out_png, title=None,
                     person_label="Person", org_label="Organization"):
    """
    Draws a dual radar chart (Person vs Organization).
    person_vals, org_vals: list/array length N
    labels: list length N
    out_png: output file path
    """
    person_vals = np.array(person_vals, dtype=float)
    org_vals = np.array(org_vals, dtype=float)
    n = len(labels)

    # Angles
    angles = np.linspace(0, 2*np.pi, n, endpoint=False).tolist()
    angles += angles[:1]  # close loop

    # Close the polygons
    p = person_vals.tolist() + [person_vals[0]]
    o = org_vals.tolist() + [org_vals[0]]

    fig = plt.figure(figsize=(6.4, 6.4), dpi=170)
    ax = plt.subplot(111, polar=True)

    # Plot person and org
    ax.plot(angles, p, linewidth=2, label=person_label)
    ax.fill(angles, p, alpha=0.12)

    ax.plot(angles, o, linewidth=2, label=org_label)
    ax.fill(angles, o, alpha=0.12)

    # Labels
    ax.set_xticks(np.linspace(0, 2*np.pi, n, endpoint=False))
    ax.set_xticklabels(labels, fontsize=8)
    ax.tick_params(pad=10)

    # Symmetric range for z-scored thetas (adjust if needed)
    ax.set_ylim(-2.5, 2.5)
    ax.grid(True, alpha=0.30)

    if title:
        plt.title(title, y=1.10, fontsize=12)

    # Legend outside
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False)

    # inside radar_chart_dual(...) after creating ax (polar)
    ticks = [-1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5]
    ax.set_yticks(ticks)
    ax.set_yticklabels([f"{t:g}" for t in ticks])

    # optionally tighten limits so the plot isn't too roomy
    ax.set_ylim(-1.75, 1.75)


    plt.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)
    return out_png


In [393]:
import os
import subprocess

def export_md_to_docx(md_path, out_docx_path):
    """
    Export MD -> DOCX with Pandoc, ensuring images resolve correctly
    by running pandoc in the MD directory.
    """
    md_dir = os.path.dirname(os.path.abspath(md_path))
    md_file = os.path.basename(md_path)
    out_docx_file = os.path.basename(out_docx_path)

    cmd = ["pandoc", md_file, "-o", out_docx_file]
    proc = subprocess.run(cmd, cwd=md_dir, capture_output=True, text=True)

    if proc.returncode != 0:
        raise RuntimeError(
            "Pandoc export failed.\n"
            f"CMD: {' '.join(cmd)}\n"
            f"CWD: {md_dir}\n\n"
            f"STDOUT:\n{proc.stdout}\n\nSTDERR:\n{proc.stderr}"
        )

    return os.path.join(md_dir, out_docx_file)



In [395]:
# =========================================================
# MASTER DICTIONARY v1.2 — ENGLISH VERSION (DIMENSIONS_EN)
# Compatible with RULE ENGINE v1.0 + REPORT GENERATOR v1.0
# =========================================================

DIMENSIONS_EN = {
  "D1_GO": {
    "name": "Goal Orientation",
    "definition": (
      "This dimension describes the extent to which an individual or organization prioritizes the achievement of final "
      "outcomes versus adherence to processes, methods, and means. From a cognitive perspective, it reflects the dominant "
      "attentional focus: whether it is directed toward the 'how' (means) or toward the 'what' (goals). "
      "It is not a value judgment: both poles can be functional depending on context, type of work, and the level of uncertainty."
    ),
    "poles": {
      "A": {"label": "Means-Oriented", "short": "Means-Oriented"},
      "B": {"label": "Goal-Oriented", "short": "Goal-Oriented"},
    },
    "theta_positive_maps_to": "B",
    "cognitive": {
      "A": [
        {"title": "Attention to process and sequence", "desc": "Performance is mentally organized around steps, methods, and controls; procedural coherence serves as a key quality criterion."},
        {"title": "Evaluation via compliance and traceability", "desc": "Consistency of methods, documentation, and the ability to justify decisions through rules and procedures are highly valued."},
        {"title": "Preference for standardization", "desc": "Variability is reduced through repeatable practices; learning is integrated as an incremental improvement of the process."},
      ],
      "B": [
        {"title": "Attention to outcomes and impact", "desc": "Decisions are anchored in goals, deliverables, and value; the process remains flexible if it improves outcomes."},
        {"title": "Evaluation via effectiveness", "desc": "Priority is given to whether actions achieve the objective; efficiency and goal attainment operate as central criteria."},
        {"title": "Goal-driven optimization", "desc": "There is a tendency to experiment, iterate, and adjust methods based on feedback about results."},
      ]
    },
    "behaviors_vs": {
      "A": [
        "Emphasis on standards, checklists, and explicitly defined steps.",
        "Expectation of documentation and traceability (decisions justified by procedure).",
        "Changes implemented in a controlled manner, with prior validation.",
      ],
      "B": [
        "Priority on deliverables, goals, and impact metrics.",
        "Tolerance for method flexibility if it accelerates outcomes.",
        "Preference for rapid iterations and pragmatic, outcome-driven execution.",
      ]
    },
    "softskills": {
      "A": "Rigor, operational discipline, organization, attention to detail, consistency.",
      "B": "Results orientation, proactivity, pragmatism, tactical adaptability, impact focus.",
    }
  },

  "D2_IDED": {
    "name": "Internal vs external motivation",
    "definition": (
      "This dimension captures the predominant source of work motivation. It distinguishes between contexts in which action "
      "is regulated mainly by external contingencies (rewards, recognition, pressure, supervision) and contexts in which "
      "internalized or intrinsic motivation predominates (interest, personal values, purpose, and autonomy). "
      "At a cognitive level, it influences persistence, commitment, and how feedback is interpreted."
    ),
    "poles": {
      "A": {"label": "Externally driven", "short": "External motivation"},
      "B": {"label": "Internally driven", "short": "Internal motivation"},
    },
    "theta_positive_maps_to": "B",
    "cognitive": {
      "A": [
        {"title": "Regulation by contingencies", "desc": "Behavior is organized around rewards, sanctions, or external expectations; incentives guide prioritization."},
        {"title": "Feedback as external validation", "desc": "Feedback is processed as a signal of approval/acceptance; social recognition carries substantial weight."},
        {"title": "Preference for structure and direction", "desc": "Clear external guidance (imposed goals, supervision, KPIs) is valued to support effort and persistence."},
      ],
      "B": [
        {"title": "Autonomy and meaning", "desc": "Behavior is guided by values, purpose, and choice; effort is sustained even without immediate incentives."},
        {"title": "Feedback as information to improve", "desc": "Feedback is processed as learning input rather than judgment; mastery and growth are prioritized."},
        {"title": "Intrinsic persistence", "desc": "Commitment is maintained through interest and meaning; motivation remains more stable under reward changes."},
      ]
    },
    "behaviors_vs": {
      "A": [
        "Effort increases in response to rewards, bonuses, recognition, or external pressure.",
        "Frequent confirmation and clear external evaluation are sought.",
        "Tasks that are highly visible or directly rewarded tend to be prioritized.",
      ],
      "B": [
        "Effort is sustained through interest, meaning, and learning.",
        "Autonomy and ownership are actively sought.",
        "Mastery/competence and continuous improvement are valued as motivators.",
      ]
    },
    "softskills": {
      "A": "compliance orientation, responsiveness to external feedback, adaptability to supervision, incentive-driven pragmatism.",
      "B": "autonomy, sustained motivation, resilience, self-directed learning, and intrinsic responsibility.",
    }
  },

  "D3_ES": {
    "name": "Flexibility vs structure",
    "definition": (
      "This dimension describes a preference for flexible, adaptive, ambiguity-tolerant environments versus normative, "
      "structured, and highly regulated environments. Cognitively, it reflects the role of rules as context-dependent guides "
      "versus stable and obligatory frameworks. It influences how exceptions are managed, how rules are interpreted, and "
      "how organizational control is perceived."
    ),
    "poles": {
      "A": {"label": "Flexible (Easygoing)", "short": "Flexibility"},
      "B": {"label": "Structured (Strict)", "short": "Structure"},
    },
    "theta_positive_maps_to": "B",
    "cognitive": {
      "A": [
        {"title": "Tolerance for ambiguity", "desc": "Exceptions are processed as a normal part of work; the framework is adjusted according to context."},
        {"title": "Rules as flexible guidelines", "desc": "Norms are interpreted situationally; adaptability and judgment are prioritized."},
        {"title": "Preference for contextual autonomy", "desc": "Room to decide and experiment is valued without excessive formality."},
      ],
      "B": [
        {"title": "Preference for predictability", "desc": "Uncertainty is reduced through clear rules; consistency and stability are valued."},
        {"title": "Rules as standards", "desc": "Norms are interpreted literally and generalizable; individual variation is minimized."},
        {"title": "Control and compliance", "desc": "Compliance, auditing, and traceability are prioritized; exceptions require formal justification."},
      ]
    },
    "behaviors_vs": {
      "A": [
        "Rapid and pragmatic changes in response to new circumstances.",
        "Exceptions are accepted if they add value or remove blockers.",
        "Preference for informal agreements and agile coordination.",
      ],
      "B": [
        "Strict compliance with policies, processes, and standards.",
        "Exceptions require approval, evidence, or documentation.",
        "Preference for routines, defined roles, and stable procedures.",
      ]
    },
    "softskills": {
      "A": "adaptability, creativity, cognitive flexibility, tolerance for change, initiative.",
      "B": "discipline, reliability, organization, self-control, and operational consistency.",
    }
  },

  "D4_OS": {
    "name": "Open vs Closed System",
    "definition": (
      "This dimension reflects the extent to which an individual or an organization orient toward the external environment "
      "(customers, market, stakeholders, external feedback) versus an orientation centered on internal processes, stability, "
      "and boundary control. It regulates collective attention: which signals are considered relevant and how information is "
      "prioritized for decision-making."
    ),
    "poles": {
      "A": {"label": "Closed System", "short": "Closed System"},
      "B": {"label": "Open System", "short": "Open System"},
    },
    "theta_positive_maps_to": "B",
    "cognitive": {
      "A": [
        {"title": "Protection of internal coherence", "desc": "Stability, control, and consistency are prioritized; the environment is filtered to avoid noise or drift."},
        {"title": "Attention to internal rules", "desc": "Decisions are grounded in internal standards, processes, and institutional priorities."},
        {"title": "Preference for clear boundaries", "desc": "A separation between 'inside' and 'outside' is maintained; external change is adopted cautiously."},
      ],
      "B": [
        {"title": "Sensitivity to external signals", "desc": "Customer/market information is prioritized; the environment guides adaptation and innovation."},
        {"title": "Learning through feedback", "desc": "External feedback is integrated as a central input to improve processes and products."},
        {"title": "Adaptive openness", "desc": "Reconfiguration of internal practices is tolerated to respond to external demands."},
      ]
    },
    "behaviors_vs": {
      "A": [
        "Emphasis on internal procedures and operational stability.",
        "External changes are adopted slowly and with formal filters.",
        "Preference for information control and internal prioritization.",
      ],
      "B": [
        "Frequent customer/market contact and continuous learning.",
        "Iterative adjustments based on external feedback.",
        "Active exploration of trends and external benchmarking.",
      ]
    },
    "softskills": {
      "A": "reliability, internal focus, procedural thinking, prudence, consistency.",
      "B": "customer orientation, curiosity, strategic adaptability, openness to learning, initiative.",
    }
  },

  "D5_EJ": {
    "name": "People vs task orientation",
    "definition": (
      "This dimension differentiates cultures that prioritize well-being, human relationships, and socioemotional climate "
      "versus those that prioritize efficiency, performance, and task execution. Cognitively, it regulates how success is "
      "defined: by human impact and team care versus productive output and work results."
    ),
    "poles": {
      "A": {"label": "People-oriented (Employee-Oriented)", "short": "People"},
      "B": {"label": "Task-oriented (Job-Oriented)", "short": "Tasks"},
    },
    "theta_positive_maps_to": "B",
    "cognitive": {
      "A": [
        {"title": "Primacy of well-being and relationship", "desc": "Decisions consider human impact; psychological safety and climate are valued."},
        {"title": "Success as social sustainability", "desc": "Effectiveness includes cohesion, support, and development; the process matters due to interpersonal impact."},
        {"title": "Attention to socioemotional signals", "desc": "Tension, workload, and motivation are monitored; coordination is adjusted to sustain the team."},
      ],
      "B": [
        {"title": "Primacy of performance and output", "desc": "Delivery, efficiency, and attainment are prioritized; resources are organized toward results."},
        {"title": "Success as execution", "desc": "Effectiveness is defined by observable performance, productivity, and goals; well-being is managed as a facilitator."},
        {"title": "Attention to task metrics", "desc": "Progress, blockers, and work quality are monitored; coordination is oriented to throughput."},
      ]
    },
    "behaviors_vs": {
      "A": [
        "Feedback emphasizes care, support, and personal development.",
        "Decisions prioritize balance, climate, and team sustainability.",
        "A culture of collaboration and interpersonal support.",
      ],
      "B": [
        "Feedback is oriented to metrics, performance, and results.",
        "Decisions are guided by efficiency, priorities, and execution.",
        "A culture of accountability and deliverable focus.",
      ]
    },
    "softskills": {
      "A": "empathy, support, caring communication, interpersonal sensitivity, collaboration.",
      "B": "assertiveness, time management, accountability, focus, prioritization, results orientation.",
    }
  },

  "D6_PP": {
    "name": "Professional identity",
    "definition": (
      "This dimension describes the primary locus of work identity: whether the professional self is anchored mainly in the "
      "specific organization one belongs to (parochial) or in the profession, expertise, and external standards (professional). "
      "Cognitively, it influences success criteria, loyalty, development, and ethical decision frameworks."
    ),
    "poles": {
      "A": {"label": "Organizational identity (Parochial)", "short": "Parochial"},
      "B": {"label": "Professional identity (Professional)", "short": "Professional"},
    },
    "theta_positive_maps_to": "B",
    "cognitive": {
      "A": [
        {"title": "Work-self as belonging", "desc": "Identity is represented as part of an organizational 'we'; affiliation and internal cultural continuity take priority."},
        {"title": "Internal success criteria", "desc": "Alignment with internal norms, institutional expectations, and recognition within the organization is valued."},
        {"title": "Decisions anchored in internal context", "desc": "Organization-defined priorities are emphasized; external standards are subordinated to local culture."},
      ],
      "B": [
        {"title": "Work-self as expertise", "desc": "Identity is represented around the profession, technical standards, and external disciplinary communities."},
        {"title": "External quality criteria", "desc": "Alignment with best practices, industry standards, and professional reputation is valued."},
        {"title": "Decisions anchored in professional principles", "desc": "Technical consistency and professional ethics are prioritized; internal culture is evaluated against external frameworks."},
      ]
    },
    "behaviors_vs": {
      "A": [
        "Loyalty and commitment centered on the organization and its culture.",
        "Preference for internal standards and 'how we do things here'.",
        "Internal recognition as the primary validator of career trajectory.",
      ],
      "B": [
        "Loyalty and commitment centered on the profession and external standards.",
        "Use of external benchmarking and best practices as reference.",
        "External professional recognition (certifications, community) as a relevant validator.",
      ]
    },
    "softskills": {
      "A": "affiliation, internal cultural adaptability, belonging-building, contextual cooperation.",
      "B": "technical judgment, professional ethics, expertise-based autonomy, continuous disciplinary learning.",
    }
  }
}


In [396]:
DIMENSIONS = DIMENSIONS_EN


In [397]:
import os
import subprocess

def export_md_to_docx(md_path, out_docx_path):
    md_dir = os.path.dirname(os.path.abspath(md_path))
    md_file = os.path.basename(md_path)
    out_docx_file = os.path.basename(out_docx_path)

    # (Opcional) borrar docx anterior si existe
    out_abs = os.path.join(md_dir, out_docx_file)
    if os.path.exists(out_abs):
        os.remove(out_abs)

    cmd = ["pandoc", md_file, "-o", out_docx_file]
    proc = subprocess.run(cmd, cwd=md_dir, capture_output=True, text=True)

    if proc.returncode != 0:
        raise RuntimeError(
            "Pandoc export failed.\n"
            f"CMD: {' '.join(cmd)}\n"
            f"CWD: {md_dir}\n\n"
            f"STDOUT:\n{proc.stdout}\n\nSTDERR:\n{proc.stderr}"
        )

    return out_abs


In [398]:
docx_path = export_md_to_docx(
    r"C:\Users\dasuarez\outputs\fit_report_en_dual_radar.md",
    r"C:\Users\dasuarez\outputs\fit_report_en_dual_radar.docx"
)
docx_path


'C:\\Users\\dasuarez\\outputs\\fit_report_en_dual_radar.docx'

In [399]:
import numpy as np

def overall_fit_from_gaps(person_thetas, org_thetas, order, thresholds=(0.25, 0.60)):
    """
    Returns:
      overall_label: High / Moderate / Low
      avg_gap: mean(|Δθ|)
      median_gap: median(|Δθ|)
      fit_pct: 0..100 (higher is better), based on avg_gap mapped to [0,2]
    """
    gaps = []
    for dk in order:
        gaps.append(abs(float(person_thetas[dk]) - float(org_thetas[dk])))

    avg_gap = float(np.mean(gaps))
    median_gap = float(np.median(gaps))

    # Label using same thresholds as dimensions (consistent)
    if avg_gap <= thresholds[0]:
        overall_label = "High"
    elif avg_gap <= thresholds[1]:
        overall_label = "Moderate"
    else:
        overall_label = "Low"

    # Compatibility % (interpretive, not a probability):
    # assumes z-scored theta typically within [-2,2] => max abs diff ~4.
    # we cap at 2.0 for an "easier" scale (same as your per-dimension pct mapping).
    avg_gap_capped = min(2.0, avg_gap)
    fit_pct = max(0.0, 100.0 * (1.0 - (avg_gap_capped / 2.0)))

    return overall_label, avg_gap, median_gap, fit_pct


def overall_fit_block_EN(overall_label, avg_gap, median_gap, fit_pct):
    """
    Nicely formatted block to insert into MD.
    """
    return (
        f"**Overall fit:** {overall_label}  \n"
        f"- Mean |Δθ| across dimensions: **{avg_gap:.2f}**  \n"
        f"- Median |Δθ| across dimensions: **{median_gap:.2f}**  \n"
        f"- Global compatibility (interpretive): **{fit_pct:.0f}%**  \n"
    )


In [400]:
def global_fit_label_en(mean_abs):
    # Mirrors your Spanish logic, translated
    if mean_abs < 0.25:
        return "Very high overall fit"
    elif mean_abs < 0.40:
        return "High overall fit"
    elif mean_abs < 0.60:
        return "Moderate overall fit"
    elif mean_abs < 0.85:
        return "Moderate-to-low overall fit"
    else:
        return "Low overall fit"


def dimension_fit_label_en(gap_abs):
    # Same buckets as global (so it feels consistent)
    if gap_abs < 0.25:
        return "Very high compatibility"
    elif gap_abs < 0.40:
        return "High compatibility"
    elif gap_abs < 0.60:
        return "Moderate compatibility"
    elif gap_abs < 0.85:
        return "Moderate-to-low compatibility"
    else:
        return "Low compatibility"


def fit_pct_from_gap(gap_abs, cap=2.0):
    # interpretive index (0..100); capped for stability
    g = min(float(cap), float(gap_abs))
    return max(0.0, 100.0 * (1.0 - (g / cap)))


In [401]:
import numpy as np

def overall_fit_from_gaps_en(person_thetas, org_thetas, order):
    gaps = [abs(float(person_thetas[k]) - float(org_thetas[k])) for k in order]
    mean_abs = float(np.mean(gaps))
    median_abs = float(np.median(gaps))

    overall_label = global_fit_label_en(mean_abs)
    overall_pct = fit_pct_from_gap(mean_abs, cap=2.0)

    return overall_label, mean_abs, median_abs, overall_pct


def overall_fit_block_EN(overall_label, mean_abs, median_abs, fit_pct):
    # Clean spacing (double newlines between blocks handled by caller)
    return (
        f"**Overall fit:** {overall_label}\n"
        f"- Mean |Δθ| across dimensions: **{mean_abs:.2f}**\n"
        f"- Median |Δθ| across dimensions: **{median_abs:.2f}**\n"
        f"- Global compatibility (interpretive): **{fit_pct:.0f}%**\n"
    )


In [402]:
def build_comparative_matrix_md(person_thetas, org_thetas, DIMENSIONS, order):
    lines = []
    lines.append("## Comparative Matrix (by Dimension)\n\n")
    lines.append("| Dimension | Person θ | Org θ | Δθ (P−O) | |Δθ| | Fit % | Compatibility |\n")
    lines.append("|---|---:|---:|---:|---:|---:|---|\n")

    for dk in order:
        dim = DIMENSIONS[dk]
        p = float(person_thetas[dk])
        o = float(org_thetas[dk])
        d = p - o
        gap = abs(d)

        fit_pct = fit_pct_from_gap(gap, cap=2.0)
        compat = dimension_fit_label_en(gap)

        p_pole = infer_pole(dim, p)
        o_pole = infer_pole(dim, o)
        p_pole_lbl = dim["poles"][p_pole]["short"]
        o_pole_lbl = dim["poles"][o_pole]["short"]

        lines.append(
            f"| {dim['name']} | {p:+.2f} | {o:+.2f} | {d:+.2f} | {gap:.2f} | {fit_pct:.0f}% | {compat} |\n"
        )

    lines.append("\n")
    lines.append("*Fit % is an interpretive index derived from |Δθ| (capped at 2.0) to provide a quick approximation of alignment.*\n\n")
    return "".join(lines)


In [403]:
HOW_TO_READ_EN = (
    "> **How to read these results**  \n"
    "> - **Δθ (P−O)** is a *signed* difference (Person minus Organization). The sign indicates direction.  \n"
    "> - **|Δθ|** is the *distance* between Person and Organization. **Closer to 0 = higher compatibility**; larger values mean greater mismatch.  \n"
    "> - **Fit %** is an *interpretive index* derived from |Δθ| (capped) to provide a quick visual proxy: **higher % = closer alignment**.  \n"
    "> - Scores are standardized latent values (θ, z-scored). Use these outputs as decision support, not as a deterministic verdict.\n"
)


In [486]:
def dimension_block_en(
    dim_key: str,
    person_theta: float,
    org_theta: float,
    dim_cfg: dict,
    cap: float = 1.5,
    evidence_light=None
) -> str:
    p = float(person_theta)
    o = float(org_theta)
    d = p - o
    gap = abs(d)

    fit_pct = fit_pct_from_gap(gap, cap=cap)
    compat = dimension_fit_label_en(gap)
    level_key = gap_to_level_key(gap)

    p_pole = infer_pole(dim_cfg, p)
    o_pole = infer_pole(dim_cfg, o)
    p_lbl = dim_cfg["poles"][p_pole]["label"]
    o_lbl = dim_cfg["poles"][o_pole]["label"]

    poleA_short = dim_cfg["poles"]["A"]["short"]
    poleB_short = dim_cfg["poles"]["B"]["short"]

    md = []
    md.append(f"### {dim_cfg['name']}\n\n")
    md.append(f"**Definition.** {dim_cfg['definition']}\n\n")

    md.append("**Compatibility diagnosis.**\n")
    md.append(f"- Person θ: **{p:+.2f}** → **{p_lbl}**\n")
    md.append(f"- Organization θ: **{o:+.2f}** → **{o_lbl}**\n")
    md.append(f"- Δθ (P−O): **{d:+.2f}** | |Δθ|: **{gap:.2f}** | Fit %: **{fit_pct:.0f}%** → **{compat}**\n\n")

    md.append("**Cognitive attributes (both poles).**\n")
    md.append(f"- **{poleA_short}:**\n")
    for it in dim_cfg.get("cognitive", {}).get("A", []):
        md.append(f"  - *{it['title']}*: {it['desc']}\n")
    md.append(f"- **{poleB_short}:**\n")
    for it in dim_cfg.get("cognitive", {}).get("B", []):
        md.append(f"  - *{it['title']}*: {it['desc']}\n")
    md.append("\n")

    md.append("**Observable cultural behaviors (both poles).**\n")
    md.append(f"- **{poleA_short}**\n")
    for b in dim_cfg.get("behaviors_vs", {}).get("A", []):
        md.append(f"- {b.strip()}\n")
    md.append(f"- **{poleB_short}**\n")
    for b in dim_cfg.get("behaviors_vs", {}).get("B", []):
        md.append(f"  - {b}\n")
    md.append("\n")

    md.append("**Associated socioemotional skills (soft skills).**\n")
    md.append(f"- {poleA_short}: {dim_cfg.get('softskills', {}).get('A','')}\n")
    md.append(f"- {poleB_short}: {dim_cfg.get('softskills', {}).get('B','')}\n\n")

    practical = dim_cfg.get("practical_implications", {})
    practical_bullets = practical.get(level_key, [])
    if practical_bullets:
        md.append(f"**Practical implications ({level_key}).**\n\n")
        for b in practical_bullets:
            md.append(f"- {b}\n")
        md.append("\n")

    recos = dim_cfg.get("actionable_recommendations", {})
    if recos:
        md.append(render_reco_categories(recos, level_key))
        md.append("\n")

    if isinstance(evidence_light, dict):
        ev = evidence_light.get(dim_key)
        if ev:
            md.append(render_evidence_light_md(dim_key, ev))
            md.append("\n")

    return "".join(md)


In [460]:
dk = "D1_GO"
print("Has practical:", "practical_implications" in DIMENSIONS[dk])
print("Has recos:", "actionable_recommendations" in DIMENSIONS[dk])
print("Practical keys:", list(DIMENSIONS[dk].get("practical_implications", {}).keys()))
print("Reco cats:", list(DIMENSIONS[dk].get("actionable_recommendations", {}).keys()))


Has practical: True
Has recos: True
Practical keys: ['very_high', 'high', 'moderate', 'moderate_low', 'low']
Reco cats: ['onboarding', 'role_design', 'coaching_training', 'risk_monitoring']


In [461]:
dk = "D1_GO"
gap = abs(float(person_thetas_hybrid[dk]) - float(org_thetas_example[dk]))
print("theta person:", person_thetas_hybrid[dk])
print("theta org:", org_thetas_example[dk])
print("gap:", gap)
print("level_key:", gap_to_level_key(gap))


theta person: 1.3175
theta org: 0.65
gap: 0.6674999999999999
level_key: moderate_low


In [462]:
dk = "D1_GO"
gap = abs(float(person_thetas_hybrid[dk]) - float(org_thetas_example[dk]))
level_key = gap_to_level_key(gap)

recos = DIMENSIONS[dk].get("actionable_recommendations", {})
out = render_reco_categories(recos, level_key)

print("Reco output length:", len(out))
print(out[:600])


Reco output length: 193
**Recommendations (moderate_low).**

**Onboarding recommendations**

- ...

**Role design recommendations**

- ...

**Coaching / training recommendations**

- ...

**Risks to monitor**

- ...




In [463]:
print("Evidence keys:", list(evidence_by_dim.keys()))
print("Evidence D1_GO:", evidence_by_dim.get("D1_GO"))


Evidence keys: ['D1_GO', 'D2_IDED', 'D3_ES', 'D4_OS', 'D5_EJ', 'D6_PP']
Evidence D1_GO: {'pole': 'Goal-Oriented', 'confidence': 0.85, 'evidence': ['La prioridad fue el resultado: lanzamos una versión funcional', 'para no bloquear el despliegue'], 'lex_hits_total': 0, 'notes': 'Lexicon low hits: LLM weighted more'}


In [464]:
snippet = dimension_block_en(
    "D1_GO",
    person_thetas_hybrid["D1_GO"],
    org_thetas_example["D1_GO"],
    DIMENSIONS["D1_GO"],
    cap=1.5,
    evidence_light=evidence_by_dim
)

print(snippet)


### Goal Orientation

**Definition.** This dimension describes the extent to which an individual or organization prioritizes the achievement of final outcomes versus adherence to processes, methods, and means. From a cognitive perspective, it reflects the dominant attentional focus: whether it is directed toward the 'how' (means) or toward the 'what' (goals). It is not a value judgment: both poles can be functional depending on context, type of work, and the level of uncertainty.

**Compatibility diagnosis.**
- Person θ: **+1.32** → **Goal-Oriented**
- Organization θ: **+0.65** → **Goal-Oriented**
- Δθ (P−O): **+0.67** | |Δθ|: **0.67** | Fit %: **56%** → **Moderate-to-low compatibility**

**Cognitive attributes (both poles).**
- **Means-Oriented:**
  - *Attention to process and sequence*: Performance is mentally organized around steps, methods, and controls; procedural coherence serves as a key quality criterion.
  - *Evaluation via compliance and traceability*: Consistency of methods,

In [405]:
def gap_to_level_key(gap_abs: float) -> str:
    g = float(gap_abs)
    if g < 0.25:
        return "very_high"
    elif g < 0.40:
        return "high"
    elif g < 0.60:
        return "moderate"
    elif g < 0.85:
        return "moderate_low"
    else:
        return "low"


In [456]:
# ============================================================
# FULL REPORT BUILDER (EN) — Cognitive–Cultural Fit Report
# Includes: fine labels, Fit%, comparative matrix, dual radar,
# spacing fixes, HR section dividers, Pandoc DOCX export.
# ============================================================

import os
import numpy as np
from datetime import datetime

# -----------------------------
# Styling blocks (MD)
# -----------------------------
HR = "\n\n---\n\n"

HOW_TO_READ_EN = (
    "> **How to read these results**  \n"
    "> - **Δθ (P−O)** is a *signed* difference (Person minus Organization). The sign indicates direction (which side is higher).  \n"
    "> - **|Δθ|** is the *distance* between Person and Organization. **Closer to 0 = higher compatibility**; larger values mean greater mismatch.  \n"
    "> - **Fit %** is an *interpretive index* derived from |Δθ| (capped) for quick scanning: **higher % = closer alignment**.  \n"
    "> - Scores are standardized latent values (θ, z-scored). Use outputs as decision support, not as deterministic verdicts.\n"
)

def method_note_en(text: str) -> str:
    return f"> **Methodological  note:** {text}\n"

METHOD_NOTE_CAP_EN = (
    "Fit % is an interpretive index derived from |Δθ| and linearly mapped to 0–100 using a cap "
    "(default **1.5**). Differences above the cap are truncated for stability. This improves visual interpretability; it is not a probability."
)

TEXT_TEMPLATES = {
    "EN": {
        "report_title": "Cognitive–Cultural Fit Report",
        "executive_summary": (
            "This report summarizes alignment patterns between an individual and an organization across six cognitive–cultural dimensions. "
            "Results are based on standardized latent scores (θ, z-scored). Fit indices are interpretive aids and should be complemented with contextual judgment."
        ),
        "closing_note": (
            "End of report. This document is intended as a decision-support artifact for onboarding, role design, and cultural risk monitoring."
        )
    }
}

# -----------------------------
# Fit label taxonomy (fine-grained)
# -----------------------------
def global_fit_label_en(mean_abs: float) -> str:
    if mean_abs < 0.25:
        return "Very high overall fit"
    elif mean_abs < 0.40:
        return "High overall fit"
    elif mean_abs < 0.60:
        return "Moderate overall fit"
    elif mean_abs < 0.85:
        return "Moderate-to-low overall fit"
    else:
        return "Low overall fit"

def dimension_fit_label_en(gap_abs: float) -> str:
    if gap_abs < 0.25:
        return "Very high compatibility"
    elif gap_abs < 0.40:
        return "High compatibility"
    elif gap_abs < 0.60:
        return "Moderate compatibility"
    elif gap_abs < 0.85:
        return "Moderate-to-low compatibility"
    else:
        return "Low compatibility"

def fit_pct_from_gap(gap_abs: float, cap: float = 1.5) -> float:
    g = min(float(cap), float(gap_abs))
    return max(0.0, 100.0 * (1.0 - (g / cap)))

# -----------------------------
# Pole inference (uses DIMENSIONS structure)
# -----------------------------
def infer_pole(dim_cfg: dict, theta: float) -> str:
    """
    Returns 'A' or 'B' depending on theta sign and theta_positive_maps_to.
    Convention:
      - If theta >= 0, maps to dim_cfg['theta_positive_maps_to'] (usually 'B')
      - If theta < 0, maps to the opposite pole
    """
    pos_pole = dim_cfg.get("theta_positive_maps_to", "B")
    neg_pole = "A" if pos_pole == "B" else "B"
    return pos_pole if float(theta) >= 0 else neg_pole

# -----------------------------
# Overall fit
# -----------------------------
def overall_fit_from_gaps_en(person_thetas: dict, org_thetas: dict, order: list, cap: float = 1.5):
    gaps = [abs(float(person_thetas[k]) - float(org_thetas[k])) for k in order]
    mean_abs = float(np.mean(gaps))
    median_abs = float(np.median(gaps))
    overall_label = global_fit_label_en(mean_abs)
    overall_pct = fit_pct_from_gap(mean_abs, cap=cap)
    return overall_label, mean_abs, median_abs, overall_pct

def overall_fit_block_EN(overall_label: str, mean_abs: float, median_abs: float, fit_pct: float) -> str:
    return (
        f"**Overall fit:** {overall_label}\n"
        f"- Mean |Δθ| across dimensions: **{mean_abs:.2f}**\n"
        f"- Median |Δθ| across dimensions: **{median_abs:.2f}**\n"
        f"- Global compatibility (interpretive): **{fit_pct:.0f}%**\n"
    )

# -----------------------------
# Comparative matrix (compact for DOCX)
# -----------------------------
def build_comparative_matrix_md_compact(person_thetas: dict, org_thetas: dict, DIMENSIONS: dict, order: list, cap: float = 1.5) -> str:
    lines = []
    lines.append("## Comparative Matrix (Quick View)\n\n")
    lines.append("| Dimension | Person θ | Org θ | Δθ (P−O) | |Δθ| | Fit % | Compatibility |\n")
    lines.append("|---|---:|---:|---:|---:|---:|---|\n")

    for dk in order:
        dim = DIMENSIONS[dk]
        p = float(person_thetas[dk])
        o = float(org_thetas[dk])
        d = p - o
        gap = abs(d)

        fit_pct = fit_pct_from_gap(gap, cap=cap)
        compat = dimension_fit_label_en(gap)

        lines.append(f"| {dim['name']} | {p:+.2f} | {o:+.2f} | {d:+.2f} | {gap:.2f} | {fit_pct:.0f}% | {compat} |\n")

    lines.append("\n")
    lines.append(method_note_en(METHOD_NOTE_CAP_EN) + "\n")
    return "".join(lines)

# -----------------------------
# Dimension section block (EN)
# - uses DIMENSIONS content for definition, cognitive, behaviors_vs, softskills
# -----------------------------
def dimension_block_en(dim_key: str, person_theta: float, org_theta: float, dim_cfg: dict, cap: float = 1.5, evidence_light=None) -> str:

    p = float(person_theta)
    o = float(org_theta)
    d = p - o
    gap = abs(d)

    fit_pct = fit_pct_from_gap(gap, cap=cap)
    compat = dimension_fit_label_en(gap)

    p_pole = infer_pole(dim_cfg, p)
    o_pole = infer_pole(dim_cfg, o)
    p_lbl = dim_cfg["poles"][p_pole]["label"]
    o_lbl = dim_cfg["poles"][o_pole]["label"]

    md = []
    md.append(f"### {dim_cfg['name']}\n\n")

    md.append(f"**Definition.** {dim_cfg['definition']}\n\n")

    md.append("**Compatibility diagnosis.**\n")
    md.append(f"- Person θ: **{p:+.2f}** → **{p_lbl}**\n")
    md.append(f"- Organization θ: **{o:+.2f}** → **{o_lbl}**\n")
    md.append(f"- Δθ (P−O): **{d:+.2f}** | |Δθ|: **{gap:.2f}** | Fit %: **{fit_pct:.0f}%** → **{compat}**\n\n")

    md.append("**Cognitive attributes (both poles).**\n")
    md.append(f"- **{dim_cfg['poles']['A']['short']}:**\n")
    for it in dim_cfg["cognitive"]["A"]:
        md.append(f"  - *{it['title']}*: {it['desc']}\n")
    md.append(f"- **{dim_cfg['poles']['B']['short']}:**\n")
    for it in dim_cfg["cognitive"]["B"]:
        md.append(f"  - *{it['title']}*: {it['desc']}\n")
    md.append("\n")

    md.append("**Observable cultural behaviors (both poles).**\n")
    md.append(f"- **{dim_cfg['poles']['A']['short']}**\n")
    for b in dim_cfg["behaviors_vs"]["A"]:
        md.append(f"  - {b}\n")
    md.append(f"- **{dim_cfg['poles']['B']['short']}**\n")
    for b in dim_cfg["behaviors_vs"]["B"]:
        md.append(f"  - {b}\n")
    md.append("\n")

    md.append("**Associated socioemotional skills (soft skills).**\n")
    md.append(f"- {dim_cfg['poles']['A']['short']}: {dim_cfg['softskills']['A']}\n")
    md.append(f"- {dim_cfg['poles']['B']['short']}: {dim_cfg['softskills']['B']}\n\n")

    return "".join(md)

# -----------------------------
# Spacing normalizer (Pandoc-friendly)
# -----------------------------
def normalize_md_spacing(md_text: str) -> str:
    # Ensure blank line before headings
    md_text = md_text.replace("\n## ", "\n\n## ")
    md_text = md_text.replace("\n### ", "\n\n### ")
    # Avoid extreme collapse
    while "\n\n\n\n" in md_text:
        md_text = md_text.replace("\n\n\n\n", "\n\n\n")
    return md_text

# -----------------------------
# Dual radar chart (matplotlib) with fine ticks
# -----------------------------
def radar_chart_dual(person_vals, org_vals, labels, out_png,
                     title="Cognitive–Cultural Fit Profile (θ)",
                     person_label="Person", org_label="Organization",
                     rmin=-1.75, rmax=1.75, ticks=None):
    import matplotlib.pyplot as plt

    if ticks is None:
        ticks = [-1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5]

    N = len(labels)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    pv = list(person_vals) + [person_vals[0]]
    ov = list(org_vals) + [org_vals[0]]

    plt.figure(figsize=(7.5, 7.5))
    ax = plt.subplot(111, polar=True)

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=9)

    ax.set_ylim(rmin, rmax)
    ax.set_yticks(ticks)
    ax.set_yticklabels([f"{t:g}" for t in ticks], fontsize=9)

    ax.plot(angles, pv, linewidth=2, label=person_label)
    ax.fill(angles, pv, alpha=0.08)

    ax.plot(angles, ov, linewidth=2, label=org_label)
    ax.fill(angles, ov, alpha=0.08)

    ax.set_title(title, pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))

    plt.tight_layout()
    plt.savefig(out_png, dpi=220)
    plt.close()

# -----------------------------
# Pandoc export (MD -> DOCX)
# -----------------------------
def export_md_to_docx(md_file: str, out_docx: str):
    import subprocess
    md_dir = os.path.dirname(os.path.abspath(md_file)) or "."
    out_docx = os.path.abspath(out_docx)

    try:
        subprocess.run(
            ["pandoc", os.path.basename(md_file), "-o", out_docx],
            cwd=md_dir,
            check=True
        )
        return out_docx
    except FileNotFoundError:
        raise RuntimeError("Pandoc not found in PATH. Install Pandoc and restart Jupyter/Anaconda.")
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"Pandoc failed: {e}")

# -----------------------------
# Full report builder (EN) — NO cover; clean meta header
# -----------------------------
def generate_fit_report_en(
    person_thetas: dict,
    org_thetas: dict,
    DIMENSIONS: dict,
    order: list,
    out_dir: str = "outputs",
    out_basename: str = "fit_report_en",
    cap: float = 1.5,
    meta: dict = None,
    evidence_by_dim: dict = None,   # ✅ NUEVO
):
    os.makedirs(out_dir, exist_ok=True)

    meta = meta or {}
    meta = {
        "report_id": meta.get("report_id", f"CCF-{datetime.now().strftime('%Y%m%d-%H%M%S')}"),
        "person_id": meta.get("person_id", "P-DEMO-01"),
        "organization_id": meta.get("organization_id", "O-DEMO-01"),
        "model_version": meta.get("model_version", "v1.2"),
        "instrument_version": meta.get("instrument_version", "P-v1.0"),
        "date": meta.get("date", datetime.now().strftime("%Y-%m-%d")),
    }

    # radar
    labels = [DIMENSIONS[k]["name"] for k in order]
    p_vals = [float(person_thetas[k]) for k in order]
    o_vals = [float(org_thetas[k]) for k in order]
    radar_png = os.path.join(out_dir, f"{out_basename}_radar_dual.png")

    radar_chart_dual(
        person_vals=p_vals,
        org_vals=o_vals,
        labels=labels,
        out_png=radar_png,
        title="Cognitive–Cultural Fit Profile (θ, z-scored)",
        person_label="Person",
        org_label="Organization",
        rmin=-1.75, rmax=1.75,
        ticks=[-1.5,-1.0,-0.5,0.0,0.5,1.0,1.5]
    )

    # overall
    overall_label, mean_abs, median_abs, overall_pct = overall_fit_from_gaps_en(person_thetas, org_thetas, order, cap=cap)

    # build md
    md = []
    md.append(f"# {TEXT_TEMPLATES['EN']['report_title']}\n\n")

    md.append("## Report Header\n\n")
    md.append(f"- **Report ID:** {meta['report_id']}\n")
    md.append(f"- **Person ID:** {meta['person_id']}\n")
    md.append(f"- **Organization ID:** {meta['organization_id']}\n")
    md.append(f"- **Model Version:** {meta['model_version']}\n")
    md.append(f"- **Instrument:** {meta['instrument_version']}\n")
    md.append(f"- **Date:** {meta['date']}\n\n")

    md.append(HR)

    md.append("## Executive Summary\n\n")
    md.append(TEXT_TEMPLATES["EN"]["executive_summary"] + "\n\n")

    md.append("## Overall Fit Result\n\n")
    md.append(overall_fit_block_EN(overall_label, mean_abs, median_abs, overall_pct) + "\n\n")
    md.append(HOW_TO_READ_EN + "\n\n")
    md.append(method_note_en(METHOD_NOTE_CAP_EN) + "\n\n")

    md.append(HR)

    # compact matrix
    md.append(build_comparative_matrix_md_compact(person_thetas, org_thetas, DIMENSIONS, order, cap=cap))
    md.append(HR)

    # radar
    md.append("## Radar Chart (Person vs Organization)\n\n")
    md.append(f"![]({os.path.basename(radar_png)})\n\n")
    md.append(HR)

    # dimension sections
    md.append("## Dimension-Level Results\n\n")
    for dk in order:
        md.append(dimension_block_en(dk, person_thetas[dk], org_thetas[dk], DIMENSIONS[dk], cap=cap, evidence_light=evidence_by_dim))
        md.append(HR)

    md.append(TEXT_TEMPLATES["EN"]["closing_note"] + "\n")

    final_md = normalize_md_spacing("".join(md))
    md_path = os.path.join(out_dir, f"{out_basename}.md")
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(final_md)


    
    return {"md": md_path, "radar_png": radar_png, "meta": meta}

# ============================================================
# Example usage
# ============================================================
# person_thetas_example = {...}
# org_thetas_example = {...}
# ORDER = ["D1_GO","D2_IDED","D3_ES","D4_OS","D5_EJ","D6_PP"]
#
# artifacts = generate_fit_report_en(
#     person_thetas=person_thetas_example,
#     org_thetas=org_thetas_example,
#     DIMENSIONS=DIMENSIONS,   # IMPORTANT: use DIMENSIONS_EN if you have it
#     order=ORDER,
#     out_dir="outputs",
#     out_basename="fit_report_en_final",
#     cap=1.5,
#     meta={"person_id":"P-DEMO-01","organization_id":"O-DEMO-01","model_version":"v1.2"}
# )
#
# docx_path = export_md_to_docx(artifacts["md"], r"outputs\fit_report_en_final.docx")
# print("MD:", artifacts["md"])
# print("Radar:", artifacts["radar_png"])
# print("DOCX:", docx_path)


In [480]:
DIMENSIONS_EN["D1_GO"]["practical_implications"] = {
    "very_high": [
        "Strong day-to-day alignment on how work should be executed (outcomes vs process), enabling fast coordination and low friction.",
        "High likelihood of shared expectations around prioritization, success criteria, and trade-offs (speed vs rigor).",
        "Minimal overhead needed to negotiate workflow norms; teams can focus on delivery rather than alignment discussions."
    ],
    "high": [
        "Overall smooth collaboration, with occasional need to clarify whether the priority is process fidelity or outcome speed.",
        "Minor friction may appear under pressure (deadlines, incidents) where implicit assumptions about 'the right way' diverge.",
        "Good fit for most roles, with light calibration on execution norms during onboarding."
    ],
    "moderate": [
        "Noticeable differences in execution style may require explicit agreement on how to balance outcomes, quality gates, and process discipline.",
        "Risk of misinterpreting each other’s decisions: one side may see flexibility as 'cutting corners' while the other sees process as 'bureaucracy'.",
        "Coordination improves significantly when expectations (definition of done, validation steps, escalation rules) are made explicit."
    ],
    "moderate_low": [
        "Frequent friction is likely in planning, estimation, and delivery pacing unless norms are clearly defined and reinforced.",
        "Higher risk of conflict around accountability: goals-first stakeholders may push speed, while process-first stakeholders push compliance and traceability.",
        "Without alignment, work may oscillate between rushed delivery and over-governance, reducing trust and perceived performance."
    ],
    "low": [
        "High probability of recurring mismatch in how success is defined (impact vs procedure) and how work is validated (speed vs rigor).",
        "Strong risk of productivity loss due to negotiation overhead and repeated rework driven by incompatible expectations.",
        "This gap should be treated as a primary onboarding and role-design risk; without intervention, cultural friction is likely to persist."
    ],
}

DIMENSIONS_EN["D1_GO"]["actionable_recommendations"] = {
    "onboarding": {
        "very_high": [
            "Confirm shared execution norms early (definition of done, success metrics) and empower rapid execution.",
            "Use lightweight check-ins to keep alignment stable during priority shifts."
        ],
        "high": [
            "Run a short onboarding calibration: clarify what 'good' looks like (impact metrics vs process quality gates).",
            "Agree on escalation and exception rules (when it is acceptable to bypass a step)."
        ],
        "moderate": [
            "Add a 'working agreement' session: define required quality gates, documentation expectations, and decision criteria under time pressure.",
            "Create a shared rubric for trade-offs (e.g., speed vs compliance vs reliability) to reduce interpretation conflicts."
        ],
        "moderate_low": [
            "Implement structured onboarding with explicit workflow standards (checklists, sign-offs, review policies) or explicit outcome targets—depending on the org’s dominant pole.",
            "Schedule early 30/60/90-day alignment checkpoints focused on execution norms and friction patterns."
        ],
        "low": [
            "Treat this as a critical onboarding risk: provide explicit playbooks, examples of acceptable execution styles, and tight feedback loops.",
            "Assign a cultural mentor (or process owner) to prevent repeated misalignment and rework."
        ],
    },

    "role_design": {
        "very_high": [
            "Assign roles with autonomy—execution style will naturally match expectations.",
            "Use the person in environments requiring consistent delivery with low coordination overhead."
        ],
        "high": [
            "If the org is strongly goal-oriented, place the person in outcome-driven work streams with minimal governance.",
            "If the org is strongly means-oriented, place the person in reliability/quality-sensitive streams where process fidelity matters."
        ],
        "moderate": [
            "Choose roles with clear boundaries: either clearly outcome-owned deliverables or clearly process-defined responsibilities.",
            "Reduce ambiguity by clarifying who decides exceptions, when to document, and what minimum process is mandatory."
        ],
        "moderate_low": [
            "Avoid high-ambiguity roles unless the org provides strong scaffolding (process) or strong outcome guardrails (metrics).",
            "Consider pairing with a complementary counterpart (goal-driven + process-driven) to stabilize delivery."
        ],
        "low": [
            "Avoid roles requiring constant negotiation of process vs outcomes; mismatch will be costly.",
            "If unavoidable, formalize execution contracts (explicit acceptance criteria, process gates, and success metrics)."
        ],
    },

    "coaching_training": {
        "very_high": [
            "Focus coaching on continuous improvement and scaling impact rather than basic alignment."
        ],
        "high": [
            "Coach on situational flexibility: when to prioritize outcomes, and when to enforce process rigor (risk-based thinking)."
        ],
        "moderate": [
            "Train on decision frameworks for trade-offs (risk-based QA, lightweight governance, agile definitions of done).",
            "Practice reframing: process-first is not 'bureaucracy'; goal-first is not 'recklessness'—both are strategies."
        ],
        "moderate_low": [
            "Provide structured coaching on either process discipline (documentation, compliance) or outcome thinking (impact, prioritization), depending on the misfit direction.",
            "Use real examples of conflicts to build shared mental models."
        ],
        "low": [
            "Deliver targeted training + frequent feedback cycles; misfit will not self-correct.",
            "If the gap remains high, consider role reassignment or stronger structural controls."
        ],
    },

    "risk_monitoring": {
        "very_high": [
            "Monitor only during major context shifts (re-org, high-pressure launches)."
        ],
        "high": [
            "Monitor under stress scenarios (deadlines, incidents) where norms may diverge."
        ],
        "moderate": [
            "Watch for recurring disputes about 'done', documentation, or review rigor; these signal norm ambiguity."
        ],
        "moderate_low": [
            "Watch for rework cycles, delayed approvals, and blame patterns around process vs outcomes."
        ],
        "low": [
            "Watch for chronic delays, repeated escalations, and persistent dissatisfaction with execution style—intervene early."
        ],
    },
}


In [408]:
DIMENSIONS_EN["D2_IDED"]["practical_implications"] = {
    "very_high": [
        "Strong motivational alignment: both sides share similar expectations about autonomy, feedback, and what sustains effort.",
        "Lower risk of disengagement or misinterpretation of performance signals; motivation remains stable across reward changes.",
        "High probability of sustainable performance without heavy external pressure or constant reinforcement."
    ],
    "high": [
        "Overall motivational fit, with occasional calibration needed (e.g., frequency of recognition, structure, or autonomy).",
        "Minor friction may appear in high-pressure cycles (deadlines, KPIs) if the dominant motivational style differs slightly.",
        "Simple agreements on feedback style and incentives typically resolve most issues."
    ],
    "moderate": [
        "Noticeable differences in motivational drivers may affect persistence, ownership, and how feedback is interpreted.",
        "Risk: the organization may over-rely on rewards/pressure while the individual seeks autonomy/purpose—or the reverse.",
        "Alignment improves if motivation supports are made explicit (recognition cadence, autonomy boundaries, growth pathways)."
    ],
    "moderate_low": [
        "High likelihood of recurring friction: one side needs autonomy and intrinsic meaning while the other expects compliance and external regulation.",
        "Risk of burnout, disengagement, or performance volatility depending on reward cycles and supervisory style.",
        "Without deliberate support, motivation may become fragile and dependent on short-term contingencies."
    ],
    "low": [
        "Strong mismatch in motivational regulation: sustained effort may be hard to maintain under the organization’s dominant incentives/controls.",
        "High risk of chronic disengagement, misread feedback (judgment vs information), and attrition.",
        "This should be treated as a primary fit risk requiring structured intervention or role re-design."
    ],
}

DIMENSIONS_EN["D2_IDED"]["actionable_recommendations"] = {
    "onboarding": {
        "very_high": [
            "Validate motivational expectations early (autonomy, mastery, recognition) and keep supports lightweight.",
            "Use short check-ins to maintain meaning and clarity as priorities shift."
        ],
        "high": [
            "Clarify feedback cadence and recognition norms (what counts as ‘good’ and how it is acknowledged).",
            "Agree on autonomy boundaries: what decisions the person owns vs what must be escalated."
        ],
        "moderate": [
            "Explicitly design motivational supports: autonomy + purpose framing OR structured KPIs + recognition (depending on mismatch direction).",
            "Align on how feedback will be delivered (coaching-oriented vs evaluative) and what success signals mean."
        ],
        "moderate_low": [
            "Add structured onboarding with early milestones and explicit reinforcement strategy (intrinsic supports + clear goals).",
            "Schedule 30/60/90-day motivational check-ins: monitor energy, engagement, and how incentives are landing."
        ],
        "low": [
            "Treat as critical onboarding risk: define a motivational contract (autonomy, rewards, supervision style, growth path).",
            "Assign a mentor/manager style match; increase feedback frequency and monitor early disengagement signals."
        ],
    },
    "role_design": {
        "very_high": [
            "Place the person in roles matching the org’s motivational ecology (autonomy-heavy or structure-heavy) without extra overhead."
        ],
        "high": [
            "If org is externally driven, choose roles with clear metrics and visible rewards; if internally driven, choose roles with ownership and learning.",
            "Ensure incentives align with the actual work (avoid rewarding the wrong behaviors)."
        ],
        "moderate": [
            "Reduce ambiguity: define what is rewarded, what is owned, and what is learning-focused.",
            "Use roles that allow both drivers: clear outcomes + autonomy in method."
        ],
        "moderate_low": [
            "Avoid roles requiring long-term intrinsic persistence if the context is heavily reward/pressure based (or vice versa).",
            "Pair with a manager who can translate incentives into meaning (or structure autonomy into goals)."
        ],
        "low": [
            "Avoid roles where sustained engagement is essential without strong motivational scaffolding.",
            "If unavoidable, redesign incentives and supervision style to prevent repeated disengagement cycles."
        ],
    },
    "coaching_training": {
        "very_high": [
            "Coach toward long-term growth goals and continuous improvement rather than motivation stabilization."
        ],
        "high": [
            "Train on self-regulation: how to stay engaged when rewards/pressure fluctuate; calibrate feedback interpretation."
        ],
        "moderate": [
            "Coach on reframing feedback (information vs evaluation) and building stable drivers (purpose, mastery, autonomy).",
            "Train managers on motivational fit: recognition style, autonomy support, and goal clarity."
        ],
        "moderate_low": [
            "Provide targeted coaching: autonomy-supportive practices OR structured reinforcement systems depending on gap direction.",
            "Use concrete examples of demotivation triggers and redesign them."
        ],
        "low": [
            "High-touch coaching + manager training is required; misfit rarely self-corrects.",
            "If repeated disengagement persists, consider role change or different incentive structure."
        ],
    },
    "risk_monitoring": {
        "very_high": [
            "Monitor only during major incentive or leadership changes."
        ],
        "high": [
            "Monitor during high-pressure cycles (KPIs, deadlines) when external regulation typically increases."
        ],
        "moderate": [
            "Watch for motivational volatility: spikes around rewards, drops after evaluation cycles, or resistance to supervision style."
        ],
        "moderate_low": [
            "Watch for disengagement, burnout language, low ownership, or constant validation seeking."
        ],
        "low": [
            "Watch for chronic dissatisfaction, avoidance, performance instability, and early attrition signals—intervene fast."
        ],
    },
}


In [409]:
DIMENSIONS_EN["D3_ES"]["practical_implications"] = {
    "very_high": [
        "Strong alignment in how rules are interpreted (as flexible guides vs strict standards), minimizing coordination friction.",
        "High predictability in how exceptions, ambiguity, and change are handled.",
        "Low overhead to agree on working norms; collaboration remains smooth under uncertainty."
    ],
    "high": [
        "Generally smooth collaboration, with occasional clarifications needed around exceptions and enforcement.",
        "Minor friction may occur during process changes or compliance-heavy moments, but is manageable with clear agreements.",
        "Overall fit supports stable execution and healthy adaptability."
    ],
    "moderate": [
        "Differences in tolerance for ambiguity may create repeated discussions about how strictly to follow policies and procedures.",
        "Risk: one side experiences flexibility as chaos; the other experiences structure as rigidity.",
        "Outcomes improve when exception-handling rules and minimum standards are explicitly defined."
    ],
    "moderate_low": [
        "Frequent friction likely in governance, approvals, and how ‘done’ is defined under time pressure.",
        "Risk of rework: strict expectations may invalidate flexible execution (or flexible shortcuts may undermine compliance).",
        "Without explicit norms, trust may erode due to perceived unreliability or over-control."
    ],
    "low": [
        "High mismatch in rule interpretation and uncertainty handling; coordination overhead and rework are likely persistent.",
        "Major risk of conflict over compliance, approvals, and autonomy boundaries.",
        "Treat as a primary fit risk: requires explicit governance design or role reassignment."
    ],
}

DIMENSIONS_EN["D3_ES"]["actionable_recommendations"] = {
    "onboarding": {
        "very_high": [
            "Confirm shared tolerance for ambiguity and exception rules; keep governance lightweight.",
            "Document minimal standards once; avoid over-process."
        ],
        "high": [
            "Clarify what is mandatory vs flexible: policies, review gates, and acceptable exceptions.",
            "Agree on how changes are introduced (who approves and what evidence is needed)."
        ],
        "moderate": [
            "Create a working agreement: minimum standards, escalation paths, and exception-handling protocol.",
            "Define ‘definition of done’ and quality gates to prevent rework and interpretation conflicts."
        ],
        "moderate_low": [
            "Use structured onboarding: explicit playbooks, checklists, and examples of acceptable flexibility.",
            "Schedule early checkpoints focused on governance friction and how exceptions are being handled."
        ],
        "low": [
            "Treat as critical onboarding risk: enforce clear rules or explicitly grant autonomy with guardrails—do not leave norms implicit.",
            "Assign a process owner/mentor to ensure stability and reduce repeated conflict."
        ],
    },
    "role_design": {
        "very_high": [
            "Assign to environments matching their preference (high ambiguity/innovation vs high compliance/reliability)."
        ],
        "high": [
            "Place in roles where either flexibility is clearly expected (exploration) or strictness is clearly required (risk control)."
        ],
        "moderate": [
            "Prefer roles with clear boundaries: stable core process + flexible edges.",
            "Reduce ambiguity by clarifying who can deviate from standards and under what conditions."
        ],
        "moderate_low": [
            "Avoid roles that constantly alternate between improvisation and heavy compliance unless norms are very explicit.",
            "Use pairing: flexible execution + structured governance partner."
        ],
        "low": [
            "Avoid roles requiring constant negotiation of governance; mismatch will be costly.",
            "If unavoidable, formalize approval workflows and exception documentation."
        ],
    },
    "coaching_training": {
        "very_high": [
            "Coach on scaling impact and continuous improvement, not on basic norm alignment."
        ],
        "high": [
            "Train on situational governance: when to be strict vs when to be adaptive (risk-based thinking)."
        ],
        "moderate": [
            "Coach on reframing: strictness protects reliability; flexibility enables learning—both are strategies.",
            "Train teams on lightweight governance (minimum viable process) to reduce friction."
        ],
        "moderate_low": [
            "Provide targeted coaching on either autonomy-with-guardrails or compliance discipline depending on gap direction.",
            "Use concrete incident reviews to align norms."
        ],
        "low": [
            "High-touch coaching + structural interventions required; misfit rarely self-corrects.",
            "If repeated conflicts persist, consider role redesign or different team placement."
        ],
    },
    "risk_monitoring": {
        "very_high": [
            "Monitor during major changes (re-org, new compliance requirements)."
        ],
        "high": [
            "Monitor during incidents or audits where strictness typically increases."
        ],
        "moderate": [
            "Watch for disputes about exceptions, approvals, and what counts as acceptable deviation."
        ],
        "moderate_low": [
            "Watch for rework loops, delayed approvals, and blame patterns around ‘rigid vs reckless’ behavior."
        ],
        "low": [
            "Watch for chronic conflict, blocked delivery, and repeated escalation; intervene early with clear governance rules."
        ],
    },
}


In [410]:
DIMENSIONS_EN["D4_OS"]["practical_implications"] = {
    "very_high": [
        "Strong alignment in attention allocation: both sides agree on how much external signals (customer/market) should shape decisions.",
        "Shared expectations about openness, learning loops, and how fast the organization should adapt.",
        "Low friction in prioritization: external feedback and internal stability are balanced similarly."
    ],
    "high": [
        "Generally compatible orientation toward external vs internal focus, with occasional calibration needed on pace of change.",
        "Minor friction may appear when external pressure increases (customer demands) or when internal stability is threatened.",
        "Good fit for most contexts with explicit decision criteria."
    ],
    "moderate": [
        "Differences in ‘signal relevance’ may cause repeated debates: adapt to external feedback vs protect internal coherence.",
        "Risk: one side perceives the other as ‘too reactive’ or ‘too insular’.",
        "Alignment improves by defining how external inputs enter the decision pipeline."
    ],
    "moderate_low": [
        "Frequent friction likely around prioritization, stakeholder management, and change adoption speed.",
        "Risk of strategic drift: externally oriented parties may push constant change while internally oriented parties resist.",
        "Without explicit rules, external feedback can be ignored or over-amplified, harming performance."
    ],
    "low": [
        "High mismatch in external sensitivity vs internal control; conflict over priorities and boundaries is likely persistent.",
        "Strong risk of misalignment in product/market decisions and stakeholder expectations.",
        "Treat as a primary fit risk; requires explicit governance for external inputs and adaptation cycles."
    ],
}

DIMENSIONS_EN["D4_OS"]["actionable_recommendations"] = {
    "onboarding": {
        "very_high": [
            "Confirm shared cadence for feedback loops (customer signals → decisions) and keep it lightweight."
        ],
        "high": [
            "Clarify what external signals matter (customers, market, stakeholders) and how they influence priorities.",
            "Agree on adaptation cadence (weekly vs quarterly) to avoid surprise change."
        ],
        "moderate": [
            "Define a decision protocol: when external feedback triggers change vs when internal stability wins.",
            "Create clear stakeholder communication norms to avoid ‘boundary confusion’."
        ],
        "moderate_low": [
            "Use structured onboarding: explain external-feedback intake process, who owns it, and what is filtered out.",
            "Schedule early checkpoints focusing on tension points: responsiveness vs stability."
        ],
        "low": [
            "Treat as critical onboarding risk: establish explicit governance for external requests and internal change control.",
            "Assign a mentor/stakeholder lead to translate external pressures into stable internal plans."
        ],
    },
    "role_design": {
        "very_high": [
            "Assign to roles matching the organization’s boundary style (customer-facing learning roles vs internal reliability roles)."
        ],
        "high": [
            "If open-system dominant: put the person in customer-feedback loops and iteration cycles; if closed-system dominant: internal optimization roles."
        ],
        "moderate": [
            "Prefer roles with explicit interfaces: clear channel for external inputs + clear internal constraints.",
            "Reduce ambiguity by clarifying who can commit to external stakeholders."
        ],
        "moderate_low": [
            "Avoid roles requiring constant boundary negotiation unless governance is very explicit.",
            "Pair open-system drivers with closed-system stabilizers to balance adaptation and coherence."
        ],
        "low": [
            "Avoid roles where external pressure is constant without strong filtering rules.",
            "If unavoidable, formalize intake, prioritization, and decision rights."
        ],
    },
    "coaching_training": {
        "very_high": [
            "Coach on scaling learning loops and sustaining healthy external sensing without overload."
        ],
        "high": [
            "Train on balancing responsiveness with stability: risk-based prioritization and stakeholder expectation management."
        ],
        "moderate": [
            "Coach on reframing: openness ≠ chaos, closed ≠ stagnation; both manage risk differently.",
            "Train teams on structured feedback loops (intake → triage → decision → communicate)."
        ],
        "moderate_low": [
            "Provide targeted coaching on either boundary management (saying no, filtering) or adaptive experimentation depending on gap direction.",
            "Use real cases where external signals were ignored or over-prioritized."
        ],
        "low": [
            "High-touch coaching + governance interventions required; misfit rarely self-corrects.",
            "If persistent, consider role reassignment to better boundary-fit context."
        ],
    },
    "risk_monitoring": {
        "very_high": [
            "Monitor during major market shifts or leadership changes."
        ],
        "high": [
            "Monitor during intense stakeholder pressure cycles."
        ],
        "moderate": [
            "Watch for recurring arguments about responsiveness, ‘scope creep’, and whether external signals are trusted."
        ],
        "moderate_low": [
            "Watch for prioritization instability, stakeholder dissatisfaction, and internal overload from external demands."
        ],
        "low": [
            "Watch for chronic conflict, repeated escalations, and delivery volatility; intervene early with governance."
        ],
    },
}


In [411]:
DIMENSIONS_EN["D5_EJ"]["practical_implications"] = {
    "very_high": [
        "Strong alignment in what ‘good performance’ means: human sustainability and relationship quality vs execution and throughput.",
        "Shared expectations about feedback tone, workload decisions, and team climate.",
        "Low risk of interpersonal friction; collaboration norms are mutually reinforcing."
    ],
    "high": [
        "Overall fit, with occasional calibration needed on feedback style (supportive vs direct) and priority setting.",
        "Minor friction may appear during high-pressure delivery where task focus increases or people focus becomes critical.",
        "Clear norms around communication typically keep alignment strong."
    ],
    "moderate": [
        "Noticeable differences in what is prioritized: well-being and cohesion vs speed and output.",
        "Risk: one side experiences task-focus as cold; the other experiences people-focus as inefficiency.",
        "Alignment improves if feedback norms and workload decisions are made explicit."
    ],
    "moderate_low": [
        "Frequent tension likely in performance conversations, pacing, and how workload trade-offs are justified.",
        "Risk of morale issues or perceived unfairness: direct execution culture may clash with high empathy expectations (or vice versa).",
        "Without explicit norms, communication style may be misread (too harsh vs too soft)."
    ],
    "low": [
        "High mismatch in socioemotional climate vs execution expectations; conflict and dissatisfaction are likely persistent.",
        "Strong risk of disengagement, feedback breakdown, and repeated interpersonal friction.",
        "Treat as a primary fit risk; requires coaching, role design, and communication governance."
    ],
}

DIMENSIONS_EN["D5_EJ"]["actionable_recommendations"] = {
    "onboarding": {
        "very_high": [
            "Confirm shared feedback tone and collaboration norms; keep it lightweight.",
            "Reinforce what ‘good teamwork’ means (either high care or high execution clarity—already aligned)."
        ],
        "high": [
            "Clarify feedback style expectations (directness, empathy) and how workload decisions are made.",
            "Agree on how to signal stress, overload, and priorities."
        ],
        "moderate": [
            "Create a working agreement: feedback tone, escalation norms, workload trade-offs, and performance expectations.",
            "Set explicit norms for meetings, communication, and decision-making speed vs inclusion."
        ],
        "moderate_low": [
            "Use structured onboarding: communication rules, feedback templates, and expectations for empathy/directness.",
            "Schedule early check-ins focusing on friction patterns (tone, workload, recognition)."
        ],
        "low": [
            "Treat as critical onboarding risk: establish explicit feedback rules and conflict-resolution protocol.",
            "Assign a mentor or manager to calibrate expectations and protect psychological safety."
        ],
    },
    "role_design": {
        "very_high": [
            "Place in roles consistent with the org’s culture (people-centered leadership roles or execution-heavy delivery roles)."
        ],
        "high": [
            "If task-oriented org: execution roles with clear outputs; if people-oriented org: roles requiring strong relational coordination."
        ],
        "moderate": [
            "Prefer roles with explicit performance expectations and clear collaboration boundaries.",
            "Avoid ambiguous roles where the definition of success is contested (people impact vs output)."
        ],
        "moderate_low": [
            "Avoid roles requiring constant high-stakes feedback unless communication norms are strong.",
            "Use pairing: strong delivery lead + strong people lead to balance climate and throughput."
        ],
        "low": [
            "Avoid roles where feedback and collaboration intensity is high without strong support.",
            "If unavoidable, redesign responsibilities and install robust communication scaffolding."
        ],
    },
    "coaching_training": {
        "very_high": [
            "Coach on scaling performance while preserving culture; focus on long-term growth."
        ],
        "high": [
            "Train on adaptive communication: when to be more direct vs more supportive depending on context."
        ],
        "moderate": [
            "Coach on reframing: people-focus sustains performance; task-focus protects delivery—both are valid strategies.",
            "Train on feedback models (clarity + empathy) to reduce misinterpretation."
        ],
        "moderate_low": [
            "Provide targeted coaching on either empathic communication or assertive delivery expectations depending on gap direction.",
            "Use scenario-based practice for performance feedback and conflict."
        ],
        "low": [
            "High-touch coaching required; misfit rarely self-corrects.",
            "If repeated friction persists, consider role change or different team placement."
        ],
    },
    "risk_monitoring": {
        "very_high": [
            "Monitor during high-pressure launches where task-focus typically increases."
        ],
        "high": [
            "Monitor tone and workload under pressure; ensure feedback remains constructive."
        ],
        "moderate": [
            "Watch for feedback breakdown, avoidance, or repeated complaints about tone and priorities."
        ],
        "moderate_low": [
            "Watch for morale decline, conflict escalation, and perception of unfairness in workload decisions."
        ],
        "low": [
            "Watch for chronic interpersonal conflict, disengagement, and high turnover intent; intervene early."
        ],
    },
}


In [412]:
DIMENSIONS_EN["D6_PP"]["practical_implications"] = {
    "very_high": [
        "Strong alignment in identity anchors: organizational belonging vs professional standards are similarly prioritized.",
        "Shared expectations about what legitimizes quality (internal norms vs external benchmarks).",
        "Low risk of conflict about ‘the right way’ to work or whose standards should prevail."
    ],
    "high": [
        "Overall identity fit, with occasional calibration needed around external best practices vs internal conventions.",
        "Minor friction may arise during audits, standards discussions, or career development decisions.",
        "Clear articulation of standards and growth pathways typically resolves tension."
    ],
    "moderate": [
        "Differences in identity anchoring may affect loyalty cues, decision framing, and quality criteria.",
        "Risk: professional-identity individuals may challenge local norms; parochial-identity individuals may perceive that as disloyalty.",
        "Alignment improves by clarifying which standards are non-negotiable and where local adaptation is expected."
    ],
    "moderate_low": [
        "Frequent tension likely around best practices, ethics, quality standards, and recognition/advancement criteria.",
        "Risk of trust erosion if one side sees the other as ‘too internal’ or ‘too external’ in allegiance.",
        "Without explicit norms, repeated debate over standards may slow delivery and reduce cohesion."
    ],
    "low": [
        "High mismatch in identity locus: conflict about legitimacy, standards, and loyalty is likely persistent.",
        "Strong risk of repeated friction in decision-making and performance evaluation criteria.",
        "Treat as a primary fit risk; requires explicit standards governance, mentoring, or role change."
    ],
}

DIMENSIONS_EN["D6_PP"]["actionable_recommendations"] = {
    "onboarding": {
        "very_high": [
            "Confirm shared view of standards (internal vs external) and keep alignment lightweight.",
            "Clarify how professional growth is recognized within the organization."
        ],
        "high": [
            "Explain which external standards matter and which internal norms are culturally important.",
            "Agree on how best practices are proposed, evaluated, and adopted."
        ],
        "moderate": [
            "Create a standards agreement: what is fixed (compliance, ethics, technical baselines) vs what is local convention.",
            "Make recognition criteria explicit: internal contribution vs external credibility."
        ],
        "moderate_low": [
            "Use structured onboarding: document norms, decision rights, and escalation paths for standards disputes.",
            "Schedule early checkpoints: monitor standard-related friction and perceived loyalty/identity signals."
        ],
        "low": [
            "Treat as critical onboarding risk: define clear governance for standards, ethics, and quality criteria.",
            "Assign a mentor who can translate between internal culture and external professional expectations."
        ],
    },
    "role_design": {
        "very_high": [
            "Place in roles where identity alignment naturally supports performance (culture-bearing roles or professional standards roles)."
        ],
        "high": [
            "If professional-oriented org: roles tied to external standards, quality, and benchmarking; if parochial-oriented: culture-centric coordination roles."
        ],
        "moderate": [
            "Prefer roles with explicit quality criteria and clear decision authority to avoid identity-driven conflict.",
            "Use roles that allow both: internal contribution + external best practice integration."
        ],
        "moderate_low": [
            "Avoid roles where standards disputes are constant unless governance is explicit.",
            "Pair internal culture carriers with external standards champions to balance cohesion and excellence."
        ],
        "low": [
            "Avoid roles requiring high identity alignment (leadership, culture shaping) if mismatch is large.",
            "If unavoidable, formalize standards governance and recognition criteria."
        ],
    },
    "coaching_training": {
        "very_high": [
            "Coach on scaling impact and developing leadership identity aligned with context."
        ],
        "high": [
            "Train on communicating standards without triggering identity conflict; build shared language for legitimacy."
        ],
        "moderate": [
            "Coach on reframing: internal loyalty and external standards are not enemies; they manage risk differently.",
            "Train on negotiation of norms and evidence-based adoption of best practices."
        ],
        "moderate_low": [
            "Provide targeted coaching on either cultural integration (for professional-oriented individuals) or external benchmarking (for parochial-oriented contexts).",
            "Use case-based coaching for standards disagreements."
        ],
        "low": [
            "High-touch coaching required; misfit rarely self-corrects.",
            "If persistent, consider role reassignment to reduce identity friction exposure."
        ],
    },
    "risk_monitoring": {
        "very_high": [
            "Monitor during major policy/standards shifts."
        ],
        "high": [
            "Monitor during audits or performance cycles where standards and recognition become salient."
        ],
        "moderate": [
            "Watch for repeated debates about ‘the right way’ and whether local norms or external standards prevail."
        ],
        "moderate_low": [
            "Watch for trust issues, perceived disloyalty, or resistance to benchmarking / change proposals."
        ],
        "low": [
            "Watch for chronic conflict, disengagement, and stalled decisions; intervene early with governance."
        ],
    },
}


In [485]:
def normalize_md_spacing(md_text: str) -> str:
    # Headings
    md_text = md_text.replace("\n## ", "\n\n## ")
    md_text = md_text.replace("\n### ", "\n\n### ")

    # Ensure space before lists
    md_text = md_text.replace("\n- ", "\n\n- ")

    # Collapse excess blank lines
    while "\n\n\n" in md_text:
        md_text = md_text.replace("\n\n\n", "\n\n")

    return md_text.strip() + "\n"


In [414]:
def render_bullets_flat(title: str, bullets: list) -> str:
    """DOCX-safe: title in bold + flat bullets (no nesting)."""
    if not bullets:
        return ""
    md = []
    md.append(f"**{title}**\n\n")
    for b in bullets:
        md.append(f"- {b}\n")
    md.append("\n")
    return "".join(md)

def render_reco_categories(recos: dict, level_key: str) -> str:
    if not recos:
        return ""

    category_titles = {
        "onboarding": "Onboarding recommendations",
        "role_design": "Role design recommendations",
        "coaching_training": "Coaching / training recommendations",
        "risk_monitoring": "Risks to monitor",
    }

    out = []
    out.append(f"**Recommendations ({level_key}).**\n\n")

    any_added = False
    for cat in ["onboarding", "role_design", "coaching_training", "risk_monitoring"]:
        if cat not in recos:
            continue

        lvl_block = recos.get(cat, {})
        bullets = []
        if isinstance(lvl_block, dict):
            bullets = lvl_block.get(level_key, []) or []
        elif isinstance(lvl_block, list):
            bullets = lvl_block
        bullets = [str(b).strip() for b in bullets if str(b).strip()]
        if not bullets:
            continue

        any_added = True
        out.append(f"**{category_titles.get(cat, cat)}**\n\n")
        for b in bullets:
            out.append(f"- {b}\n")
        out.append("\n")

    return "".join(out) if any_added else ""



In [415]:
def render_poles_bullets_md(section_title, poleA_label, itemsA, poleB_label, itemsB):
    """
    DOCX-safe Markdown:
    - No nested lists.
    - Uses bold headings for poles, then flat bullet lists.
    itemsA/itemsB can be:
      - list[str]  OR
      - list[dict] with keys {"title","desc"} (for cognitive)
    """
    md = []
    md.append(f"**{section_title}**\n\n")

    def render_items(items):
        lines = []
        if not items:
            return lines
        if isinstance(items[0], dict):
            for it in items:
                lines.append(f"- *{it['title']}*: {it['desc']}\n")
        else:
            for it in items:
                lines.append(f"- {it}\n")
        return lines

    md.append(f"**{poleA_label}**\n")
    md.extend(render_items(itemsA))
    md.append("\n")

    md.append(f"**{poleB_label}**\n")
    md.extend(render_items(itemsB))
    md.append("\n")

    return "".join(md)


In [416]:
# =========================================================
# Create org_thetas (synthetic demo)
# Balanced organization profile (close to 0)
# =========================================================

org_thetas = {
    "D1_GO":  0.30,  # slightly Goal-oriented
    "D2_IDED": 0.10, # slightly Internally driven
    "D3_ES":  0.20,  # slightly Structured
    "D4_OS":  0.40,  # Open System
    "D5_EJ":  0.15,  # slightly Task-oriented
    "D6_PP":  0.25,  # slightly Professional identity
}

print("✅ org_thetas created:")
org_thetas


✅ org_thetas created:


{'D1_GO': 0.3,
 'D2_IDED': 0.1,
 'D3_ES': 0.2,
 'D4_OS': 0.4,
 'D5_EJ': 0.15,
 'D6_PP': 0.25}

In [444]:
# ========== STEP 0: SANITY CHECK ==========

required_vars = [
    "person_thetas_hybrid",
    "org_thetas",
    "DIMENSIONS_EN",
    "generate_fit_report_en",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    print("❌ Missing in notebook:", missing)
else:
    print("✅ All required variables/functions exist!")

print("\nPerson thetas (hybrid):", person_thetas_hybrid)
print("\nOrg thetas:", org_thetas)
print("\nDIMENSIONS_EN keys:", list(DIMENSIONS_EN.keys())[:3], "... total:", len(DIMENSIONS_EN))


✅ All required variables/functions exist!

Person thetas (hybrid): {'D1_GO': 0.0, 'D2_IDED': -0.3333333333333333, 'D3_ES': 0.16666666666666666, 'D4_OS': 1.0, 'D5_EJ': 0.16666666666666666, 'D6_PP': 0.5}

Org thetas: {'D1_GO': 0.3, 'D2_IDED': 0.1, 'D3_ES': 0.2, 'D4_OS': 0.4, 'D5_EJ': 0.15, 'D6_PP': 0.25}

DIMENSIONS_EN keys: ['D1_GO', 'D2_IDED', 'D3_ES'] ... total: 6


In [500]:
import os

# ========== STEP 1: GENERATE REPORT (NLP thetas) ==========

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

order = ["D1_GO","D2_IDED","D3_ES","D4_OS","D5_EJ","D6_PP"]

outputs = generate_fit_report_en(
    person_thetas=person_thetas_hybrid,   # 👈 NLP output
    org_thetas=org_thetas,
    DIMENSIONS=DIMENSIONS_EN,
    order=order,
    out_basename="fit_report_nlp_demo",
    out_dir=OUT_DIR,
    
)

print(outputs)


{'md': 'outputs\\fit_report_nlp_demo.md', 'radar_png': 'outputs\\fit_report_nlp_demo_radar_dual.png', 'meta': {'report_id': 'CCF-20260209-002954', 'person_id': 'P-DEMO-01', 'organization_id': 'O-DEMO-01', 'model_version': 'v1.2', 'instrument_version': 'P-v1.0', 'date': '2026-02-09'}}


In [ ]:
import os
import subprocess
from pathlib import Path

# === CONFIG: ajusta si tu carpeta/nombre es distinto ===
OUT_DIR = "outputs"                 # carpeta donde quedó el .md
MD_NAME = "fit_report_nlp_demo.md"  # <- cambia si tu archivo se llama distinto

md_path = Path(OUT_DIR) / MD_NAME
docx_path = md_path.with_suffix(".docx")

print("MD:", md_path)
print("DOCX will be:", docx_path)

# --- checks ---
if not md_path.exists():
    raise FileNotFoundError(f"No encuentro el .md aquí: {md_path}")

# --- convert with pandoc ---
try:
    subprocess.run(
        ["pandoc", str(md_path), "-o", str(docx_path)],
        cwd=str(md_path.parent),
        check=True
    )
    print("✅ DOCX generado:", docx_path)
except FileNotFoundError:
    print("❌ No encontré 'pandoc' en tu PATH. Instálalo y reinicia Jupyter/Anaconda.")
except subprocess.CalledProcessError as e:
    print("❌ Pandoc falló. Error:", e)


In [ ]:
import subprocess
from pathlib import Path

OUT_DIR = Path("outputs")
md_path = OUT_DIR / "fit_report_nlp_demo.md"
docx_path = OUT_DIR / "fit_report_nlp_demo.docx"

cmd = ["pandoc", str(md_path), "-o", str(docx_path)]
p = subprocess.run(cmd, capture_output=True, text=True)

print("Return code:", p.returncode)
print("\n--- STDOUT ---\n", p.stdout)
print("\n--- STDERR ---\n", p.stderr)


In [ ]:
import subprocess
from pathlib import Path

OUT_DIR = Path("outputs")

md_name = "fit_report_nlp_demo.md"
docx_name = "fit_report_nlp_demo.docx"

# sanity: lista archivos por si quieres confirmar que el radar está ahí
print("Files in outputs:")
for p in sorted(OUT_DIR.iterdir()):
    if p.suffix.lower() in [".md", ".png", ".docx"]:
        print(" -", p.name)

# run pandoc from inside outputs
subprocess.run(
    ["pandoc", md_name, "-o", docx_name],
    cwd=str(OUT_DIR),
    check=True
)

print("✅ DOCX generado:", OUT_DIR / docx_name)


In [ ]:
from pathlib import Path

OUT_DIR = Path("outputs")
md = (OUT_DIR/"fit_report_nlp_demo.md").read_text(encoding="utf-8")

print("Radar referenced in MD?:", "fit_report_nlp_demo_radar_dual.png" in md)
print("Radar exists?:", (OUT_DIR/"fit_report_nlp_demo_radar_dual.png").exists())


In [ ]:
import subprocess
from pathlib import Path

OUT_DIR = Path("outputs")
md_name = "fit_report_nlp_demo.md"
docx_name = "fit_report_nlp_demo.docx"

subprocess.run(
    ["pandoc",
     "-f", "markdown+hard_line_breaks",  # 👈 clave para spacing
     md_name,
     "-o", docx_name],
    cwd=str(OUT_DIR),
    check=True
)

print("✅ DOCX generado (spacing fixed):", OUT_DIR / docx_name)


In [ ]:
subprocess.run(
    [
        "pandoc",
        "-f", "markdown+hard_line_breaks",
        "--wrap=none",
        md_name,
        "-o", docx_name
    ],
    cwd=str(OUT_DIR),
    check=True
)


## Evidence Pack - Simulation

In [ ]:
{
  "dimension": "D1_GO",
  "dimension_name": "Goal vs Means Orientation",
  "final_theta": 1.32,
  "final_pole": "Goal-Oriented",
  "confidence": 0.85,

  "evidence": {
    "llm": [
      {
        "text": "The priority was the result: we launched a functional version even if the code review was not as rigorous",
        "pole": "Goal-Oriented",
        "weight": 0.9,
        "reason": "Explicit prioritization of outcome over process"
      }
    ],
    "lexicon": [
      {
        "term": "resultado",
        "pole": "Goal-Oriented",
        "count": 2
      }
    ]
  },

  "notes": "Lexicon had low hits; LLM evidence weighted more heavily."
}


In [ ]:
def collect_evidence(dimension, llm_output, lex_hits):
    ...


In [ ]:
def explain_theta(dimension, theta_llm, theta_lex, theta_final, evidence):
    ...


In [ ]:
llm_evidence_D1_GO = [
    {
        "text": "The priority was the result: we launched a functional version even if the code review was not as rigorous as always.",
        "pole": "Goal-Oriented",
        "weight": 0.90,
        "reason": (
            "Explicit prioritization of outcome over process rigor; "
            "process standards were flexibly adjusted to achieve the goal."
        )
    },
    {
        "text": "I was flexible with our usual structure to avoid blocking the deployment.",
        "pole": "Goal-Oriented",
        "weight": 0.80,
        "reason": (
            "Demonstrates willingness to relax structural norms "
            "to ensure delivery and impact."
        )
    }
]


In [ ]:
lexicon_hits_D1_GO = {
    "Goal-Oriented": {
        "terms": ["priority", "result", "launched", "functional version"],
        "count": 0
    },
    "Means-Oriented": {
        "terms": ["process", "review", "procedure"],
        "count": 0
    }
}


In [ ]:
theta_lex = 0.0
theta_llm = 1.70
llm_confidence = 0.85


In [ ]:
theta_hybrid = round(
    (theta_llm * llm_confidence) + (theta_lex * (1 - llm_confidence)),
    3
)
# Resultado: 1.317


In [ ]:
evidence_pack_D1_GO = {
    "dimension": "D1_GO",
    "dimension_name": "Goal-Oriented vs Means-Oriented",
    "final_theta": 1.317,
    "final_pole": "Goal-Oriented",
    "confidence": 0.85,

    "evidence": {
        "llm": llm_evidence_D1_GO,
        "lexicon": lexicon_hits_D1_GO
    },

    "notes": (
        "Lexicon evidence was sparse; "
        "LLM evidence was weighted more heavily due to explicit outcome-focused language."
    )
}


In [ ]:
llm_evidence_D2_IDED = [
    {
        "text": "What drives me is the satisfaction of seeing that what I build works and helps someone.",
        "pole": "Internally Driven",
        "weight": 0.95,
        "reason": (
            "Motivation is explicitly described as intrinsic satisfaction derived from impact and usefulness, "
            "rather than external rewards."
        )
    },
    {
        "text": "I like having autonomy and being trusted in my professional judgment.",
        "pole": "Internally Driven",
        "weight": 0.90,
        "reason": (
            "Strong emphasis on autonomy and internal regulation of effort."
        )
    },
    {
        "text": "I don’t need to be praised every day if I see that my code is solid.",
        "pole": "Internally Driven",
        "weight": 0.85,
        "reason": (
            "Explicit devaluation of external recognition as a primary motivator."
        )
    }
]


In [ ]:
lexicon_hits_D2_IDED = {
    "Internally Driven": {
        "terms": ["satisfaction", "autonomy", "trusted", "professional judgment"],
        "count": 0
    },
    "Externally Driven": {
        "terms": ["praise", "reward", "recognition", "bonus"],
        "count": 0
    }
}


In [ ]:
theta_lex = 0.0
theta_llm = 1.80
llm_confidence = 0.90


In [ ]:
theta_hybrid = round(
    (theta_llm * llm_confidence) + (theta_lex * (1 - llm_confidence)),
    3
)
# Resultado: 1.440


In [ ]:
evidence_pack_D2_IDED = {
    "dimension": "D2_IDED",
    "dimension_name": "Internally Driven vs Externally Driven",
    "final_theta": 1.440,
    "final_pole": "Internally Driven",
    "confidence": 0.90,

    "evidence": {
        "llm": llm_evidence_D2_IDED,
        "lexicon": lexicon_hits_D2_IDED
    },

    "notes": (
        "Strong intrinsic motivation signals detected. "
        "External rewards and recognition are explicitly downplayed. "
        "Lexicon evidence was sparse; LLM evidence dominated the final score."
    )
}


In [ ]:
llm_evidence_D3_ES = [
    {
        "text": "The priority was the result: we released a functional version even though the code review process was not as rigorous as usual.",
        "pole": "Easygoing / Flexible",
        "weight": 0.90,
        "reason": (
            "Shows willingness to relax formal processes to adapt to situational urgency and ensure delivery."
        )
    },
    {
        "text": "I was flexible with our usual structure to avoid blocking the deployment.",
        "pole": "Easygoing / Flexible",
        "weight": 0.95,
        "reason": (
            "Explicit reference to flexibility in structure when context demands it."
        )
    },
    {
        "text": "Next time, I would reserve a technical buffer from the start so as not to sacrifice order for urgency.",
        "pole": "Balanced (Reflective Flexibility)",
        "weight": 0.75,
        "reason": (
            "Indicates meta-cognitive awareness: flexibility is used pragmatically, but structure is still valued."
        )
    }
]


In [ ]:
lexicon_hits_D3_ES = {
    "Easygoing / Flexible": {
        "terms": ["flexible", "avoid blocking", "priority", "urgency"],
        "count": 0
    },
    "Strict / Structured": {
        "terms": ["process", "code review", "structure", "order"],
        "count": 0
    }
}


In [ ]:
theta_lex = 0.0
theta_llm = -1.50   # negativo → polo Easygoing / Flexible
llm_confidence = 0.75


In [ ]:
theta_hybrid = round(
    (theta_llm * llm_confidence) + (theta_lex * (1 - llm_confidence)),
    3
)
# Resultado: -1.087


In [ ]:
evidence_pack_D3_ES = {
    "dimension": "D3_ES",
    "dimension_name": "Easygoing / Flexible vs Strict / Structured",
    "final_theta": -1.087,
    "final_pole": "Easygoing / Flexible",
    "confidence": 0.75,

    "evidence": {
        "llm": llm_evidence_D3_ES,
        "lexicon": lexicon_hits_D3_ES
    },

    "notes": (
        "Evidence indicates contextual flexibility rather than disregard for structure. "
        "Rules and processes are temporarily relaxed to achieve critical outcomes, "
        "with reflective awareness about restoring order in future scenarios. "
        "Lexicon signals were limited; LLM semantic reasoning dominated the score."
    )
}


In [ ]:
llm_evidence_D4_OS = [
    {
        "text": "If users say the interface is confusing, even if I love how the code turned out, I change it.",
        "pole": "Open System",
        "weight": 0.95,
        "reason": (
            "Explicit prioritization of external user feedback over internal preferences or technical pride."
        )
    },
    {
        "text": "I use feedback to pivot quickly.",
        "pole": "Open System",
        "weight": 0.90,
        "reason": (
            "Demonstrates rapid adaptation driven by external signals."
        )
    },
    {
        "text": "The success of my work is measured by how easily the customer uses the platform.",
        "pole": "Open System",
        "weight": 0.95,
        "reason": (
            "Defines success through external impact and user experience rather than internal metrics."
        )
    }
]


In [ ]:
lexicon_hits_D4_OS = {
    "Open System": {
        "terms": ["users", "feedback", "customer", "pivot"],
        "count": 0
    },
    "Closed System": {
        "terms": ["internal process", "policy", "procedure", "stability"],
        "count": 0
    }
}


In [ ]:
theta_lex = 0.0
theta_llm = 1.76   # positivo → Open System
llm_confidence = 0.88


In [ ]:
theta_hybrid = round(
    (theta_llm * llm_confidence) + (theta_lex * (1 - llm_confidence)),
    3
)
# Resultado: 1.549


In [ ]:
evidence_pack_D4_OS = {
    "dimension": "D4_OS",
    "dimension_name": "Open System vs Closed System",
    "final_theta": 1.549,
    "final_pole": "Open System",
    "confidence": 0.88,

    "evidence": {
        "llm": llm_evidence_D4_OS,
        "lexicon": lexicon_hits_D4_OS
    },

    "notes": (
        "Strong Open System orientation. Decisions are guided primarily by external feedback "
        "from users and customers, even when this requires abandoning internally preferred solutions. "
        "The individual defines success through external impact and usability rather than internal coherence."
    )
}


In [ ]:
llm_evidence_D5_EJ = [
    {
        "text": "If the team is at the limit, I prefer to sit down with the Project Manager and be honest.",
        "pole": "People-Oriented",
        "weight": 0.85,
        "reason": (
            "Shows explicit concern for team wellbeing and open communication under pressure."
        )
    },
    {
        "text": "A burned-out team makes expensive mistakes.",
        "pole": "People-Oriented",
        "weight": 0.90,
        "reason": (
            "Links human sustainability directly to quality and risk management."
        )
    },
    {
        "text": "For me, a 'good job' is one that meets industry standards, but that lands in the reality of the company.",
        "pole": "Balanced (People-leaning)",
        "weight": 0.75,
        "reason": (
            "Integrates task standards with organizational and human constraints."
        )
    }
]


In [ ]:
lexicon_hits_D5_EJ = {
    "People-Oriented": {
        "terms": ["team", "burned-out", "honest", "reality of the company"],
        "count": 0
    },
    "Task-Oriented": {
        "terms": ["deliverables", "performance", "efficiency", "targets"],
        "count": 0
    }
}


In [ ]:
theta_lex = 0.0
theta_llm = -1.40   # negativo → People-Oriented
llm_confidence = 0.70


In [ ]:
theta_hybrid = round(
    (theta_llm * llm_confidence) + (theta_lex * (1 - llm_confidence)),
    3
)
# Resultado: -0.980


In [ ]:
evidence_pack_D5_EJ = {
    "dimension": "D5_EJ",
    "dimension_name": "People-Oriented vs Task-Oriented",
    "final_theta": -0.980,
    "final_pole": "People-Oriented",
    "confidence": 0.70,

    "evidence": {
        "llm": llm_evidence_D5_EJ,
        "lexicon": lexicon_hits_D5_EJ
    },

    "notes": (
        "Clear People-Oriented tendency. The individual prioritizes team wellbeing "
        "and sustainability as prerequisites for quality and long-term performance. "
        "Task execution and standards are important but subordinated to human limits "
        "and organizational reality."
    )
}


In [ ]:
llm_evidence_D6_PP = [
    {
        "text": "A 'good job' is one that meets the standards of the industry.",
        "pole": "Professional Identity",
        "weight": 0.95,
        "reason": (
            "Explicit reference to industry standards as the primary criterion of quality."
        )
    },
    {
        "text": "Clean, scalable and accessible.",
        "pole": "Professional Identity",
        "weight": 0.90,
        "reason": (
            "Use of professional and technical quality standards independent of organizational context."
        )
    },
    {
        "text": "That lands in the reality of the company.",
        "pole": "Balanced (Professional-leaning)",
        "weight": 0.75,
        "reason": (
            "Professional standards are adapted to organizational constraints without abandoning them."
        )
    }
]


In [ ]:
lexicon_hits_D6_PP = {
    "Professional": {
        "terms": ["industry standards", "scalable", "clean", "accessible"],
        "count": 0
    },
    "Parochial": {
        "terms": ["internal norms", "company way", "organizational tradition"],
        "count": 0
    }
}


In [ ]:
theta_lex = 0.0
theta_llm = 1.84   # positivo → Professional Identity
llm_confidence = 0.92


In [ ]:
theta_hybrid = round(
    (theta_llm * llm_confidence) + (theta_lex * (1 - llm_confidence)),
    3
)
# Resultado: 1.693


In [ ]:
evidence_pack_D6_PP = {
    "dimension": "D6_PP",
    "dimension_name": "Parochial vs Professional Identity",
    "final_theta": 1.693,
    "final_pole": "Professional Identity",
    "confidence": 0.92,

    "evidence": {
        "llm": llm_evidence_D6_PP,
        "lexicon": lexicon_hits_D6_PP
    },

    "notes": (
        "Strong Professional Identity orientation. The individual defines quality "
        "and success through external professional and industry standards, while "
        "pragmatically adapting them to organizational constraints. Identity is "
        "anchored in expertise rather than institutional affiliation."
    )
}


In [ ]:
evidence_packs = {
    "D1_GO": evidence_pack_D1_GO,
    "D2_IDED": evidence_pack_D2_IDED,
    "D3_ES": evidence_pack_D3_ES,
    "D4_OS": evidence_pack_D4_OS,
    "D5_EJ": evidence_pack_D5_EJ,
    "D6_PP": evidence_pack_D6_PP,
}



In [ ]:
def build_person_profile_from_evidence(evidence_packs):
    profile = {}

    for dim, pack in evidence_packs.items():
        profile[dim] = {
            "theta": pack["final_theta"],
            "pole": pack["final_pole"],
            "confidence": pack["confidence"],
            "notes": pack.get("notes", "")
        }

    return profile


In [ ]:
person_profile = build_person_profile_from_evidence(evidence_packs)


In [ ]:
org_profile = {
    "D1_GO": 0.75,
    "D2_IDED": -0.10,
    "D3_ES": -0.75,
    "D4_OS": 0.25,
    "D5_EJ": 0.75,
    "D6_PP": 0.25,
}


In [ ]:
import numpy as np

def compute_fit(person_profile, org_profile):
    rows = []

    for dim, p in person_profile.items():
        theta_p = p["theta"]
        theta_o = org_profile[dim]

        delta = theta_p - theta_o
        abs_delta = abs(delta)

        rows.append({
            "dimension": dim,
            "person_theta": round(theta_p, 3),
            "org_theta": round(theta_o, 3),
            "delta_theta": round(delta, 3),
            "abs_delta": round(abs_delta, 3),
            "person_pole": p["pole"],
        })

    return rows


In [ ]:
fit_table = compute_fit(person_profile, org_profile)


In [ ]:
def overall_fit_metrics(fit_table):
    abs_deltas = [r["abs_delta"] for r in fit_table]

    mean_abs = np.mean(abs_deltas)
    median_abs = np.median(abs_deltas)

    return {
        "mean_abs_delta": round(mean_abs, 3),
        "median_abs_delta": round(median_abs, 3),
        "fit_label": global_fit_label(mean_abs)
    }


In [ ]:
report_input = {
    "person_profile": person_profile,
    "org_profile": org_profile,
    "fit_table": fit_table,
    "overall_fit": overall_fit_metrics(fit_table)
}


In [ ]:
import pandas as pd

# Orden recomendado (el mismo que tu reporte)
ORDER = ["D1_GO","D2_IDED","D3_ES","D4_OS","D5_EJ","D6_PP"]

def build_nlp_profile_table(evidence_packs, dimensions_meta=None, order=ORDER):
    """
    evidence_packs: dict {dim_code: evidence_pack}
    dimensions_meta: opcional, dict con nombres bonitos (DIMENSIONS_EN o similar)
    """
    rows = []

    for dim in order:
        pack = evidence_packs.get(dim)
        if pack is None:
            continue

        theta = pack.get("final_theta", None)
        pole  = pack.get("final_pole", "")
        conf  = pack.get("confidence", None)
        notes = pack.get("notes", "")

        # nombre legible (opcional)
        pretty_name = dim
        if dimensions_meta and dim in dimensions_meta:
            pretty_name = dimensions_meta[dim].get("name", dim)

        # evidencia LLM count (opcional)
        llm_evs = pack.get("evidence", {}).get("llm", [])
        evidence_n = len(llm_evs) if isinstance(llm_evs, list) else None

        rows.append({
            "dimension": dim,
            "name": pretty_name,
            "theta_nlp": theta,
            "pole_nlp": pole,
            "llm_conf": conf,
            "evidence_n": evidence_n,
            "notes": notes
        })

    df = pd.DataFrame(rows)

    # Redondeo y orden final
    if "theta_nlp" in df.columns:
        df["theta_nlp"] = df["theta_nlp"].astype(float).round(3)
    if "llm_conf" in df.columns and df["llm_conf"].notna().any():
        df["llm_conf"] = df["llm_conf"].astype(float).round(2)

    return df


In [ ]:
evidence_packs = {
    "D1_GO": evidence_pack_D1_GO,
    "D2_IDED": evidence_pack_D2_IDED,
    "D3_ES": evidence_pack_D3_ES,
    "D4_OS": evidence_pack_D4_OS,
    "D5_EJ": evidence_pack_D5_EJ,
    "D6_PP": evidence_pack_D6_PP,
}


In [ ]:
df_nlp = build_nlp_profile_table(evidence_packs, dimensions_meta=None)
df_nlp


In [ ]:
out_csv = "nlp_profile_table.csv"
df_nlp.to_csv(out_csv, index=False, encoding="utf-8")
print("Saved:", out_csv)


In [ ]:
df_nlp_paper = df_nlp[["dimension","theta_nlp","pole_nlp","llm_conf"]].copy()
df_nlp_paper


In [ ]:
df_extremes = df_nlp.copy()
df_extremes["abs_theta"] = df_extremes["theta_nlp"].abs()
df_extremes = df_extremes.sort_values("abs_theta", ascending=False)
df_extremes


In [ ]:
evidence_packs = {
    "D1_GO": evidence_pack_D1_GO,
    "D2_IDED": evidence_pack_D2_IDED,
    "D3_ES": evidence_pack_D3_ES,
    "D4_OS": evidence_pack_D4_OS,
    "D5_EJ": evidence_pack_D5_EJ,
    "D6_PP": evidence_pack_D6_PP,
}


In [ ]:
df_nlp = build_nlp_profile_table(evidence_packs, dimensions_meta=None)
df_nlp


In [ ]:
df_nlp = build_nlp_profile_table(evidence_packs, dimensions_meta=None)
df_nlp


In [ ]:
df_nlp.to_csv("nlp_profile_table.csv", index=False, encoding="utf-8")


In [ ]:
org_profile = {
    "D1_GO": 0.75,
    "D2_IDED": -0.10,
    "D3_ES": -0.75,
    "D4_OS": 0.25,
    "D5_EJ": 0.75,
    "D6_PP": 0.25,
}


In [ ]:
def build_person_org_gap_table(df_nlp, org_profile):
    df = df_nlp.copy()

    # θ organización
    df["theta_org"] = df["dimension"].map(org_profile)

    # deltas
    df["delta_theta"] = (df["theta_nlp"] - df["theta_org"]).round(3)
    df["abs_delta"] = df["delta_theta"].abs().round(3)

    # polos organización (por signo)
    def org_pole(theta, dim):
        if theta is None:
            return None
        meta = DIMENSIONS_EN.get(dim, {})
        if theta >= 0:
            return meta.get("poles", {}).get("B", {}).get("short", "Pole B")
        else:
            return meta.get("poles", {}).get("A", {}).get("short", "Pole A")

    df["org_pole"] = df.apply(
        lambda r: org_pole(r["theta_org"], r["dimension"]),
        axis=1
    )

    # ranking: menor gap = mejor fit
    df = df.sort_values("abs_delta", ascending=True).reset_index(drop=True)
    df["fit_rank"] = df.index + 1

    return df


In [ ]:
df_fit = build_person_org_gap_table(df_nlp, org_profile)
df_fit


In [ ]:
top_strengths = df_fit.head(2)
top_risks = df_fit.tail(2)

top_strengths, top_risks


In [ ]:
df_fit.to_csv("fit_person_vs_org.csv", index=False, encoding="utf-8")


In [ ]:
def needs_fallback(evidence_pack, min_conf=0.6, min_evidence=1):
    conf = evidence_pack.get("confidence", 0)
    ev_n = len(evidence_pack.get("evidence", {}).get("llm", []))
    return (conf < min_conf) or (ev_n < min_evidence)


In [ ]:
needs_fallback(evidence_pack_D1_GO)


In [ ]:
def enrich_with_fallback(evidence_packs, answers_by_dimension):
    """
    answers_by_dimension: dict {dim: text_answer}
    """
    updated = {}

    for dim, pack in evidence_packs.items():
        if needs_fallback(pack):
            extra_text = answers_by_dimension.get(dim)
            if extra_text:
                # Aquí reaplicas tu pipeline NLP
                updated_pack = run_nlp_pipeline(extra_text, dim)
                updated[dim] = updated_pack
            else:
                updated[dim] = pack
        else:
            updated[dim] = pack

    return updated


## LLM Integration

In [ ]:
import json
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

# ---- Canonical pole labels per dimension (must match your DIMENSIONS_EN poles) ----
POLES_CANON = {
    "D1_GO": ["Means-Oriented", "Goal-Oriented"],
    "D2_IDED": ["Externally Driven", "Internally Driven"],
    "D3_ES": ["Easygoing / Flexible", "Strict / Structured"],
    "D4_OS": ["Closed System", "Open System"],
    "D5_EJ": ["People-Oriented", "Task-Oriented"],
    "D6_PP": ["Parochial Identity", "Professional Identity"],
}

@dataclass
class LLMExtraction:
    dimension: str
    pole: str
    confidence: float
    evidence: List[str]
    notes: str = ""

def validate_llm_json(obj: Dict[str, Any]) -> LLMExtraction:
    # required fields
    for k in ["dimension", "pole", "confidence", "evidence"]:
        if k not in obj:
            raise ValueError(f"Missing field: {k}")

    dim = obj["dimension"]
    pole = obj["pole"]
    conf = float(obj["confidence"])
    ev = obj["evidence"]
    notes = obj.get("notes", "")

    if dim not in POLES_CANON:
        raise ValueError(f"Unknown dimension: {dim}")

    if pole not in POLES_CANON[dim]:
        raise ValueError(f"Pole '{pole}' not allowed for {dim}. Allowed: {POLES_CANON[dim]}")

    if not (0.0 <= conf <= 1.0):
        raise ValueError("confidence must be between 0 and 1")

    if not isinstance(ev, list) or any(not isinstance(x, str) for x in ev):
        raise ValueError("evidence must be a list of strings")

    # keep evidence short-ish (optional guard)
    ev = ev[:5]

    return LLMExtraction(dimension=dim, pole=pole, confidence=conf, evidence=ev, notes=notes)


In [ ]:
def build_llm_prompt(text: str, dimension: str) -> str:
    poles = POLES_CANON[dimension]
    return f"""
You are an evidence extractor for organizational culture inference.
Analyze the text ONLY for dimension: {dimension}.

Allowed poles (choose exactly one):
- {poles[0]}
- {poles[1]}

Return ONLY valid JSON with this schema:
{{
  "dimension": "{dimension}",
  "pole": "<one of the allowed poles>",
  "confidence": <float between 0 and 1>,
  "evidence": ["<short quote 1>", "<short quote 2>"],
  "notes": "<optional short rationale>"
}}

Rules:
- Evidence must be short excerpts or paraphrases grounded in the text.
- If insufficient evidence, set confidence <= 0.55 and keep evidence minimal.
- Do NOT output any text outside JSON.

TEXT:
\"\"\"{text}\"\"\"
""".strip()


In [ ]:
# ---- Offline stub: replace later with a real LLM call ----
STUB_DB = {
    "D1_GO": {"pole":"Goal-Oriented","confidence":0.85,"evidence":[
        "The priority was the result: we shipped a functional version.",
        "I would document those shortcuts next time."
    ],"notes":"Outcome-first with controlled trade-off."},
    "D2_IDED": {"pole":"Internally Driven","confidence":0.90,"evidence":[
        "What motivates me is seeing what I build helps someone.",
        "I don't need daily praise if I see the code is solid."
    ],"notes":"Intrinsic motivation + autonomy."},
    "D3_ES": {"pole":"Easygoing / Flexible","confidence":0.75,"evidence":[
        "I was flexible with our usual structure to avoid blocking the deployment."
    ],"notes":"Flexibility under pressure."},
    "D4_OS": {"pole":"Open System","confidence":0.88,"evidence":[
        "If users say the interface is confusing, I change it.",
        "I use feedback to pivot quickly."
    ],"notes":"External feedback integration."},
    "D5_EJ": {"pole":"People-Oriented","confidence":0.70,"evidence":[
        "A burned-out team makes costly mistakes.",
        "I'd rather slow down than deliver garbage."
    ],"notes":"Team wellbeing prioritized."},
    "D6_PP": {"pole":"Professional Identity","confidence":0.92,"evidence":[
        "A good job meets industry standards (clean, scalable).",
        "I do things right by professional ethics."
    ],"notes":"External standards + professional principles."},
}

def llm_extract_stub(text: str, dimension: str) -> LLMExtraction:
    base = STUB_DB[dimension]
    obj = {
        "dimension": dimension,
        "pole": base["pole"],
        "confidence": base["confidence"],
        "evidence": base["evidence"],
        "notes": base.get("notes","")
    }
    return validate_llm_json(obj)


In [ ]:
def pole_to_theta_llm(dimension: str, pole: str, confidence: float) -> float:
    # pole A -> negative, pole B -> positive
    A, B = POLES_CANON[dimension]
    sign = 1 if pole == B else -1
    mag = 1.0 + float(confidence)  # 1.0..2.0
    return round(sign * mag, 3)


In [ ]:
def build_llm_evidence_packs(text: str, dims=("D1_GO","D2_IDED","D3_ES","D4_OS","D5_EJ","D6_PP")):
    packs = {}
    for d in dims:
        ex = llm_extract_stub(text, d)  # replace with real call later
        theta_llm = pole_to_theta_llm(ex.dimension, ex.pole, ex.confidence)
        packs[d] = {
            "dimension": d,
            "llm": {
                "pole": ex.pole,
                "confidence": ex.confidence,
                "evidence": ex.evidence,
                "notes": ex.notes,
                "theta_llm": theta_llm
            }
        }
    return packs


In [ ]:
def hybrid_theta(theta_lex: float, theta_llm: float, llm_conf: float) -> float:
    w = float(llm_conf)
    return round(w*theta_llm + (1-w)*theta_lex, 3)


In [ ]:
sample_text = """
1. Sobre la presión y la priorización: "Hace poco tuvimos que sacar una actualización crítica de seguridad antes de un fin de semana largo. La prioridad fue el resultado: lanzamos una versión funcional aunque el proceso de code review no fue tan riguroso como siempre. Fui flexible con nuestra estructura habitual para no bloquear el despliegue, pero me aseguré de que las funciones críticas no se rompieran. ¿Qué haría distinto? Documentar esos 'atajos' en el momento para no dejarlos en el olvido. La próxima vez, reservaría un 'buffer' técnico desde el inicio para no tener que sacrificar el orden por la urgencia."

2. Sobre la motivación y el feedback del mercado: "Lo que me mueve es la satisfacción de ver que lo que construyo funciona y ayuda a alguien; me gusta tener autonomía y que confíen en mi criterio profesional. No necesito que me feliciten cada día si veo que mi código es sólido. Respecto al mercado, no me caso con mis ideas: si los usuarios dicen que la interfaz es confusa, aunque a mí me encante cómo quedó el código, la cambio. Uso el feedback para pivotar rápido; al final, el éxito de mi trabajo se mide por la facilidad con la que el cliente usa la plataforma, no por mi ego."

3. Sobre el bienestar del equipo y los estándares: "Si el equipo está al límite, prefiero sentarme con el Project Manager y ser honesto: 'o bajamos el ritmo o vamos a entregar basura'. Un equipo quemado comete errores caros. Para mí, un 'buen trabajo' es el que cumple con los estándares de la industria (limpio, escalable y accesible), pero que aterriza en la realidad de la empresa. No saco nada con usar la última tecnología de moda si el resto del equipo no la entiende o no ayuda al negocio. Busco ese equilibrio: hacer las cosas bien por ética profesional, pero que le sirvan a la organización.
"""

llm_packs = build_llm_evidence_packs(sample_text)

# Mostrar 1 dimensión como ejemplo
llm_packs["D1_GO"]


In [ ]:
person_id = "P_demo_001"
org_id = "Org_demo_001"

# Pega aquí el texto de la persona (entrevista / respuestas)
person_text = sample_text  # o pega string directamente


In [ ]:
# ======= TU FUNCIÓN LEXICON (reemplaza por la tuya real) =======
# Debe devolver:
# - theta_lex (float)
# - lex_hits_total (int)
# - opcional: hits_detail (dict/list) si lo tienes

def compute_theta_lex_and_hits(text: str, dimension: str):
    # TODO: reemplaza con tu implementación real
    # Si aún no la tienes a mano, deja esto temporalmente.
    return 0.0, 0


In [ ]:
import pandas as pd

DIMS_ORDER = ["D1_GO","D2_IDED","D3_ES","D4_OS","D5_EJ","D6_PP"]

# 1) LLM packs (stub por ahora)
llm_packs = build_llm_evidence_packs(person_text, dims=DIMS_ORDER)

rows = []
for d in DIMS_ORDER:
    # 2) Lexicon score
    theta_lex, lex_hits_total = compute_theta_lex_and_hits(person_text, d)

    # 3) LLM score
    pole = llm_packs[d]["llm"]["pole"]
    conf = llm_packs[d]["llm"]["confidence"]
    theta_llm = llm_packs[d]["llm"]["theta_llm"]
    notes = llm_packs[d]["llm"].get("notes","")
    evidence = llm_packs[d]["llm"].get("evidence", [])
    evidence_n = len(evidence)

    # 4) Hybrid
    theta_h = hybrid_theta(theta_lex, theta_llm, conf)

    # 5) Notes when lexicon is weak
    extra_note = ""
    if lex_hits_total == 0:
        extra_note = "Lexicon low hits: LLM weighted more"

    rows.append({
        "person_id": person_id,
        "org_id": org_id,
        "dimension": d,
        "theta_lex": theta_lex,
        "theta_llm": theta_llm,
        "theta_hybrid": theta_h,
        "llm_conf": conf,
        "llm_pole": pole,
        "evidence_n": evidence_n,
        "lex_hits_total": lex_hits_total,
        "notes": (extra_note + (" | " if extra_note and notes else "") + notes).strip()
    })

df_hybrid = pd.DataFrame(rows)

df_hybrid


In [ ]:
# Check dimensiones completas
missing = set(DIMS_ORDER) - set(df_hybrid["dimension"].tolist())
print("Missing dimensions:", missing)

# Conf en rango
bad_conf = df_hybrid[(df_hybrid["llm_conf"] < 0) | (df_hybrid["llm_conf"] > 1)]
print("Bad confidence rows:", len(bad_conf))

# NaNs
print("NaNs per column:\n", df_hybrid.isna().sum())


In [ ]:
def evidence_flag(row, min_conf=0.60, min_evidence=1):
    if row["llm_conf"] < min_conf:
        return "LOW_CONF"
    if row["evidence_n"] < min_evidence:
        return "LOW_EVIDENCE"
    return "OK"

df_hybrid["evidence_flag"] = df_hybrid.apply(evidence_flag, axis=1)
df_hybrid[["dimension","llm_conf","evidence_n","lex_hits_total","evidence_flag"]]


In [ ]:
out_csv = "outputs/nlp_hybrid_scores_demo.csv"
import os
os.makedirs("outputs", exist_ok=True)
df_hybrid.to_csv(out_csv, index=False)
print("Saved:", out_csv)


In [ ]:
# person_thetas para el reporte (usando theta_hybrid)
person_thetas = {row["dimension"]: float(row["theta_hybrid"]) for _, row in df_hybrid.iterrows()}
person_thetas


In [ ]:
# Muestra todas las funciones definidas en el entorno actual
[name for name in globals().keys() if callable(globals()[name])]


In [ ]:
required = ["mini_score_all_dimensions", "pole_to_sign"]
for fn in required:
    if fn not in globals():
        print("❌ Missing:", fn)
    else:
        print("✅ Found:", fn)


In [ ]:
import numpy as np
import pandas as pd


In [ ]:
text = """
We shipped a functional version quickly, prioritizing results over strict process.
I care about autonomy and using feedback to improve outcomes.
"""

llm_json = {
    "D1_GO": {
        "pole": "Goal-Oriented",
        "confidence": 0.85,
        "evidence": ["We shipped a functional version quickly"],
        "tags": ["outcome", "delivery"]
    },
    "D2_IDED": {
        "pole": "Internally Driven",
        "confidence": 0.90,
        "evidence": ["I care about autonomy"],
        "tags": ["autonomy"]
    }
}


In [ ]:
df, person_thetas = fuse_all_dimensions(text, llm_json)


In [ ]:
df[[
    "dimension",
    "theta_lex",
    "theta_llm",
    "theta_hybrid",
    "llm_conf",
    "lex_hits_total",
    "notes"
]]


In [ ]:
df["needs_followup"] = df["notes"].str.contains("follow-up", case=False, na=False)
df


In [ ]:
df["abs_delta_from_lex"] = (df["theta_hybrid"] - df["theta_lex"]).abs()
df


In [ ]:
df["needs_followup"] = df["notes"].str.contains("follow-up", case=False, na=False)
df[["dimension","theta_lex","theta_llm","theta_hybrid","llm_conf","lex_hits_total","notes","needs_followup"]]


In [ ]:
org_thetas = {
    "D1_GO": 0.8,
    "D2_IDED": 0.6,
    "D3_ES": -0.7,
    "D4_OS": 0.5,
    "D5_EJ": -0.4,
    "D6_PP": 1.0
}


In [ ]:
df, person_thetas = fuse_all_dimensions(text, llm_json)


In [ ]:
outputs = generate_fit_report_en(
    person_thetas=person_thetas,
    org_thetas=org_thetas,
    DIMENSIONS=DIMENSIONS_EN,
    order=["D1_GO","D2_IDED","D3_ES","D4_OS","D5_EJ","D6_PP"],
    out_basename="fit_report_nlp_demo1",
    out_dir="outputs"
)

outputs


In [ ]:
outputs


In [ ]:
!pandoc --version


In [ ]:
outputs = generate_fit_report_en(
    person_thetas,
    org_thetas,
    DIMENSIONS_EN,
    order=["D1_GO","D2_IDED","D3_ES","D4_OS","D5_EJ","D6_PP"],
    out_basename="fit_report_nlp_demo_X",
    out_dir="outputs",
    cap=1.5,
    meta={"person_id":"P-DEMO-01","organization_id":"O-DEMO-01","model_version":"v1.2"}
)


In [ ]:
!pandoc --version


In [ ]:
import os, subprocess, textwrap, datetime

out_dir = "outputs"
os.makedirs(out_dir, exist_ok=True)

md_test = os.path.join(out_dir, "pandoc_test.md")
docx_test = os.path.join(out_dir, "pandoc_test.docx")

with open(md_test, "w", encoding="utf-8") as f:
    f.write(textwrap.dedent(f"""\
    # Pandoc Test
    Generated: {datetime.datetime.now().isoformat()}

    - Bullet 1
    - Bullet 2

    **Bold text** and a paragraph.
    """))

res = subprocess.run(
    ["pandoc", md_test, "-o", docx_test],
    capture_output=True,
    text=True
)

print("Return code:", res.returncode)
print("STDERR:\n", res.stderr[:2000])
print("DOCX exists?", os.path.exists(docx_test), docx_test)


In [ ]:
import os, subprocess

md_file = os.path.join("outputs", "fit_report_nlp_demo.md")
out_docx = os.path.join("outputs", "fit_report_nlp_demo.docx")

res = subprocess.run(
    ["pandoc", os.path.basename(md_file), "-o", os.path.basename(out_docx)],
    cwd=os.path.dirname(md_file),          # clave
    capture_output=True,
    text=True
)

print("Return code:", res.returncode)
print("STDERR:\n", res.stderr[:3000])
print("DOCX exists?", os.path.exists(out_docx), out_docx)


## Evidence Pack

In [ ]:
# =========================================================
# Evidence Pack LIGHT → Integración en Reporte NLP (EN)
# - Mantiene el evidence pack completo en notebook/JSON
# - Solo inserta 2–3 evidencias por dimensión en el reporte
# =========================================================

import os
import subprocess
import pandas as pd
import numpy as np

# -----------------------------
# 1) Construir Evidence Pack light desde tu df (hybrid table)
# -----------------------------
def build_evidence_light_from_df(df_hybrid, llm_json=None, top_k=3):
    """
    df_hybrid: dataframe que ya generas con fuse_all_dimensions(...).
              Idealmente contiene: dimension, llm_conf, llm_pole, notes, lex_hits_total, etc.
    llm_json:  dict (si lo tienes) con evidencias completas por dimensión (recomendado).
              Estructura: llm_json[dim] = {"pole":..., "confidence":..., "evidence":[...], "tags":[...], "notes":...}
    top_k:     máximo de snippets a mostrar en el reporte por dimensión
    """
    evidence_light = {}

    for _, r in df_hybrid.iterrows():
        dim = r["dimension"]
        conf = float(r.get("llm_conf", 0.0) or 0.0)
        pole = r.get("llm_pole", None)
        notes = r.get("notes", "")

        lex_hits = int(r.get("lex_hits_total", 0) or 0)

        # evidencia: preferimos llm_json si está disponible (porque df suele traer solo counts)
        evid_list = []
        tags_list = []

        if llm_json is not None and isinstance(llm_json, dict) and dim in llm_json:
            pole = llm_json[dim].get("pole", pole)
            conf = float(llm_json[dim].get("confidence", conf) or conf)
            evid_list = llm_json[dim].get("evidence", []) or []
            tags_list = llm_json[dim].get("tags", []) or []
            notes = llm_json[dim].get("notes", notes) or notes

        # recorta evidencias
        evid_list = [e.strip() for e in evid_list if isinstance(e, str) and e.strip()]
        evid_list = evid_list[:top_k]

        evidence_light[dim] = {
            "pole": pole,
            "confidence": conf,
            "evidence": evid_list,
            "tags": tags_list[:6],
            "lex_hits_total": lex_hits,
            "notes": notes
        }

    return evidence_light


# -----------------------------
# 2) Render de Evidence Pack light en Markdown
# -----------------------------
def render_evidence_light_md(dim, ev):
    """
    Devuelve un bloque Markdown corto y "docx-friendly".
    """
    if ev is None:
        return ""

    pole = ev.get("pole", ev.get("llm_pole", None))
    conf = ev.get("confidence", ev.get("llm_conf", 0.0))
    evid = ev.get("evidence", []) or []
    lex_hits = ev.get("lex_hits_total", 0)
    notes = ev.get("notes", "")

    # Si no hay señal del LLM y tampoco evidencias, no insertamos nada
    if (pole is None or str(pole).strip() == "") and len(evid) == 0 and conf == 0:
        return ""

    md = []
    md.append("**Evidence Pack (light)**")
    md.append("")
    md.append(f"- **LLM inferred pole:** {pole if pole else 'N/A'}")
    md.append(f"- **LLM confidence:** {conf:.2f}")
    md.append(f"- **Lexicon signal (hits):** {lex_hits}")
    if notes:
        md.append(f"- **Notes:** {notes}")

    # Evidencias (2–3 bullets)
    if len(evid) > 0:
        md.append("")
        md.append("**Textual evidence (snippets)**")
        md.append("")
        for e in evid:
            md.append(f"- {e}")

    md.append("")  # cierre con línea en blanco (importante para spacing en docx)
    return "\n".join(md)


# -----------------------------
# 3) Hook: insertar Evidence Pack light en tu reporte por dimensión
#    (usa esta función dentro de tu builder MD)
# -----------------------------
def append_dimension_block_with_evidence(dim_code, dim_block_md, evidence_light=None):
    """
    dim_code: "D1_GO"..."D6_PP"
    dim_block_md: el markdown ya generado para esa dimensión (tu plantilla actual EN)
    evidence_light: dict construido con build_evidence_light_from_df(...)
    """
    ev_md = ""
    if evidence_light is not None and dim_code in evidence_light:
        ev_md = render_evidence_light_md(dim_code, evidence_light[dim_code])

    # Inserta el evidence pack AL FINAL del bloque de la dimensión
    # (queda perfecto: primero diagnóstico/atributos/conductas/recomendaciones, luego evidencia)
    if ev_md.strip():
        return dim_block_md.rstrip() + "\n\n" + ev_md + "\n"
    return dim_block_md


# -----------------------------
# 4) Export robusto: MD → DOCX con Pandoc (cwd=out_dir)
# -----------------------------
def md_to_docx(md_file, out_docx):
    md_dir = os.path.dirname(md_file)
    md_base = os.path.basename(md_file)
    docx_base = os.path.basename(out_docx)

    res = subprocess.run(
        ["pandoc", md_base, "-o", docx_base],
        cwd=md_dir,
        capture_output=True,
        text=True
    )
    if res.returncode != 0:
        raise RuntimeError(f"Pandoc failed:\n{res.stderr}")

    return out_docx


# =========================================================
# ✅ USO (ejemplo)
# =========================================================
# 1) Asumiendo que ya tienes:
#    - df_hybrid  (tabla del híbrido por dimensión)
#    - person_thetas (dict)
#    - org_thetas    (dict)
#    - DIMENSIONS_EN (diccionario maestro en inglés)
#
# 2) Si tienes también llm_json con evidencias completas, pásalo:
#    evidence_light = build_evidence_light_from_df(df_hybrid, llm_json=llm_json, top_k=3)
#    Si NO lo tienes, igual funciona pero solo con lo que venga en df_hybrid:
#    evidence_light = build_evidence_light_from_df(df_hybrid, llm_json=None, top_k=3)

# evidence_light = build_evidence_light_from_df(df_hybrid, llm_json=llm_json, top_k=3)

# --- Luego, dentro de TU builder (donde generas cada bloque por dimensión), haces:
# dim_md = build_dimension_section_md_EN(dim_code, ...)   # tu función actual
# dim_md = append_dimension_block_with_evidence(dim_code, dim_md, evidence_light)

# Finalmente exportas:
# md_file = "outputs/fit_report_nlp_demo.md"
# out_docx = "outputs/fit_report_nlp_demo.docx"
# md_to_docx(md_file, out_docx)


In [ ]:
evidence_light = build_evidence_light_from_df(df_hybrid, llm_json=llm_json, top_k=3)


In [ ]:
import inspect
print(inspect.signature(dimension_block_en))


In [428]:
def fused_to_evidence_light(fused: dict) -> dict:
    """
    Adapter: fuse_one_dimension output -> render_evidence_light_md expected schema
    Expected by renderer:
      { pole, confidence, evidence, lex_hits_total, notes }
    """
    if fused is None:
        return {}

    return {
        "pole": fused.get("llm_pole", None),
        "confidence": float(fused.get("llm_conf", 0.0) or 0.0),
        "evidence": fused.get("evidence", []) or [],
        "lex_hits_total": int(fused.get("lex_hits_total", 0) or 0),
        "notes": fused.get("notes", "") or ""
    }


In [429]:
print("Has practical?", "practical_implications" in DIMENSIONS_EN["D1_GO"])
print("Has recos?", "actionable_recommendations" in DIMENSIONS_EN["D1_GO"])
print("Practical keys:", list(DIMENSIONS_EN["D1_GO"].get("practical_implications", {}).keys()))
print("Reco cats:", list(DIMENSIONS_EN["D1_GO"].get("actionable_recommendations", {}).keys()))


Has practical? True
Has recos? True
Practical keys: ['very_high', 'high', 'moderate', 'moderate_low', 'low']
Reco cats: ['onboarding', 'role_design', 'coaching_training', 'risk_monitoring']


In [430]:
print("id(DIMENSIONS_EN):", id(DIMENSIONS_EN))
print("id(DIMENSIONS):", id(DIMENSIONS))
print("id(DIMENSIONS_EN['D1_GO']):", id(DIMENSIONS_EN["D1_GO"]))
print("id(DIMENSIONS['D1_GO']):", id(DIMENSIONS["D1_GO"]))


id(DIMENSIONS_EN): 2117538959872
id(DIMENSIONS): 2117538959872
id(DIMENSIONS_EN['D1_GO']): 2117538957184
id(DIMENSIONS['D1_GO']): 2117538957184


In [432]:
print("Has practical?", "practical_implications" in DIMENSIONS["D1_GO"])
print("Has recos?", "actionable_recommendations" in DIMENSIONS["D1_GO"])
print("Practical keys:", list(DIMENSIONS["D1_GO"].get("practical_implications", {}).keys()))
print("Reco cats:", list(DIMENSIONS["D1_GO"].get("actionable_recommendations", {}).keys()))


Has practical? True
Has recos? True
Practical keys: ['very_high', 'high', 'moderate', 'moderate_low', 'low']
Reco cats: ['onboarding', 'role_design', 'coaching_training', 'risk_monitoring']


In [433]:
text_input = """
1) When I had to deliver an important outcome under strong time pressure, I focused first on clarifying the final objective and the key success criteria. I identified which steps were truly necessary and which could be simplified or postponed. I followed core procedures related to quality and security, but allowed flexibility in internal workflows to save time. I prioritized communication with stakeholders and kept short feedback loops. If I faced a similar situation again, I would involve the team earlier in defining priorities and document key decisions more clearly.

2) What keeps me motivated when recognition or supervision is minimal is the sense of responsibility toward the final impact of my work. I value knowing that my contribution is useful for others and improves real outcomes. I regularly analyze customer and market feedback, especially negative signals, and use it to adjust my methods, refine processes, and redefine priorities. I see feedback as a learning mechanism rather than as criticism.

3) When there is tension between team well-being and performance goals, I try to balance both by first understanding the underlying causes of stress or overload. I usually negotiate realistic timelines and redistribute tasks when possible. I believe sustainable performance requires psychological safety and trust. When defining what “good work” looks like, I tend to rely on both internal organizational standards and external professional benchmarks, using internal guidelines as a baseline and external standards as a reference for continuous improvement.
"""


In [438]:
# Mock LLM output for testing (demo)

llm_json = {
    "D1_GO": {
        "pole": "B",
        "confidence": 0.82,
        "evidence": [
            "focused on clarifying the final objective",
            "prioritized deliverables under time pressure",
            "allowed flexibility in workflows"
        ],
        "tags": ["goal_focus", "outcome_driven", "flexibility"]
    },
    "D2_IDED": {
        "pole": "B",
        "confidence": 0.65,
        "evidence": [
            "sense of responsibility toward final impact",
            "values transparency and accountability"
        ],
        "tags": ["ethics", "accountability"]
    },
    "D3_ES": {
        "pole": "A",
        "confidence": 0.58,
        "evidence": [
            "followed core procedures",
            "kept short feedback loops"
        ],
        "tags": ["process", "structure"]
    },
    "D4_OS": {
        "pole": "B",
        "confidence": 0.60,
        "evidence": [
            "tolerance for method flexibility",
            "adapted approach based on feedback"
        ],
        "tags": ["adaptability", "openness"]
    },
    "D5_EJ": {
        "pole": "B",
        "confidence": 0.74,
        "evidence": [
            "negotiated realistic timelines",
            "focused on psychological safety"
        ],
        "tags": ["emotional_regulation", "empathy"]
    },
    "D6_PP": {
        "pole": "B",
        "confidence": 0.80,
        "evidence": [
            "balanced team well-being and performance",
            "redistributed tasks to reduce overload"
        ],
        "tags": ["team_focus", "support"]
    }
}


In [439]:
df_fused, person_thetas_hybrid, evidence_by_dim = fuse_all_dimensions(text_input, llm_json)


In [436]:
print("Evidence keys:", evidence_by_dim.keys())


Evidence keys: dict_keys(['D1_GO', 'D2_IDED', 'D3_ES', 'D4_OS', 'D5_EJ', 'D6_PP'])


In [437]:
print(dimension_block_en.__defaults__)


(1.5, None)


In [424]:
import inspect
print(inspect.signature(dimension_block_en))


(dim_key: str, person_theta: float, org_theta: float, dim_cfg: dict, cap: float = 1.5, evidence_light=None) -> str


In [447]:
# Demo organizational profile (example)

org_thetas_example = {
    "D1_GO":  0.65,   # Goal vs Means (moderately goal-oriented)
    "D2_IDED": 0.70,  # Integrity / Decision Ethics (high)
    "D3_ES":  0.45,   # Execution & Structure (moderate)
    "D4_OS":  0.20,   # Openness / Social adaptability (low-moderate)
    "D5_EJ":  0.30,   # Emotional judgment (moderate)
    "D6_PP":  0.80    # People / Prosociality (high)
}


In [449]:
ORDER = ["D1_GO", "D2_IDED", "D3_ES", "D4_OS", "D5_EJ", "D6_PP"]


In [454]:
def render_evidence_light_md(dim_key: str, ev: dict) -> str:
    """
    Short, docx-friendly Evidence Pack block.
    Expected ev schema:
      { pole, confidence, evidence, lex_hits_total, notes }
    """
    if not ev:
        return ""

    pole = ev.get("pole", ev.get("llm_pole", None))
    conf = float(ev.get("confidence", ev.get("llm_conf", 0.0)) or 0.0)
    evid = ev.get("evidence", []) or []
    lex_hits = int(ev.get("lex_hits_total", 0) or 0)
    notes = (ev.get("notes", "") or "").strip()

    # If there's truly nothing, don't render
    if (pole is None or str(pole).strip() == "") and not evid and conf == 0.0 and lex_hits == 0 and not notes:
        return ""

    out = []
    out.append("**Evidence Pack (light).**\n\n")

    if pole is not None and str(pole).strip():
        out.append(f"- **Inferred pole:** {str(pole).strip()}\n")
    if conf:
        out.append(f"- **Confidence:** {conf:.2f}\n")
    if lex_hits:
        out.append(f"- **Lexicon hits (total):** {lex_hits}\n")
    if notes:
        out.append(f"- **Notes:** {notes}\n")

    # Evidence bullets (strings or dicts)
    ev_items = []
    for item in evid:
        if item is None:
            continue
        if isinstance(item, str):
            s = item.strip()
        elif isinstance(item, dict):
            s = (item.get("text") or item.get("snippet") or item.get("evidence") or "").strip()
            if not s:
                s = str(item).strip()
        else:
            s = str(item).strip()

        if s:
            ev_items.append(s.replace("\n", " "))

    if ev_items:
        out.append("\n**Key evidence signals**\n\n")
        for s in ev_items[:8]:  # keep compact
            out.append(f"- {s}\n")

    out.append("\n")
    return "".join(out)


In [477]:
artifacts = generate_fit_report_en(
    person_thetas=person_thetas_hybrid,
    org_thetas=org_thetas_example,
    DIMENSIONS=DIMENSIONS,
    order=ORDER,
    out_dir="outputs",
    out_basename="fit_report_en_hybrid_demo",
    cap=1.5,
    meta={"instrument_version":"Hybrid-v1.0"},
    evidence_by_dim=evidence_by_dim
)


In [458]:
import inspect
print(inspect.getsource(dimension_block_en))


def dimension_block_en(dim_key: str, person_theta: float, org_theta: float, dim_cfg: dict, cap: float = 1.5, evidence_light=None) -> str:

    p = float(person_theta)
    o = float(org_theta)
    d = p - o
    gap = abs(d)

    fit_pct = fit_pct_from_gap(gap, cap=cap)
    compat = dimension_fit_label_en(gap)

    p_pole = infer_pole(dim_cfg, p)
    o_pole = infer_pole(dim_cfg, o)
    p_lbl = dim_cfg["poles"][p_pole]["label"]
    o_lbl = dim_cfg["poles"][o_pole]["label"]

    md = []
    md.append(f"### {dim_cfg['name']}\n\n")

    md.append(f"**Definition.** {dim_cfg['definition']}\n\n")

    md.append("**Compatibility diagnosis.**\n")
    md.append(f"- Person θ: **{p:+.2f}** → **{p_lbl}**\n")
    md.append(f"- Organization θ: **{o:+.2f}** → **{o_lbl}**\n")
    md.append(f"- Δθ (P−O): **{d:+.2f}** | |Δθ|: **{gap:.2f}** | Fit %: **{fit_pct:.0f}%** → **{compat}**\n\n")

    md.append("**Cognitive attributes (both poles).**\n")
    md.append(f"- **{dim_cfg['poles']['A']['short']}:**\n")
 

In [465]:
import inspect
print("dimension_block_en signature:", inspect.signature(dimension_block_en))
print("generate_fit_report_en signature:", inspect.signature(generate_fit_report_en))


dimension_block_en signature: (dim_key: str, person_theta: float, org_theta: float, dim_cfg: dict, cap: float = 1.5, evidence_light=None) -> str
generate_fit_report_en signature: (person_thetas: dict, org_thetas: dict, DIMENSIONS: dict, order: list, out_dir: str = 'outputs', out_basename: str = 'fit_report_en', cap: float = 1.5, meta: dict = None, evidence_by_dim: dict = None)


In [466]:
dk = "D1_GO"
lvl = gap_to_level_key(abs(person_thetas_hybrid[dk] - org_thetas_example[dk]))

print("level:", lvl)
print("practical value:", DIMENSIONS[dk].get("practical_implications", {}).get(lvl))
print("type:", type(DIMENSIONS[dk].get("practical_implications", {}).get(lvl)))


level: moderate_low
practical value: ['...']
type: <class 'list'>


In [467]:
snippet = dimension_block_en(
    "D1_GO",
    person_thetas_hybrid["D1_GO"],
    org_thetas_example["D1_GO"],
    DIMENSIONS["D1_GO"],
    cap=1.5,
    evidence_light=evidence_by_dim
)

print("HAS PRACTICAL HEADER:", "Practical implications" in snippet)
print("HAS RECO HEADER:", "Recommendations" in snippet)
print("HAS EVIDENCE HEADER:", "Evidence Pack" in snippet)

# Print only the practical section slice if present
ix = snippet.find("Practical implications")
print("\n--- PRACTICAL SLICE ---\n")
print(snippet[ix:ix+500] if ix != -1 else "NOT FOUND")


HAS PRACTICAL HEADER: False
HAS RECO HEADER: False
HAS EVIDENCE HEADER: False

--- PRACTICAL SLICE ---

NOT FOUND


In [468]:
import inspect
src = inspect.getsource(dimension_block_en)
print(src)


def dimension_block_en(dim_key: str, person_theta: float, org_theta: float, dim_cfg: dict, cap: float = 1.5, evidence_light=None) -> str:

    p = float(person_theta)
    o = float(org_theta)
    d = p - o
    gap = abs(d)

    fit_pct = fit_pct_from_gap(gap, cap=cap)
    compat = dimension_fit_label_en(gap)

    p_pole = infer_pole(dim_cfg, p)
    o_pole = infer_pole(dim_cfg, o)
    p_lbl = dim_cfg["poles"][p_pole]["label"]
    o_lbl = dim_cfg["poles"][o_pole]["label"]

    md = []
    md.append(f"### {dim_cfg['name']}\n\n")

    md.append(f"**Definition.** {dim_cfg['definition']}\n\n")

    md.append("**Compatibility diagnosis.**\n")
    md.append(f"- Person θ: **{p:+.2f}** → **{p_lbl}**\n")
    md.append(f"- Organization θ: **{o:+.2f}** → **{o_lbl}**\n")
    md.append(f"- Δθ (P−O): **{d:+.2f}** | |Δθ|: **{gap:.2f}** | Fit %: **{fit_pct:.0f}%** → **{compat}**\n\n")

    md.append("**Cognitive attributes (both poles).**\n")
    md.append(f"- **{dim_cfg['poles']['A']['short']}:**\n")
 

In [473]:
docx_path = export_md_to_docx(artifacts["md"], r"outputs\fit_report_en_hybrid_demo.docx")
print("DOCX:", docx_path)


DOCX: C:\Users\dasuarez\outputs\fit_report_en_hybrid_demo.docx


In [481]:
DIMENSIONS = DIMENSIONS_EN


In [482]:
print("Has practical:",
      "practical_implications" in DIMENSIONS["D1_GO"])

print("Has recos:",
      "actionable_recommendations" in DIMENSIONS["D1_GO"])


Has practical: True
Has recos: True


In [487]:
artifacts = generate_fit_report_en(
    person_thetas=person_thetas_hybrid,
    org_thetas=org_thetas_example,
    DIMENSIONS=DIMENSIONS,
    order=ORDER,
    out_dir="outputs",
    out_basename="fit_report_en_hybrid_demo",
    cap=1.5,
    meta={"instrument_version":"Hybrid-v1.0"},
    evidence_by_dim=evidence_by_dim
)


In [498]:
docx_path = export_md_to_docx(
    artifacts["md"],
    r"outputs\fit_report_en_hybrid_demo.docx"
)

print(docx_path)


C:\Users\dasuarez\outputs\fit_report_en_hybrid_demo.docx


# Demo

In [491]:
person_thetas_demo = {
    "D1_GO": 1.32,
    "D2_IDED": 0.48,
    "D3_ES": -0.35,
    "D4_OS": 0.72,
    "D5_EJ": 0.15,
    "D6_PP": 0.91,
}


In [492]:
org_thetas_demo = {
    "D1_GO": 0.65,
    "D2_IDED": 0.10,
    "D3_ES": -0.10,
    "D4_OS": 0.90,
    "D5_EJ": 0.40,
    "D6_PP": 0.30,
}


### Reference: person_thetas-hybrid (results)

person_thetas_hybrid
    "D1_GO":  1.32,
    "D2_IDED": 1.44,
    "D3_ES": -1.09,
    "D4_OS":  1.39,
    "D5_EJ": -0.98,
    "D6_PP":  1.49


In [495]:
def fused_to_evidence_light(fused: dict) -> dict:
    """
    Adapter: fuse_one_dimension output -> schema expected by render_evidence_light_md
    """
    if not fused:
        return {}

    return {
        "pole": fused.get("llm_pole", None),
        "confidence": float(fused.get("llm_conf", 0.0) or 0.0),
        "evidence": fused.get("evidence", []) or [],
        "lex_hits_total": int(fused.get("lex_hits_total", 0) or 0),
        "notes": fused.get("notes", "") or ""
    }


In [496]:
print("Has evidence keys:", list(evidence_by_dim.keys()))
print("D1_GO example:", evidence_by_dim.get("D1_GO"))
print("Person thetas keys:", list(person_thetas_hybrid.keys()))


Has evidence keys: ['D1_GO', 'D2_IDED', 'D3_ES', 'D4_OS', 'D5_EJ', 'D6_PP']
D1_GO example: {'pole': 'Goal-Oriented', 'confidence': 0.85, 'evidence': ['La prioridad fue el resultado: lanzamos una versión funcional', 'para no bloquear el despliegue'], 'lex_hits_total': 0, 'notes': 'Lexicon low hits: LLM weighted more'}
Person thetas keys: ['D1_GO', 'D2_IDED', 'D3_ES', 'D4_OS', 'D5_EJ', 'D6_PP']


In [497]:
ORDER = ["D1_GO","D2_IDED","D3_ES","D4_OS","D5_EJ","D6_PP"]

artifacts = generate_fit_report_en(
    person_thetas=person_thetas_hybrid,
    org_thetas=org_thetas_demo,     # (lo definimos como “org canónico”)
    DIMENSIONS=DIMENSIONS,
    order=ORDER,
    out_dir="outputs",
    out_basename="fit_report_en_hybrid_demo",
    cap=1.5,
    meta={"instrument_version":"Hybrid-v1.0"},
    evidence_by_dim=evidence_by_dim
)


In [501]:
docx_path = export_md_to_docx(
    artifacts["md"],
    r"outputs\fit_report_en_hybrid_demo.docx"
)

print(docx_path)

C:\Users\dasuarez\outputs\fit_report_en_hybrid_demo.docx


## Results (Paper)

In [506]:
import pandas as pd

ORDER = ["D1_GO","D2_IDED","D3_ES","D4_OS","D5_EJ","D6_PP"]

DIM_NAMES = {
    "D1_GO": "Goal Orientation",
    "D2_IDED": "Internal vs External Motivation",
    "D3_ES": "Flexibility vs Structure",
    "D4_OS": "Open vs Closed System",
    "D5_EJ": "People vs Task Orientation",
    "D6_PP": "Professional Identity"
}

rows = []

for d in ORDER:
    p = person_thetas_hybrid[d]
    o = org_thetas_demo[d]
    diff = p - o
    gap = abs(diff)
    fit = fit_pct_from_gap(gap)
    compat = dimension_fit_label_en(gap)

    rows.append({
        "Dimension": DIM_NAMES[d],
        "Person θ": round(p, 2),
        "Organization θ": round(o, 2),
        "Δθ (P–O)": round(diff, 2),
        "|Δθ|": round(gap, 2),
        "Fit %": round(fit, 0),
        "Compatibility": compat
    })

df_table = pd.DataFrame(rows)
df_table


,Dimension,Person θ,Organization θ,Δθ (P–O),|Δθ|,Fit %,Compatibility
0,Goal Orientation,1.32,0.65,0.67,0.67,56.0,Moderate-to-low compatibility
1,Internal vs External Motivation,1.44,0.10,1.34,1.34,11.0,Low compatibility
2,Flexibility vs Structure,-1.09,-0.10,-0.99,0.99,34.0,Low compatibility
3,Open vs Closed System,1.39,0.90,0.49,0.49,67.0,Moderate compatibility
4,People vs Task Orientation,-0.98,0.40,-1.38,1.38,8.0,Low compatibility
5,Professional Identity,1.49,0.30,1.19,1.19,21.0,Low compatibility


In [507]:
df_table.to_csv("outputs/table1_demo_results.csv", index=False)


In [504]:
labels = [DIM_NAMES[d] for d in ORDER]

p_vals = [person_thetas_hybrid[d] for d in ORDER]
o_vals = [org_thetas_demo[d] for d in ORDER]

radar_path = "outputs/figure1_radar_demo.png"

radar_chart_dual(
    person_vals=p_vals,
    org_vals=o_vals,
    labels=labels,
    out_png=radar_path,
    title="Cognitive–Cultural Fit Profile (θ, z-scored)",
    person_label="Person",
    org_label="Organization"
)

radar_path


'outputs/figure1_radar_demo.png'

In [505]:
evidence_by_dim["D1_GO"]["evidence"]


['La prioridad fue el resultado: lanzamos una versión funcional',
 'para no bloquear el despliegue']